<h1>Updated to Optimise a threshold for every feature using validation data and train only independent/leaf features and calculate inherited features deterministically.</h1>

In [4]:
#!/usr/bin/env python3
"""
Fine-tune RoBERTa for hierarchical multi-label semantic feature tagging of
NOUN/PROPN tokens.

Compatible with Python 3.9.21.

Key design changes
------------------
1. The model is trained only on independent semantic features. For each token,
   any feature that can be inherited from a more specific present feature is
   removed from the neural target. This is a per-token reduction rather than a
   global "terminal leaf only" rule, so features such as Human, Time, Shape,
   Communication, etc. can still be predicted directly when they are the most
   specific available feature for that token.

2. All inherited features are reconstructed deterministically from
   FEATURE_PARENTS for validation, testing, reporting, and later inference.

3. A separate sigmoid threshold is optimised for every neural output using the
   validation split only. The selected thresholds are frozen before the test
   split is evaluated and are saved to prediction_thresholds.json.

4. The official test corpus may contain a direct independent target that never
   occurred independently in training. Such a label is never added from test
   data. Complete test truth is read directly from the source annotations so
   held-out evaluation remains valid without test leakage.

Tagged data:
    Training/validation:
        ewt_train_wordnet_wsd_features_wordnet_fixed.jsonl
    Test:
        ewt_test_wordnet_wsd_features_wordnet_fixed.jsonl

Sentence/token sources:
    Training/validation:
        UD_English-EWT-master/
        UD_English-EWT-master/
        en_ewt-ud-train.conllu
    Test:
        UD_English-EWT-master/
        UD_English-EWT-master/
        en_ewt-ud-test.conllu

The validation split is created only from the training corpus. The official
test corpus remains untouched until final evaluation.

Loss:
    BCEWithLogitsLoss over INDEPENDENT feature labels only.

Headline metrics:
    Computed after deterministic hierarchy expansion, so they describe the
    complete semantic feature sets rather than only the neural outputs.

Requirements:
    python -m pip install torch transformers datasets accelerate conllu scikit-learn

CUDA is required.
"""

from __future__ import annotations

import json
import random
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Set, Tuple

import numpy as np
import torch
import torch.nn as nn

from conllu import parse_incr
from sklearn.metrics import precision_recall_fscore_support
from torch.utils.data import Dataset

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)


# =============================================================================
# Configuration
# =============================================================================

TRAIN_JSONL_PATH = Path(
    "../data/ewt_train_wordnet_wsd_features_wordnet_fixed.jsonl"
    #"ewt_train_wordnet_wsd_features_fully_tagged_llm_reviewed.jsonl"
)

TEST_JSONL_PATH = Path(
    "../data/ewt_test_wordnet_wsd_features_wordnet_fixed.jsonl"
    #"ewt_test_wordnet_wsd_features_fully_tagged_llm_reviewed.jsonl"
)

TRAIN_CONLLU_PATH = Path(
    "../data/UD_English-EWT-master/"
    "UD_English-EWT-master/"
    "en_ewt-ud-train.conllu"
)

TEST_CONLLU_PATH = Path(
    "../data/UD_English-EWT-master/"
    "UD_English-EWT-master/"
    "en_ewt-ud-test.conllu"
)

# MODEL_NAME = "FacebookAI/roberta-base"
MODEL_NAME = "FacebookAI/roberta-large"

OUTPUT_DIR = Path("../models/roberta_semantic_features")

# Validation is sampled only from the eligible training sentences.
# The separate test file is never split and is used only for final evaluation.
VALIDATION_FRACTION = 0.20

# A sentence is not moved into validation if doing so would reduce any
# independent feature to fewer than this many positive TRAINING examples.
# This prevents rare direct targets (for example Human) from disappearing
# entirely from the model's training subset merely because of the random split.
MIN_TRAIN_POSITIVES_PER_FEATURE = 1

RANDOM_SEED = 42

MAX_LENGTH = 256

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16

GRADIENT_ACCUMULATION_STEPS = 2

NUM_EPOCHS = 6

# Threshold used during epoch-by-epoch Trainer evaluation. Final thresholds are
# optimised per independent feature on the validation set after training.
DEFAULT_PREDICTION_THRESHOLD = 0.5

# Validation-only threshold search grid.
THRESHOLD_MIN = 0.05
THRESHOLD_MAX = 0.95
THRESHOLD_STEP = 0.01

# Only these POS categories receive feature supervision.
TARGET_POS = {"NOUN", "PROPN"}

# A NOUN/PROPN with no features is considered a failed source annotation rather
# than a valid all-negative target.
EXCLUDE_EMPTY_FEATURE_RECORDS = True

EVAL_STRATEGY = "epoch"

SPLIT_PATH = OUTPUT_DIR / "sentence_split.json"
LABELS_PATH = OUTPUT_DIR / "semantic_features.json"
THRESHOLDS_PATH = OUTPUT_DIR / "prediction_thresholds.json"
TEST_RESULTS_PATH = OUTPUT_DIR / "test_results.json"


# =============================================================================
# Feature hierarchy
# =============================================================================

# child -> direct parent(s)
FEATURE_PARENTS: Dict[str, Set[str]] = {
    "Organisation": {
        "Internal_Control",
        "Abstract",
    },
    "Natural_Force": {
        "Internal_Control",
    },
    "Animate": {
        "Internal_Control",
        "Natural",
    },
    "Human": {
        "Animate",
    },
    "Feminine": {
        "Human",
    },
    "Masculine": {
        "Human",
    },
    "Unspecified_Gender_or_Neuter": {
        "Human",
    },
    "Animal": {
        "Animate",
    },
    "Plant": {
        "Animate",
    },
    "Body_Part": {
        "Animate",
    },
    "Machine": {
        "Internal_Control",
        "Artefact",
    },
    "Vehicle": {
        "Machine",
    },
    "Natural": {
        "Concrete",
    },
    "Artefact": {
        "Concrete",
    },
    "Comestible": {
        "Concrete",
    },
    "Substance": {
        "Concrete",
    },
    "State_of_Matter": {
        "Concrete",
    },
    "Inanimate": {
        "Concrete",
    },
    "Garment": {
        "Artefact",
    },
    "Solid": {
        "State_of_Matter",
    },
    "Fluid": {
        "State_of_Matter",
    },
    "Rigidity": {
        "Solid",
    },
    "Shape": {
        "Solid",
    },
    "Rigid": {
        "Rigidity",
    },
    "Non_Rigid": {
        "Rigidity",
    },
    "Elongated": {
        "Shape",
    },
    "Pointy": {
        "Shape",
    },
    "Point_in_Time": {
        "Time",
    },
    "Duration": {
        "Time",
    },
    "Time": {
        "Abstract",
    },
    "Language": {
        "Abstract",
    },
    "Sound": {
        "Abstract",
    },
    "Communication": {
        "Abstract",
    },
    "Eventive": {
        "Abstract",
    },
    "Statement": {
        "Communication",
    },
    "Question": {
        "Communication",
    },
}


def validate_feature_hierarchy() -> None:
    """Fail early if FEATURE_PARENTS contains a cycle or invalid name."""

    all_features: Set[str] = set(FEATURE_PARENTS)

    for parents in FEATURE_PARENTS.values():
        all_features.update(parents)

    for feature in all_features:
        if not isinstance(feature, str) or not feature.strip():
            raise ValueError(
                "FEATURE_PARENTS contains an empty or non-string feature name."
            )

    visiting: Set[str] = set()
    visited: Set[str] = set()

    def visit(feature: str) -> None:
        if feature in visited:
            return

        if feature in visiting:
            raise ValueError(
                f"Cycle detected in FEATURE_PARENTS involving {feature!r}."
            )

        visiting.add(feature)

        for parent in FEATURE_PARENTS.get(feature, set()):
            visit(parent)

        visiting.remove(feature)
        visited.add(feature)

    for feature in sorted(all_features):
        visit(feature)


@lru_cache(maxsize=None)
def get_feature_ancestors(feature: str) -> frozenset:
    """Return every direct and indirect parent of *feature*."""

    ancestors: Set[str] = set()

    for parent in FEATURE_PARENTS.get(feature, set()):
        ancestors.add(parent)
        ancestors.update(get_feature_ancestors(parent))

    return frozenset(ancestors)


def expand_feature_hierarchy(features: Iterable[str]) -> Set[str]:
    """Add every deterministic ancestor required by FEATURE_PARENTS."""

    expanded: Set[str] = set(features)

    for feature in list(expanded):
        expanded.update(get_feature_ancestors(feature))

    return expanded


def reduce_to_independent_features(features: Iterable[str]) -> Set[str]:
    """
    Reduce a complete feature set to the features that must be predicted.

    A feature is redundant when it is an ancestor of another feature present
    for the same token. Such an ancestor can be reconstructed deterministically.

    Examples:
        {Human, Animate, Natural, Concrete, Internal_Control}
            -> {Human}

        {Feminine, Human, Animate, Natural, Concrete, Internal_Control}
            -> {Feminine}

        {Statement, Communication, Abstract, Time}
            -> {Statement, Time}

    This is deliberately PER TOKEN. It is not the same as training only labels
    that are global terminal leaves in the ontology. A generic Human can still
    be a direct neural target if no more specific Human child is present.
    """

    feature_set = set(features)

    inherited_from_other_features: Set[str] = set()

    for feature in feature_set:
        inherited_from_other_features.update(
            get_feature_ancestors(feature)
        )

    return feature_set - inherited_from_other_features


def missing_hierarchy_features(features: Iterable[str]) -> Set[str]:
    """Return ancestors required by the hierarchy but absent from *features*."""

    feature_set = set(features)
    return expand_feature_hierarchy(feature_set) - feature_set


# =============================================================================
# CUDA
# =============================================================================

def check_cuda() -> None:
    """Require CUDA rather than silently falling back to the CPU."""

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. This script is configured to require "
            "a CUDA-capable GPU."
        )

    print("=" * 80)
    print("CUDA")
    print("=" * 80)
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"CUDA runtime: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    props = torch.cuda.get_device_properties(0)

    print(
        "VRAM: "
        f"{props.total_memory / (1024 ** 3):.2f} GiB"
    )
    print()


# =============================================================================
# Load CoNLL-U
# =============================================================================

def load_conllu(path: Path) -> List[Any]:
    """Load all CoNLL-U sentences."""

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find CoNLL-U file:\n{path.resolve()}"
        )

    sentences: List[Any] = []

    with path.open("r", encoding="utf-8") as handle:
        for sentence in parse_incr(handle):
            sentences.append(sentence)

    return sentences


def get_sentence_words(sentence: Any) -> Tuple[List[str], List[int]]:
    """
    Return ordinary integer-ID tokens and their original CoNLL-U token IDs.

    Multiword-token placeholder rows and empty nodes are excluded.
    """

    words: List[str] = []
    token_ids: List[int] = []

    for token in sentence:
        token_id = token.get("id")

        if not isinstance(token_id, int):
            continue

        form = token.get("form")

        if form is None:
            continue

        words.append(str(form))
        token_ids.append(token_id)

    return words, token_ids


# =============================================================================
# Load tagged JSONL
# =============================================================================

def load_tagged_records(
    path: Path,
) -> Dict[int, Dict[int, Dict[str, Any]]]:
    """
    Return:

        sentence_index
            -> token_id
                -> JSONL record

    Only NOUN and PROPN records are retained.
    """

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find tagged JSONL:\n{path.resolve()}"
        )

    records: Dict[int, Dict[int, Dict[str, Any]]] = defaultdict(dict)

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                print(
                    "Warning: invalid JSON on line "
                    f"{line_number}: {exc}"
                )
                continue

            if record.get("upos") not in TARGET_POS:
                continue

            sentence_index = record.get("sentence_index")
            token_id = record.get("token_id")

            if not isinstance(sentence_index, int):
                continue

            if not isinstance(token_id, int):
                continue

            records[sentence_index][token_id] = record

    return dict(records)


# =============================================================================
# Labels
# =============================================================================

def clean_features(record: Dict[str, Any]) -> List[str]:
    """Return unique, non-empty COMPLETE feature names from a JSONL record."""

    raw = record.get("features", [])

    if not isinstance(raw, list):
        return []

    return sorted(
        {
            str(feature).strip()
            for feature in raw
            if str(feature).strip()
        }
    )


def get_independent_record_features(record: Dict[str, Any]) -> List[str]:
    """Return the per-token minimal feature set that the model must predict."""

    return sorted(
        reduce_to_independent_features(
            clean_features(record)
        )
    )


def build_full_label_vocabulary(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    training_sentence_indices: Sequence[int],
) -> List[str]:
    """
    Build the COMPLETE evaluation vocabulary from training data only.

    Deterministic ancestors are included even if a training record omitted one,
    although hierarchy closure is separately verified and should normally make
    this addition unnecessary.
    """

    labels: Set[str] = set()

    for sentence_index in training_sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            features = clean_features(record)
            labels.update(features)
            labels.update(expand_feature_hierarchy(features))

    if not labels:
        raise RuntimeError(
            "No semantic features were found in the training split."
        )

    return sorted(labels)


def build_prediction_label_vocabulary(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    training_sentence_indices: Sequence[int],
) -> List[str]:
    """Build the neural-output vocabulary from independent training targets."""

    labels: Set[str] = set()

    for sentence_index in training_sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            labels.update(
                get_independent_record_features(record)
            )

    if not labels:
        raise RuntimeError(
            "No independent semantic features were found in the training split."
        )

    return sorted(labels)


# =============================================================================
# Sentence selection and splitting
# =============================================================================

def get_eligible_sentence_indices(
    sentences: Sequence[Any],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> List[int]:
    """Eligible sentences contain at least one non-empty NOUN/PROPN target."""

    eligible: List[int] = []

    for sentence_index in range(len(sentences)):
        token_records = tagged_records.get(sentence_index, {})

        valid_records = [
            record
            for record in token_records.values()
            if clean_features(record)
        ]

        if valid_records:
            eligible.append(sentence_index)

    return eligible


def get_sentence_independent_feature_counts(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_index: int,
) -> Counter:
    """Count independent positive targets in one source sentence."""

    counts: Counter = Counter()

    for record in tagged_records.get(sentence_index, {}).values():
        if not clean_features(record):
            continue

        for feature in get_independent_record_features(record):
            counts[feature] += 1

    return counts


def make_train_validation_split(
    eligible_indices: Sequence[int],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> Dict[str, List[int]]:
    """Create a label-aware train/validation split from TRAINING only.

    The validation set is sampled from the training corpus, but a candidate
    sentence is moved to validation only if every independent feature occurring
    in that sentence will still have at least
    MIN_TRAIN_POSITIVES_PER_FEATURE positive examples left in the training
    subset.

    This matters for the per-token independent-target design: an internal
    hierarchy feature such as Human can occasionally be a direct neural target.
    A purely random split can otherwise place all of its rare direct examples
    in validation, leaving no Human output head to learn.

    The official test corpus is deliberately not accepted by this function, so
    test labels cannot influence the split.
    """

    if not 0.0 < VALIDATION_FRACTION < 1.0:
        raise ValueError(
            "VALIDATION_FRACTION must be strictly between 0 and 1."
        )

    if MIN_TRAIN_POSITIVES_PER_FEATURE < 1:
        raise ValueError(
            "MIN_TRAIN_POSITIVES_PER_FEATURE must be at least 1."
        )

    if len(eligible_indices) < 2:
        raise RuntimeError(
            "Need at least two eligible training sentences to create a "
            "train/validation split."
        )

    rng = random.Random(RANDOM_SEED)

    shuffled = list(eligible_indices)
    rng.shuffle(shuffled)

    target_validation_count = int(
        round(len(shuffled) * VALIDATION_FRACTION)
    )
    target_validation_count = max(1, target_validation_count)
    target_validation_count = min(
        target_validation_count,
        len(shuffled) - 1,
    )

    sentence_feature_counts: Dict[int, Counter] = {
        sentence_index: get_sentence_independent_feature_counts(
            tagged_records,
            sentence_index,
        )
        for sentence_index in shuffled
    }

    all_training_file_counts: Counter = Counter()

    for counts in sentence_feature_counts.values():
        all_training_file_counts.update(counts)

    remaining_train_counts: Counter = Counter(all_training_file_counts)

    validation_indices: List[int] = []
    validation_set: Set[int] = set()
    blocked_by_coverage = 0

    for sentence_index in shuffled:
        if len(validation_indices) >= target_validation_count:
            break

        sentence_counts = sentence_feature_counts[sentence_index]

        can_move_to_validation = all(
            remaining_train_counts[feature] - count
            >= MIN_TRAIN_POSITIVES_PER_FEATURE
            for feature, count in sentence_counts.items()
        )

        if not can_move_to_validation:
            blocked_by_coverage += 1
            continue

        validation_indices.append(sentence_index)
        validation_set.add(sentence_index)

        for feature, count in sentence_counts.items():
            remaining_train_counts[feature] -= count

    train_indices = [
        sentence_index
        for sentence_index in shuffled
        if sentence_index not in validation_set
    ]

    if not validation_indices:
        raise RuntimeError(
            "Could not allocate any validation sentences without removing "
            "the last training example of at least one independent feature."
        )

    if len(validation_indices) < target_validation_count:
        print(
            "Warning: label-coverage constraints reduced the validation "
            f"split from {target_validation_count:,} to "
            f"{len(validation_indices):,} sentences."
        )

    protected_features = sorted(
        feature
        for feature, count in remaining_train_counts.items()
        if count <= MIN_TRAIN_POSITIVES_PER_FEATURE
    )

    print()
    print("Label-aware train/validation split:")
    print(
        f"  Target validation sentences : {target_validation_count:,}"
    )
    print(
        f"  Actual validation sentences : {len(validation_indices):,}"
    )
    print(
        f"  Training sentences retained : {len(train_indices):,}"
    )
    print(
        f"  Candidates blocked to preserve rare labels: "
        f"{blocked_by_coverage:,}"
    )

    if protected_features:
        print(
            "  Features at minimum retained training support: "
            + ", ".join(protected_features)
        )

    # Internal consistency check: every independent feature found anywhere in
    # the eligible training file must still occur in the final training subset.
    all_training_file_labels = {
        feature
        for feature, count in all_training_file_counts.items()
        if count > 0
    }

    final_train_labels: Set[str] = set()

    for sentence_index in train_indices:
        final_train_labels.update(
            sentence_feature_counts[sentence_index].keys()
        )

    missing_after_split = all_training_file_labels - final_train_labels

    if missing_after_split:
        raise RuntimeError(
            "Internal split error: independent labels disappeared from the "
            f"training subset: {sorted(missing_after_split)}"
        )

    return {
        "train": train_indices,
        "validation": validation_indices,
    }


# =============================================================================
# Dataset
# =============================================================================

class SemanticFeatureDataset(Dataset):
    """
    RoBERTa multi-label token-classification dataset.

    IMPORTANT:
        labels contain only INDEPENDENT neural targets.

    Labels have shape:
        [sequence_length, number_of_prediction_features]

    For supervised tokens:
        0 = independent feature absent
        1 = independent feature present

    For ignored tokens:
        -100 across all labels

    Only the first RoBERTa subword corresponding to each source token receives
    supervision.
    """

    def __init__(
        self,
        sentences: Sequence[Any],
        sentence_indices: Sequence[int],
        tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
        tokenizer: Any,
        label2id: Dict[str, int],
        max_length: int,
    ) -> None:
        self.sentences = sentences
        self.sentence_indices = list(sentence_indices)
        self.tagged_records = tagged_records
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.num_labels = len(label2id)
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.sentence_indices)

    def __getitem__(
        self,
        dataset_index: int,
    ) -> Dict[str, Any]:
        sentence_index = self.sentence_indices[dataset_index]
        sentence = self.sentences[sentence_index]

        words, token_ids = get_sentence_words(sentence)

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=True,
        )

        word_ids = encoding.word_ids()
        labels: List[List[float]] = []
        previous_word_id: Optional[int] = None
        sentence_records = self.tagged_records.get(sentence_index, {})

        for word_id in word_ids:
            if word_id is None:
                labels.append([-100.0] * self.num_labels)
                previous_word_id = word_id
                continue

            # Ignore all but the first subword.
            if word_id == previous_word_id:
                labels.append([-100.0] * self.num_labels)
                continue

            previous_word_id = word_id

            if not 0 <= word_id < len(token_ids):
                labels.append([-100.0] * self.num_labels)
                continue

            conllu_token_id = token_ids[word_id]
            record = sentence_records.get(conllu_token_id)

            if record is None:
                labels.append([-100.0] * self.num_labels)
                continue

            full_features = clean_features(record)

            if not full_features and EXCLUDE_EMPTY_FEATURE_RECORDS:
                labels.append([-100.0] * self.num_labels)
                continue

            independent_features = reduce_to_independent_features(
                full_features
            )

            target = [0.0] * self.num_labels

            for feature in independent_features:
                label_id = self.label2id.get(feature)

                if label_id is not None:
                    target[label_id] = 1.0

            labels.append(target)

        encoding["labels"] = labels
        encoding["sentence_index"] = sentence_index

        return encoding


# =============================================================================
# Data collator
# =============================================================================

class MultiLabelTokenCollator:
    """Dynamically pad inputs and 3-D multi-label token targets."""

    def __init__(
        self,
        tokenizer: Any,
        num_labels: int,
    ) -> None:
        self.tokenizer = tokenizer
        self.num_labels = num_labels

    def __call__(
        self,
        features: List[Dict[str, Any]],
    ) -> Dict[str, torch.Tensor]:
        clean_features_list: List[Dict[str, Any]] = []
        labels_list: List[List[List[float]]] = []

        for item in features:
            item = dict(item)
            item.pop("sentence_index", None)

            labels = item.pop("labels")
            labels_list.append(labels)
            clean_features_list.append(item)

        batch = self.tokenizer.pad(
            clean_features_list,
            padding=True,
            return_tensors="pt",
        )

        batch_size = len(labels_list)
        max_seq_length = batch["input_ids"].shape[1]

        padded_labels = torch.full(
            (
                batch_size,
                max_seq_length,
                self.num_labels,
            ),
            -100.0,
            dtype=torch.float32,
        )

        for batch_index, labels in enumerate(labels_list):
            label_tensor = torch.tensor(
                labels,
                dtype=torch.float32,
            )

            sequence_length = label_tensor.shape[0]

            padded_labels[
                batch_index,
                :sequence_length,
                :,
            ] = label_tensor

        batch["labels"] = padded_labels

        return batch


# =============================================================================
# Model
# =============================================================================

class RobertaMultiLabelModel(nn.Module):
    """RoBERTa token classifier with BCE loss for independent feature labels."""

    def __init__(
        self,
        model_name: str,
        num_labels: int,
        id2label: Dict[int, str],
        label2id: Dict[str, int],
    ) -> None:
        super().__init__()

        self.model = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            id2label=id2label,
            label2id=label2id,
        )

        self.config = self.model.config

        self.loss_function = nn.BCEWithLogitsLoss(
            reduction="mean"
        )

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        **kwargs,
    ):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs,
        )

        logits = outputs.logits
        loss = None

        if labels is not None:
            valid_mask = labels[:, :, 0] != -100.0

            valid_logits = logits[valid_mask]
            valid_labels = labels[valid_mask]

            if valid_logits.numel() == 0:
                loss = logits.sum() * 0.0
            else:
                loss = self.loss_function(
                    valid_logits,
                    valid_labels,
                )

        return {
            "loss": loss,
            "logits": logits,
        }

    def save_pretrained(
        self,
        save_directory,
        **kwargs,
    ):
        return self.model.save_pretrained(
            save_directory,
            **kwargs,
        )


# =============================================================================
# Hierarchy matrices and array helpers
# =============================================================================

def build_hierarchy_expansion_matrix(
    prediction_labels: Sequence[str],
    full_labels: Sequence[str],
) -> np.ndarray:
    """
    Build a boolean matrix mapping neural outputs to complete feature sets.

    Shape:
        [number_of_prediction_features, number_of_full_features]
    """

    full_label2id = {
        label: index
        for index, label in enumerate(full_labels)
    }

    matrix = np.zeros(
        (len(prediction_labels), len(full_labels)),
        dtype=np.int8,
    )

    for prediction_id, feature in enumerate(prediction_labels):
        expanded = expand_feature_hierarchy({feature})

        for expanded_feature in expanded:
            full_id = full_label2id.get(expanded_feature)

            if full_id is None:
                raise RuntimeError(
                    "Hierarchy expansion produced a feature absent from the "
                    f"full vocabulary: {expanded_feature!r}."
                )

            matrix[prediction_id, full_id] = 1

    return matrix


def expand_binary_feature_matrix(
    independent_matrix: np.ndarray,
    hierarchy_matrix: np.ndarray,
) -> np.ndarray:
    """Expand [tokens, independent] binary values to [tokens, complete]."""

    expanded = independent_matrix.astype(np.int32) @ hierarchy_matrix.astype(
        np.int32
    )

    return (expanded > 0).astype(np.int32)


def extract_valid_token_arrays(
    logits: np.ndarray,
    labels: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return probabilities and true independent labels for supervised tokens."""

    valid_token_mask = labels[:, :, 0] != -100.0

    probabilities = sigmoid_numpy(logits)[
        valid_token_mask
    ]

    true_labels = labels[
        valid_token_mask
    ].astype(np.int32)

    return probabilities, true_labels


def build_full_truth_matrix_for_dataset(
    dataset: "SemanticFeatureDataset",
    full_labels: Sequence[str],
) -> np.ndarray:
    """Build complete source truth for every supervised token in dataset order.

    This is intentionally independent of the neural-output vocabulary.  It is
    therefore safe and necessary for a held-out test corpus that may contain a
    feature such as ``Human`` as a direct independent target even when that
    feature never occurred independently in the training corpus.

    The model is still trained only on training-derived independent output
    heads.  This function merely preserves the complete gold annotations for
    fair hierarchy-expanded evaluation.
    """

    full_label2id = {
        label: index
        for index, label in enumerate(full_labels)
    }

    rows: List[np.ndarray] = []

    for sentence_index in dataset.sentence_indices:
        sentence = dataset.sentences[sentence_index]
        words, token_ids = get_sentence_words(sentence)

        encoding = dataset.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=dataset.max_length,
            add_special_tokens=True,
        )

        word_ids = encoding.word_ids()
        previous_word_id: Optional[int] = None
        sentence_records = dataset.tagged_records.get(sentence_index, {})

        for word_id in word_ids:
            if word_id is None:
                previous_word_id = word_id
                continue

            # Match SemanticFeatureDataset.__getitem__: only the first subword
            # for each source token receives supervision.
            if word_id == previous_word_id:
                continue

            previous_word_id = word_id

            if not 0 <= word_id < len(token_ids):
                continue

            conllu_token_id = token_ids[word_id]
            record = sentence_records.get(conllu_token_id)

            if record is None:
                continue

            features = clean_features(record)

            if not features and EXCLUDE_EMPTY_FEATURE_RECORDS:
                continue

            expanded_features = expand_feature_hierarchy(features)
            row = np.zeros(len(full_labels), dtype=np.int32)

            for feature in expanded_features:
                label_id = full_label2id.get(feature)

                if label_id is None:
                    raise RuntimeError(
                        "Complete source truth contains a feature absent from "
                        f"the training-derived full vocabulary: {feature!r}."
                    )

                row[label_id] = 1

            rows.append(row)

    if not rows:
        return np.zeros((0, len(full_labels)), dtype=np.int32)

    return np.stack(rows, axis=0)


# =============================================================================
# Metrics and thresholds
# =============================================================================

def sigmoid_numpy(x: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid."""

    x = np.clip(x, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-x))


def make_threshold_vector(
    prediction_labels: Sequence[str],
    thresholds: Dict[str, float],
) -> np.ndarray:
    """Return thresholds in neural-output column order."""

    return np.asarray(
        [
            float(
                thresholds.get(
                    label,
                    DEFAULT_PREDICTION_THRESHOLD,
                )
            )
            for label in prediction_labels
        ],
        dtype=np.float32,
    )


def threshold_probabilities(
    probabilities: np.ndarray,
    threshold_vector: np.ndarray,
) -> np.ndarray:
    """Apply one threshold per independent feature."""

    return (
        probabilities >= threshold_vector.reshape(1, -1)
    ).astype(np.int32)


def calculate_average_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    prefix: str = "",
) -> Dict[str, float]:
    """Calculate micro and macro precision/recall/F1."""

    if y_true.shape[0] == 0:
        return {
            f"{prefix}precision_micro": 0.0,
            f"{prefix}recall_micro": 0.0,
            f"{prefix}f1_micro": 0.0,
            f"{prefix}precision_macro": 0.0,
            f"{prefix}recall_macro": 0.0,
            f"{prefix}f1_macro": 0.0,
        }

    (
        precision_micro,
        recall_micro,
        f1_micro,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="micro",
        zero_division=0,
    )

    (
        precision_macro,
        recall_macro,
        f1_macro,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    return {
        f"{prefix}precision_micro": float(precision_micro),
        f"{prefix}recall_micro": float(recall_micro),
        f"{prefix}f1_micro": float(f1_micro),
        f"{prefix}precision_macro": float(precision_macro),
        f"{prefix}recall_macro": float(recall_macro),
        f"{prefix}f1_macro": float(f1_macro),
    }


def calculate_hierarchical_metrics_from_probabilities(
    probabilities: np.ndarray,
    y_true_independent: np.ndarray,
    threshold_vector: np.ndarray,
    hierarchy_matrix: np.ndarray,
    y_true_full: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    """
    Report both direct neural-output metrics and complete hierarchy metrics.

    Unprefixed metrics are the COMPLETE hierarchy metrics.  During ordinary
    training/validation, complete truth can be reconstructed from the
    independent targets.  For a genuinely held-out test corpus, ``y_true_full``
    may instead be supplied directly from the source annotations so an unseen
    direct independent label is not silently erased from the gold truth.
    """

    y_pred_independent = threshold_probabilities(
        probabilities,
        threshold_vector,
    )

    if y_true_full is None:
        y_true_full = expand_binary_feature_matrix(
            y_true_independent,
            hierarchy_matrix,
        )

    if y_true_full.shape[0] != y_true_independent.shape[0]:
        raise RuntimeError(
            "Complete and independent truth matrices contain different "
            f"numbers of supervised tokens: {y_true_full.shape[0]} vs "
            f"{y_true_independent.shape[0]}."
        )

    y_pred_full = expand_binary_feature_matrix(
        y_pred_independent,
        hierarchy_matrix,
    )

    metrics = calculate_average_metrics(
        y_true_full,
        y_pred_full,
        prefix="",
    )

    metrics.update(
        calculate_average_metrics(
            y_true_independent,
            y_pred_independent,
            prefix="direct_",
        )
    )

    return metrics


def make_compute_metrics(
    prediction_labels: Sequence[str],
    hierarchy_matrix: np.ndarray,
):
    """Create Trainer metrics using the fixed default 0.5 thresholds."""

    default_thresholds = np.full(
        len(prediction_labels),
        DEFAULT_PREDICTION_THRESHOLD,
        dtype=np.float32,
    )

    def compute_metrics(eval_prediction) -> Dict[str, float]:
        logits, labels = eval_prediction

        probabilities, y_true_independent = extract_valid_token_arrays(
            logits,
            labels,
        )

        return calculate_hierarchical_metrics_from_probabilities(
            probabilities=probabilities,
            y_true_independent=y_true_independent,
            threshold_vector=default_thresholds,
            hierarchy_matrix=hierarchy_matrix,
        )

    return compute_metrics


def build_threshold_grid() -> np.ndarray:
    """Build an inclusive, stable validation threshold search grid."""

    if THRESHOLD_STEP <= 0:
        raise ValueError("THRESHOLD_STEP must be > 0.")

    if not 0.0 < THRESHOLD_MIN < 1.0:
        raise ValueError("THRESHOLD_MIN must be between 0 and 1.")

    if not 0.0 < THRESHOLD_MAX < 1.0:
        raise ValueError("THRESHOLD_MAX must be between 0 and 1.")

    if THRESHOLD_MIN > THRESHOLD_MAX:
        raise ValueError("THRESHOLD_MIN cannot exceed THRESHOLD_MAX.")

    count = int(
        round(
            (THRESHOLD_MAX - THRESHOLD_MIN)
            / THRESHOLD_STEP
        )
    ) + 1

    return np.linspace(
        THRESHOLD_MIN,
        THRESHOLD_MAX,
        count,
        dtype=np.float64,
    )


def binary_metrics_for_column(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> Tuple[float, float, float, int]:
    """Return binary precision, recall, F1 and positive support."""

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    support = int(y_true.sum())

    return (
        float(precision),
        float(recall),
        float(f1),
        support,
    )


def optimise_feature_thresholds(
    probabilities: np.ndarray,
    y_true_independent: np.ndarray,
    prediction_labels: Sequence[str],
) -> Tuple[Dict[str, float], Dict[str, Dict[str, float]]]:
    """
    Optimise one sigmoid threshold per independent feature on VALIDATION ONLY.

    Objective:
        maximise that feature's binary F1.

    Stability rules:
    - If validation has no positives or no negatives for a feature, keep 0.5.
    - If several thresholds tie for best F1, choose the one closest to 0.5.
      This avoids unnecessarily extreme thresholds for flat validation curves.
    """

    grid = build_threshold_grid()

    thresholds: Dict[str, float] = {}
    diagnostics: Dict[str, Dict[str, float]] = {}

    for label_id, feature in enumerate(prediction_labels):
        y_true = y_true_independent[:, label_id]
        probs = probabilities[:, label_id]

        positive_support = int(y_true.sum())
        negative_support = int((1 - y_true).sum())

        default_pred = (
            probs >= DEFAULT_PREDICTION_THRESHOLD
        ).astype(np.int32)

        (
            default_precision,
            default_recall,
            default_f1,
            _,
        ) = binary_metrics_for_column(
            y_true,
            default_pred,
        )

        if positive_support == 0 or negative_support == 0:
            best_threshold = DEFAULT_PREDICTION_THRESHOLD
        else:
            best_f1 = -1.0
            best_candidates: List[float] = []

            for threshold in grid:
                y_pred = (probs >= threshold).astype(np.int32)

                _, _, f1, _ = binary_metrics_for_column(
                    y_true,
                    y_pred,
                )

                if f1 > best_f1 + 1e-12:
                    best_f1 = f1
                    best_candidates = [float(threshold)]
                elif abs(f1 - best_f1) <= 1e-12:
                    best_candidates.append(float(threshold))

            best_threshold = min(
                best_candidates,
                key=lambda value: (
                    abs(value - DEFAULT_PREDICTION_THRESHOLD),
                    value,
                ),
            )

        best_pred = (
            probs >= best_threshold
        ).astype(np.int32)

        (
            best_precision,
            best_recall,
            best_f1,
            _,
        ) = binary_metrics_for_column(
            y_true,
            best_pred,
        )

        thresholds[feature] = float(best_threshold)

        diagnostics[feature] = {
            "threshold": float(best_threshold),
            "support": positive_support,
            "negative_support": negative_support,
            "precision_default": default_precision,
            "recall_default": default_recall,
            "f1_default": default_f1,
            "precision_optimised": best_precision,
            "recall_optimised": best_recall,
            "f1_optimised": best_f1,
            "f1_change": best_f1 - default_f1,
        }

    return thresholds, diagnostics


def calculate_per_feature_metrics_from_binary(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    labels: Sequence[str],
) -> Dict[str, Dict[str, float]]:
    """Calculate binary precision/recall/F1 independently for each column."""

    results: Dict[str, Dict[str, float]] = {}

    for label_id, feature in enumerate(labels):
        precision, recall, f1, support = binary_metrics_for_column(
            y_true[:, label_id],
            y_pred[:, label_id],
        )

        results[feature] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        }

    return results


# =============================================================================
# Dataset diagnostics and validation
# =============================================================================

def count_feature_occurrences(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_indices: Sequence[int],
    independent_only: bool = False,
) -> Counter:
    """Count complete or independent positive labels in a split."""

    counts: Counter = Counter()

    for sentence_index in sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            if independent_only:
                features = get_independent_record_features(record)
            else:
                features = clean_features(record)

            for feature in features:
                counts[feature] += 1

    return counts


def count_empty_records(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_indices: Sequence[int],
) -> int:
    """Count NOUN/PROPN records having no complete feature tags."""

    count = 0

    for sentence_index in sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            if not clean_features(record):
                count += 1

    return count


def verify_full_label_coverage(
    full_labels: Sequence[str],
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    """Ensure validation/test contain no complete label unseen in training."""

    training_labels = set(full_labels)

    for split_name in ("validation", "test"):
        tagged_records, sentence_indices = split_sources[split_name]
        observed: Set[str] = set()

        for sentence_index in sentence_indices:
            for record in tagged_records.get(sentence_index, {}).values():
                observed.update(clean_features(record))

        unknown = observed - training_labels

        if unknown:
            raise RuntimeError(
                f"{split_name} contains full labels unseen in training: "
                f"{sorted(unknown)}"
            )


def verify_prediction_label_coverage(
    prediction_labels: Sequence[str],
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
    training_file_prediction_labels: Optional[Sequence[str]] = None,
) -> Dict[str, Dict[str, int]]:
    """Check independent-target coverage without leaking test labels.

    Validation remains strict: every validation independent target must have a
    training-derived neural output head.

    Test is different.  A genuine held-out corpus may contain a direct
    independent target that never occurred independently in training.  Such a
    label must NOT be added to the model from the test set.  Instead, report it
    and let complete-feature evaluation use the source gold matrix directly.
    """

    training_labels = set(prediction_labels)
    training_file_labels = (
        set(training_file_prediction_labels)
        if training_file_prediction_labels is not None
        else set(training_labels)
    )

    unseen_test_support: Dict[str, Dict[str, int]] = {}

    for split_name in ("validation", "test"):
        tagged_records, sentence_indices = split_sources[split_name]
        observed_counts: Counter = Counter()

        for sentence_index in sentence_indices:
            for record in tagged_records.get(sentence_index, {}).values():
                for feature in get_independent_record_features(record):
                    observed_counts[feature] += 1

        observed = set(observed_counts)
        unknown = observed - training_labels

        if not unknown:
            continue

        lost_due_to_split = unknown & training_file_labels
        absent_from_training_file = unknown - training_file_labels

        if split_name == "validation":
            details: List[str] = []

            if lost_due_to_split:
                details.append(
                    "present somewhere in the training file but absent from "
                    f"the training subset: {sorted(lost_due_to_split)}"
                )

            if absent_from_training_file:
                details.append(
                    "never occur as independent targets anywhere in the "
                    f"training file: {sorted(absent_from_training_file)}"
                )

            raise RuntimeError(
                "validation contains independent labels with no learnable "
                "training output. "
                + "; ".join(details)
            )

        # For the official test set, do not modify the model vocabulary.
        unseen_test_support = {
            feature: {
                "support": int(observed_counts[feature]),
                "present_in_training_file_as_independent": int(
                    feature in training_file_labels
                ),
            }
            for feature in sorted(unknown)
        }

        print()
        print("=" * 80)
        print("HELD-OUT TEST LABEL COVERAGE WARNING")
        print("=" * 80)
        print(
            "The test corpus contains direct independent targets that have no "
            "training-derived neural output head. They will NOT be added to "
            "the model vocabulary. Complete test truth will be read directly "
            "from the source annotations instead."
        )

        for feature in sorted(unknown):
            print(
                f"  {feature:35s} support={observed_counts[feature]:,}"
            )

        print()

    return unseen_test_support


def verify_hierarchy_closure(
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    """
    Ensure source feature sets already contain all ancestors required by the
    hierarchy. This prevents evaluation against inconsistent source targets.
    """

    problems: List[str] = []
    max_examples = 20

    for split_name in ("train", "validation", "test"):
        tagged_records, sentence_indices = split_sources[split_name]
        problem_count = 0

        for sentence_index in sentence_indices:
            for token_id, record in tagged_records.get(
                sentence_index,
                {},
            ).items():
                features = clean_features(record)

                if not features:
                    continue

                missing = missing_hierarchy_features(features)

                if not missing:
                    continue

                problem_count += 1

                if len(problems) < max_examples:
                    problems.append(
                        f"{split_name}: sentence={sentence_index}, "
                        f"token={token_id}, "
                        f"form={record.get('form')!r}, "
                        f"missing={sorted(missing)}"
                    )

        if problem_count:
            print(
                f"Hierarchy closure problems in {split_name}: "
                f"{problem_count:,} token(s)"
            )

    if problems:
        example_text = "\n".join(
            f"  - {problem}"
            for problem in problems
        )

        raise RuntimeError(
            "The input dataset is not hierarchy-closed. Fix the source tags "
            "before training so deterministic reduction/expansion is exact.\n"
            f"Examples:\n{example_text}"
        )


def print_split_statistics(
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    print("=" * 80)
    print("DATASET SPLIT")
    print("=" * 80)

    for split_name in ("train", "validation", "test"):
        tagged_records, indices = split_sources[split_name]

        full_counts = count_feature_occurrences(
            tagged_records,
            indices,
            independent_only=False,
        )

        independent_counts = count_feature_occurrences(
            tagged_records,
            indices,
            independent_only=True,
        )

        empty_count = count_empty_records(
            tagged_records,
            indices,
        )

        noun_count = sum(
            len(tagged_records.get(index, {}))
            for index in indices
        )

        source_name = (
            "training file"
            if split_name in {"train", "validation"}
            else "test file"
        )

        print(
            f"{split_name:12s}: "
            f"{len(indices):,} sentences | "
            f"{noun_count:,} NOUN/PROPN tokens | "
            f"{sum(full_counts.values()):,} complete positives -> "
            f"{sum(independent_counts.values()):,} independent positives | "
            f"{empty_count:,} empty-feature tokens excluded | "
            f"source={source_name}"
        )

    print()


def print_prediction_label_statistics(
    prediction_labels: Sequence[str],
    full_labels: Sequence[str],
    train_tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    train_sentence_indices: Sequence[int],
) -> None:
    """Show which labels are neural outputs versus deterministic-only parents."""

    train_independent_counts = count_feature_occurrences(
        train_tagged_records,
        train_sentence_indices,
        independent_only=True,
    )

    prediction_set = set(prediction_labels)
    deterministic_only = [
        feature
        for feature in full_labels
        if feature not in prediction_set
    ]

    print("=" * 80)
    print("NEURAL OUTPUT FEATURES")
    print("=" * 80)
    print(
        f"Complete evaluation features: {len(full_labels)}"
    )
    print(
        f"Independent neural outputs: {len(prediction_labels)}"
    )
    print(
        f"Deterministic-only features in this training split: "
        f"{len(deterministic_only)}"
    )
    print()

    print(
        f"{'Feature':35s} "
        f"{'Independent train support':>26s}"
    )
    print("-" * 64)

    for feature in prediction_labels:
        print(
            f"{feature:35s} "
            f"{train_independent_counts[feature]:26d}"
        )

    if deterministic_only:
        print()
        print("Deterministic-only in this split:")

        for feature in deterministic_only:
            print(f"  - {feature}")

    print()


# =============================================================================
# Reporting helpers
# =============================================================================

def print_metric_dictionary(
    metrics: Dict[str, float],
) -> None:
    for key, value in sorted(metrics.items()):
        print(f"{key}: {value}")


def print_threshold_table(
    prediction_labels: Sequence[str],
    diagnostics: Dict[str, Dict[str, float]],
) -> None:
    print()
    print("=" * 100)
    print("VALIDATION-OPTIMISED THRESHOLDS")
    print("=" * 100)

    heading = (
        f"{'Feature':35s} "
        f"{'Threshold':>10s} "
        f"{'Support':>9s} "
        f"{'F1@0.5':>10s} "
        f"{'F1 opt':>10s} "
        f"{'Change':>10s}"
    )

    print(heading)
    print("-" * len(heading))

    for feature in prediction_labels:
        result = diagnostics[feature]

        print(
            f"{feature:35s} "
            f"{result['threshold']:10.2f} "
            f"{int(result['support']):9d} "
            f"{result['f1_default']:10.4f} "
            f"{result['f1_optimised']:10.4f} "
            f"{result['f1_change']:+10.4f}"
        )


def print_per_feature_table(
    title: str,
    labels: Sequence[str],
    results: Dict[str, Dict[str, float]],
) -> None:
    print()
    print("=" * 80)
    print(title)
    print("=" * 80)

    heading = (
        f"{'Feature':35s} "
        f"{'Precision':>10s} "
        f"{'Recall':>10s} "
        f"{'F1':>10s} "
        f"{'Support':>10s}"
    )

    print(heading)
    print("-" * len(heading))

    for feature in labels:
        result = results[feature]

        print(
            f"{feature:35s} "
            f"{result['precision']:10.4f} "
            f"{result['recall']:10.4f} "
            f"{result['f1']:10.4f} "
            f"{result['support']:10d}"
        )


# =============================================================================
# Main
# =============================================================================

def main() -> None:
    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    set_seed(RANDOM_SEED)

    validate_feature_hierarchy()
    check_cuda()

    # -------------------------------------------------------------------------
    # Load the independent train and test corpora
    # -------------------------------------------------------------------------

    print("=" * 80)
    print("LOADING DATA")
    print("=" * 80)

    train_sentences = load_conllu(TRAIN_CONLLU_PATH)
    test_sentences = load_conllu(TEST_CONLLU_PATH)

    train_tagged_records = load_tagged_records(TRAIN_JSONL_PATH)
    test_tagged_records = load_tagged_records(TEST_JSONL_PATH)

    print(
        "Training CoNLL-U sentences: "
        f"{len(train_sentences):,}"
    )
    print(
        "Test CoNLL-U sentences: "
        f"{len(test_sentences):,}"
    )
    print(
        "Training sentences represented in tagged JSONL: "
        f"{len(train_tagged_records):,}"
    )
    print(
        "Test sentences represented in tagged JSONL: "
        f"{len(test_tagged_records):,}"
    )

    train_target_record_count = sum(
        len(records)
        for records in train_tagged_records.values()
    )
    test_target_record_count = sum(
        len(records)
        for records in test_tagged_records.values()
    )

    print(
        "Training tagged NOUN/PROPN records: "
        f"{train_target_record_count:,}"
    )
    print(
        "Test tagged NOUN/PROPN records: "
        f"{test_target_record_count:,}"
    )
    print()

    # -------------------------------------------------------------------------
    # Build train/validation from TRAIN only; keep TEST untouched
    # -------------------------------------------------------------------------

    eligible_train_indices = get_eligible_sentence_indices(
        train_sentences,
        train_tagged_records,
    )
    eligible_test_indices = get_eligible_sentence_indices(
        test_sentences,
        test_tagged_records,
    )

    print(
        "Eligible training-file sentences with at least one non-empty "
        "feature target: "
        f"{len(eligible_train_indices):,}"
    )
    print(
        "Eligible test-file sentences with at least one non-empty "
        "feature target: "
        f"{len(eligible_test_indices):,}"
    )

    train_validation_split = make_train_validation_split(
        eligible_train_indices,
        train_tagged_records,
    )

    train_indices = train_validation_split["train"]
    validation_indices = train_validation_split["validation"]
    test_indices = list(eligible_test_indices)

    split_sources = {
        "train": (train_tagged_records, train_indices),
        "validation": (train_tagged_records, validation_indices),
        "test": (test_tagged_records, test_indices),
    }

    split_document = {
        "random_seed": RANDOM_SEED,
        "validation_fraction": VALIDATION_FRACTION,
        "train_jsonl": str(TRAIN_JSONL_PATH),
        "test_jsonl": str(TEST_JSONL_PATH),
        "train_conllu": str(TRAIN_CONLLU_PATH),
        "test_conllu": str(TEST_CONLLU_PATH),
        "index_spaces": {
            "train": "training corpus",
            "validation": "training corpus",
            "test": "test corpus",
        },
        "train": train_indices,
        "validation": validation_indices,
        "test": test_indices,
    }

    with SPLIT_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            split_document,
            handle,
            ensure_ascii=False,
            indent=2,
        )

    # The reduction is exact only when the source data are hierarchy-closed.
    verify_hierarchy_closure(split_sources)

    # -------------------------------------------------------------------------
    # Separate complete and neural-output vocabularies
    # -------------------------------------------------------------------------

    # Inventory of independent labels in the entire TRAINING FILE. This is used
    # only as a diagnostic/consistency check; model vocabularies below are still
    # built from the actual training subset.
    training_file_prediction_labels = build_prediction_label_vocabulary(
        train_tagged_records,
        eligible_train_indices,
    )

    # IMPORTANT: both model/evaluation vocabularies are built from the TRAIN
    # subset only.
    full_labels = build_full_label_vocabulary(
        train_tagged_records,
        train_indices,
    )

    prediction_labels = build_prediction_label_vocabulary(
        train_tagged_records,
        train_indices,
    )

    verify_full_label_coverage(
        full_labels,
        split_sources,
    )

    unseen_test_independent_labels = verify_prediction_label_coverage(
        prediction_labels,
        split_sources,
        training_file_prediction_labels=training_file_prediction_labels,
    )

    prediction_label2id = {
        label: index
        for index, label in enumerate(prediction_labels)
    }

    prediction_id2label = {
        index: label
        for label, index in prediction_label2id.items()
    }

    full_label2id = {
        label: index
        for index, label in enumerate(full_labels)
    }

    hierarchy_matrix = build_hierarchy_expansion_matrix(
        prediction_labels,
        full_labels,
    )

    # Keep "labels" / "label2id" / "id2label" as aliases for the MODEL HEADS
    # so existing inference code can still discover the neural output order.
    # The complete post-inheritance vocabulary is stored separately.
    label_document = {
        "labels": prediction_labels,
        "label2id": prediction_label2id,
        "id2label": {
            str(key): value
            for key, value in prediction_id2label.items()
        },
        "prediction_labels": prediction_labels,
        "prediction_label2id": prediction_label2id,
        "full_labels": full_labels,
        "full_label2id": full_label2id,
        "feature_parents": {
            child: sorted(parents)
            for child, parents in sorted(FEATURE_PARENTS.items())
        },
        "target_representation": "per_token_independent_features",
    }

    with LABELS_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            label_document,
            handle,
            ensure_ascii=False,
            indent=2,
        )

    print_split_statistics(split_sources)

    print_prediction_label_statistics(
        prediction_labels,
        full_labels,
        train_tagged_records,
        train_indices,
    )

    # -------------------------------------------------------------------------
    # Tokeniser and datasets
    # -------------------------------------------------------------------------

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        use_fast=True,
        add_prefix_space=True,
    )

    train_dataset = SemanticFeatureDataset(
        sentences=train_sentences,
        sentence_indices=train_indices,
        tagged_records=train_tagged_records,
        tokenizer=tokenizer,
        label2id=prediction_label2id,
        max_length=MAX_LENGTH,
    )

    validation_dataset = SemanticFeatureDataset(
        sentences=train_sentences,
        sentence_indices=validation_indices,
        tagged_records=train_tagged_records,
        tokenizer=tokenizer,
        label2id=prediction_label2id,
        max_length=MAX_LENGTH,
    )

    test_dataset = SemanticFeatureDataset(
        sentences=test_sentences,
        sentence_indices=test_indices,
        tagged_records=test_tagged_records,
        tokenizer=tokenizer,
        label2id=prediction_label2id,
        max_length=MAX_LENGTH,
    )

    data_collator = MultiLabelTokenCollator(
        tokenizer=tokenizer,
        num_labels=len(prediction_labels),
    )

    # -------------------------------------------------------------------------
    # Model
    # -------------------------------------------------------------------------

    model = RobertaMultiLabelModel(
        model_name=MODEL_NAME,
        num_labels=len(prediction_labels),
        id2label=prediction_id2label,
        label2id=prediction_label2id,
    )

    # -------------------------------------------------------------------------
    # Training configuration
    # -------------------------------------------------------------------------

    use_bf16 = torch.cuda.is_bf16_supported()

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),

        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,

        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,

        gradient_accumulation_steps=(
            GRADIENT_ACCUMULATION_STEPS
        ),

        num_train_epochs=NUM_EPOCHS,

        eval_strategy=EVAL_STRATEGY,
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,

        load_best_model_at_end=True,

        # This is the COMPLETE hierarchy-expanded F1 at the fixed 0.5
        # threshold. Threshold optimisation happens only after the best model
        # has been selected, preventing test leakage.
        metric_for_best_model="f1_macro",
        greater_is_better=True,

        save_total_limit=2,

        bf16=use_bf16,
        fp16=not use_bf16,

        dataloader_num_workers=4,
        dataloader_pin_memory=True,

        optim="adamw_torch",

        report_to="none",

        seed=RANDOM_SEED,
        data_seed=RANDOM_SEED,

        remove_unused_columns=False,
        eval_accumulation_steps=16,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=make_compute_metrics(
            prediction_labels,
            hierarchy_matrix,
        ),
    )

    # -------------------------------------------------------------------------
    # Train
    # -------------------------------------------------------------------------

    print("=" * 80)
    print("TRAINING")
    print("=" * 80)

    trainer.train()

    # -------------------------------------------------------------------------
    # Save best model
    # -------------------------------------------------------------------------

    best_model_dir = OUTPUT_DIR / "best_model"

    trainer.save_model(str(best_model_dir))
    tokenizer.save_pretrained(str(best_model_dir))

    # Save the richer semantic metadata inside best_model as well so inference
    # does not need to know the parent output directory.
    with (best_model_dir / "semantic_features.json").open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            label_document,
            handle,
            ensure_ascii=False,
            indent=2,
        )

    # -------------------------------------------------------------------------
    # Validation prediction at default threshold
    # -------------------------------------------------------------------------

    print()
    print("=" * 80)
    print("FINAL VALIDATION RESULTS AT DEFAULT THRESHOLD")
    print("=" * 80)

    validation_output = trainer.predict(
        validation_dataset,
        metric_key_prefix="validation_default",
    )

    print_metric_dictionary(validation_output.metrics)

    validation_probabilities, validation_true_independent = (
        extract_valid_token_arrays(
            validation_output.predictions,
            validation_output.label_ids,
        )
    )

    # -------------------------------------------------------------------------
    # Validation-only per-feature threshold optimisation
    # -------------------------------------------------------------------------

    optimised_thresholds, threshold_diagnostics = (
        optimise_feature_thresholds(
            probabilities=validation_probabilities,
            y_true_independent=validation_true_independent,
            prediction_labels=prediction_labels,
        )
    )

    threshold_vector = make_threshold_vector(
        prediction_labels,
        optimised_thresholds,
    )

    print_threshold_table(
        prediction_labels,
        threshold_diagnostics,
    )

    validation_optimised_metrics = (
        calculate_hierarchical_metrics_from_probabilities(
            probabilities=validation_probabilities,
            y_true_independent=validation_true_independent,
            threshold_vector=threshold_vector,
            hierarchy_matrix=hierarchy_matrix,
        )
    )

    print()
    print("=" * 80)
    print("FINAL VALIDATION RESULTS WITH OPTIMISED THRESHOLDS")
    print("=" * 80)
    print_metric_dictionary(validation_optimised_metrics)

    threshold_document = {
        "optimised_on": "validation",
        "objective": "per_feature_binary_f1",
        "default_threshold": DEFAULT_PREDICTION_THRESHOLD,
        "grid": {
            "minimum": THRESHOLD_MIN,
            "maximum": THRESHOLD_MAX,
            "step": THRESHOLD_STEP,
        },
        "thresholds": optimised_thresholds,
        "diagnostics": threshold_diagnostics,
    }

    with THRESHOLDS_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            threshold_document,
            handle,
            ensure_ascii=False,
            indent=2,
        )

    with (best_model_dir / "prediction_thresholds.json").open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            threshold_document,
            handle,
            ensure_ascii=False,
            indent=2,
        )

    # -------------------------------------------------------------------------
    # Test: thresholds are now frozen
    # -------------------------------------------------------------------------

    print()
    print("=" * 80)
    print("TEST RESULTS WITH VALIDATION-OPTIMISED THRESHOLDS")
    print("=" * 80)

    test_output = trainer.predict(
        test_dataset,
        metric_key_prefix="test_default",
    )

    test_probabilities, test_true_independent = (
        extract_valid_token_arrays(
            test_output.predictions,
            test_output.label_ids,
        )
    )

    # IMPORTANT: build complete gold truth from the held-out source records, not
    # by expanding the training-derived neural target matrix.  The latter would
    # silently erase a test-only direct independent feature such as Human.
    test_true_full = build_full_truth_matrix_for_dataset(
        test_dataset,
        full_labels,
    )

    if test_true_full.shape[0] != test_probabilities.shape[0]:
        raise RuntimeError(
            "Test source-truth alignment failed: model predictions contain "
            f"{test_probabilities.shape[0]:,} supervised tokens but source "
            f"truth contains {test_true_full.shape[0]:,}."
        )

    test_optimised_metrics = (
        calculate_hierarchical_metrics_from_probabilities(
            probabilities=test_probabilities,
            y_true_independent=test_true_independent,
            threshold_vector=threshold_vector,
            hierarchy_matrix=hierarchy_matrix,
            y_true_full=test_true_full,
        )
    )

    # Preserve test loss/runtime information from Trainer, but do not use its
    # default-threshold F1 values as the final headline scores.
    test_auxiliary_metrics = {
        key: value
        for key, value in test_output.metrics.items()
        if (
            key.endswith("loss")
            or key.endswith("runtime")
            or "samples_per_second" in key
            or "steps_per_second" in key
        )
    }

    final_test_metrics: Dict[str, float] = {}
    final_test_metrics.update(test_optimised_metrics)
    final_test_metrics.update(test_auxiliary_metrics)

    print_metric_dictionary(final_test_metrics)

    # -------------------------------------------------------------------------
    # Detailed test metrics: complete hierarchy
    # -------------------------------------------------------------------------

    test_pred_independent = threshold_probabilities(
        test_probabilities,
        threshold_vector,
    )

    test_pred_full = expand_binary_feature_matrix(
        test_pred_independent,
        hierarchy_matrix,
    )

    per_feature_full = calculate_per_feature_metrics_from_binary(
        y_true=test_true_full,
        y_pred=test_pred_full,
        labels=full_labels,
    )

    per_feature_independent = (
        calculate_per_feature_metrics_from_binary(
            y_true=test_true_independent,
            y_pred=test_pred_independent,
            labels=prediction_labels,
        )
    )

    print_per_feature_table(
        title="TEST PERFORMANCE BY COMPLETE FEATURE (AFTER INHERITANCE)",
        labels=full_labels,
        results=per_feature_full,
    )

    print_per_feature_table(
        title="TEST PERFORMANCE BY INDEPENDENT NEURAL OUTPUT",
        labels=prediction_labels,
        results=per_feature_independent,
    )

    # -------------------------------------------------------------------------
    # Save final results
    # -------------------------------------------------------------------------

    result_document = {
        "model": MODEL_NAME,
        "target_representation": "per_token_independent_features",
        "data_sources": {
            "train_jsonl": str(TRAIN_JSONL_PATH),
            "test_jsonl": str(TEST_JSONL_PATH),
            "train_conllu": str(TRAIN_CONLLU_PATH),
            "test_conllu": str(TEST_CONLLU_PATH),
        },
        "validation_fraction": VALIDATION_FRACTION,
        "split_sizes": {
            "train": len(train_indices),
            "validation": len(validation_indices),
            "test": len(test_indices),
        },
        "prediction_labels": prediction_labels,
        "full_labels": full_labels,
        "thresholds_optimised_on": "validation",
        "thresholds": optimised_thresholds,
        "unseen_test_independent_labels": unseen_test_independent_labels,
        "validation_metrics_optimised": validation_optimised_metrics,
        "test_metrics": {
            key: float(value)
            if isinstance(
                value,
                (float, int, np.floating, np.integer),
            )
            else value
            for key, value in final_test_metrics.items()
        },
        "per_feature_complete": per_feature_full,
        "per_feature_independent": per_feature_independent,
    }

    with TEST_RESULTS_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            result_document,
            handle,
            ensure_ascii=False,
            indent=2,
        )

    print()
    print(
        "Best model saved to:\n"
        f"  {best_model_dir.resolve()}"
    )
    print(
        "Semantic feature metadata saved to:\n"
        f"  {LABELS_PATH.resolve()}"
    )
    print(
        "Validation-optimised thresholds saved to:\n"
        f"  {THRESHOLDS_PATH.resolve()}"
    )
    print(
        "Test results saved to:\n"
        f"  {TEST_RESULTS_PATH.resolve()}"
    )


if __name__ == "__main__":
    main()


CUDA
PyTorch version: 2.13.0+cu130
CUDA available: True
CUDA runtime: 13.0
GPU: NVIDIA GeForce RTX 5070 Ti
VRAM: 15.92 GiB

LOADING DATA
Training CoNLL-U sentences: 12,544
Test CoNLL-U sentences: 2,077
Training sentences represented in tagged JSONL: 11,511
Test sentences represented in tagged JSONL: 1,815
Training tagged NOUN/PROPN records: 47,371
Test tagged NOUN/PROPN records: 6,198

Eligible training-file sentences with at least one non-empty feature target: 11,198
Eligible test-file sentences with at least one non-empty feature target: 1,759

Label-aware train/validation split:
  Target validation sentences : 2,240
  Actual validation sentences : 2,240
  Training sentences retained : 8,958
  Candidates blocked to preserve rare labels: 0

HELD-OUT TEST LABEL COVERAGE WARNING
The test corpus contains direct independent targets that have no training-derived neural output head. They will NOT be added to the model vocabulary. Complete test truth will be read directly from the source ann

Loading weights: 100%|███████████████████████████████████████████████████████████████| 389/389 [00:01<00:00, 211.48it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TRAINING


Epoch,Training Loss,Validation Loss,Precision Micro,Recall Micro,F1 Micro,Precision Macro,Recall Macro,F1 Macro,Direct Precision Micro,Direct Recall Micro,Direct F1 Micro,Direct Precision Macro,Direct Recall Macro,Direct F1 Macro
1,0.098583,0.042019,0.925701,0.871003,0.897519,0.813047,0.708419,0.745206,0.863875,0.804102,0.832918,0.711073,0.603502,0.639731
2,0.069303,0.034151,0.934478,0.899058,0.916426,0.885960,0.811066,0.826765,0.884322,0.838625,0.860868,0.851846,0.735821,0.759072
3,0.049718,0.031218,0.942397,0.907675,0.924710,0.911340,0.824152,0.856556,0.893892,0.857748,0.875447,0.870322,0.755100,0.794925
4,0.035419,0.030239,0.938225,0.924365,0.931243,0.884449,0.863166,0.870540,0.887586,0.878257,0.882897,0.850044,0.800245,0.815664
5,0.030127,0.030690,0.938918,0.928946,0.933905,0.890798,0.876755,0.882041,0.888651,0.880155,0.884383,0.860199,0.818130,0.830292
6,0.023923,0.030561,0.938964,0.931894,0.935416,0.893076,0.878910,0.884487,0.889680,0.884096,0.886879,0.861553,0.821888,0.833120



FINAL VALIDATION RESULTS AT DEFAULT THRESHOLD


validation_default_direct_f1_macro: 0.8331195135484581
validation_default_direct_f1_micro: 0.8868794845511788
validation_default_direct_precision_macro: 0.8615530128614457
validation_default_direct_precision_micro: 0.889680499449137
validation_default_direct_recall_macro: 0.8218884850154972
validation_default_direct_recall_micro: 0.8840960513831108
validation_default_f1_macro: 0.8844869677843514
validation_default_f1_micro: 0.9354157387318783
validation_default_loss: 0.03056052140891552
validation_default_precision_macro: 0.8930759934416468
validation_default_precision_micro: 0.9389638860043844
validation_default_recall_macro: 0.8789098638809884
validation_default_recall_micro: 0.93189430591738
validation_default_runtime: 9.6086
validation_default_samples_per_second: 233.124
validation_default_steps_per_second: 14.57

VALIDATION-OPTIMISED THRESHOLDS
Feature                              Threshold   Support     F1@0.5     F1 opt     Change
------------------------------------------------

direct_f1_macro: 0.7414059148033084
direct_f1_micro: 0.8567555354722096
direct_precision_macro: 0.7619784949628814
direct_precision_micro: 0.8559819413092551
direct_recall_macro: 0.7398710114973831
direct_recall_micro: 0.8575305291723202
f1_macro: 0.8188039952823036
f1_micro: 0.9128899744643055
precision_macro: 0.8363900875973416
precision_micro: 0.9199373461624525
recall_macro: 0.8085199684837404
recall_micro: 0.9059497576024681
test_default_loss: 0.04044955223798752
test_default_runtime: 11.6262
test_default_samples_per_second: 151.297
test_default_steps_per_second: 9.461

TEST PERFORMANCE BY COMPLETE FEATURE (AFTER INHERITANCE)
Feature                              Precision     Recall         F1    Support
-------------------------------------------------------------------------------
Abstract                                0.9524     0.9330     0.9426       3091
Animal                                  0.8929     0.9804     0.9346         51
Animate                                 0

<h1>Adding stratified splitting, repetitions, feature threshold calibration, bootstrapping, k-fold cross validation, and feature switches</h1>

In [1]:
#!/usr/bin/env python3
"""
Fine-tune RoBERTa for hierarchical multi-label semantic feature tagging of
NOUN/PROPN tokens.

Compatible with Python 3.9.21.

Key design changes
------------------
1. The model is trained only on independent semantic features. For each token,
   any feature that can be inherited from a more specific present feature is
   removed from the neural target. This is a per-token reduction rather than a
   global "terminal leaf only" rule, so features such as Human, Time, Shape,
   Communication, etc. can still be predicted directly when they are the most
   specific available feature for that token.

2. All inherited features are reconstructed deterministically from
   FEATURE_PARENTS for validation, testing, reporting, and later inference.

3. A separate sigmoid threshold is optimised for every neural output using the
   validation split only. The selected thresholds are frozen before the test
   split is evaluated and are saved to prediction_thresholds.json.

4. Train/validation splitting is performed at SENTENCE level with iterative
   multilabel stratification over sentence-level independent-feature presence.
   The split is fixed across repeated fine-tuning runs.

5. Fine-tuning is repeated over a predeclared list of random seeds. The same
   split is used for every run, and results are summarised with mean and SD.

6. The official test corpus may contain a direct independent target that never
   occurred independently in training. Such a label is never added from test
   data. Complete test truth is read directly from source annotations.

7. Held-out test uncertainty is estimated with a SENTENCE-CLUSTER bootstrap.
   All supervised noun/proper-noun targets from a sampled sentence move together.

Tagged data:
    Training/validation:
        ewt_train_wordnet_wsd_features_wordnet_fixed.jsonl
    Test:
        ewt_test_wordnet_wsd_features_wordnet_fixed.jsonl

Sentence/token sources:
    Training/validation:
        UD_English-EWT-master/
        UD_English-EWT-master/
        en_ewt-ud-train.conllu
    Test:
        UD_English-EWT-master/
        UD_English-EWT-master/
        en_ewt-ud-test.conllu

The validation split is created only from the training corpus. The official
test corpus remains untouched until final evaluation.

Loss:
    BCEWithLogitsLoss over INDEPENDENT feature labels only.

Headline metrics:
    Computed after deterministic hierarchy expansion, so they describe the
    complete semantic feature sets rather than only the neural outputs.

Requirements:
    python -m pip install torch transformers datasets accelerate conllu scikit-learn iterative-stratification

CUDA is required.
"""

from __future__ import annotations

import gc
import json
import shutil
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Set, Tuple

import numpy as np
import torch
import torch.nn as nn

from conllu import parse_incr
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import precision_recall_fscore_support
from torch.utils.data import Dataset

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)


# =============================================================================
# Configuration
# =============================================================================

TRAIN_JSONL_PATH = Path(
    "../data/ewt_train_wordnet_wsd_features_wordnet_fixed.jsonl"
    #"ewt_train_wordnet_wsd_features_fully_tagged_llm_reviewed.jsonl"
)

TEST_JSONL_PATH = Path(
    "../data/ewt_test_wordnet_wsd_features_wordnet_fixed.jsonl"
    #"ewt_test_wordnet_wsd_features_fully_tagged_llm_reviewed.jsonl"
)

TRAIN_CONLLU_PATH = Path(
    "../data/UD_English-EWT-master/"
    "UD_English-EWT-master/"
    "en_ewt-ud-train.conllu"
)

TEST_CONLLU_PATH = Path(
    "../data/UD_English-EWT-master/"
    "UD_English-EWT-master/"
    "en_ewt-ud-test.conllu"
)

# MODEL_NAME = "FacebookAI/roberta-base"
MODEL_NAME = "FacebookAI/roberta-large"

OUTPUT_DIR = Path("../models/roberta_semantic_features")

# Validation is sampled only from eligible TRAINING sentences.
# Stratification is performed at SENTENCE level using the multilabel feature
# presence vector for each sentence. The separate official test file is never
# split and is used only for final evaluation.
VALIDATION_FRACTION = 0.20

# Fixed seed used ONLY to create the sentence-level train/validation split.
# Keeping this fixed across model runs means repeated runs measure stochastic
# fine-tuning variation rather than variation in the validation partition.
SPLIT_SEED = 42

# Repeated fine-tuning seeds. Use the same list when comparing model variants.
# Thirty runs follow the experiment-design recommendation; reduce this list if
# RoBERTa-large makes 30 runs computationally impractical, and report the actual
# number used.
RUN_SEEDS = [
    11, 23, 37, 53, 71, 89, 107, 127, 149, 173,
    197, 223, 251, 281, 313, 347, 383, 421, 461, 503,
    547, 593, 641, 691, 743, 797, 853, 911, 971, 1031,
]

# The multilabel stratifier is retried with deterministic derived seeds until
# every independent feature still has at least this many positive token targets
# in the TRAIN subset. This prevents a rare direct target from disappearing.
MIN_TRAIN_POSITIVES_PER_FEATURE = 1
MAX_STRATIFICATION_ATTEMPTS = 200

# Sentence-cluster bootstrap for uncertainty in the held-out test sample.
# The same sampled test sentences are used for every run within each bootstrap
# replicate, and the statistic is the mean metric across run seeds.
TEST_BOOTSTRAP_SAMPLES = 2000
TEST_BOOTSTRAP_CONFIDENCE = 0.95
TEST_BOOTSTRAP_SEED = 20260926

# Saving 30 RoBERTa-large models would use a great deal of disk space. The first
# run is therefore saved as a neutral REFERENCE model; it is not selected for
# having the best test result. Metrics and thresholds are saved for every run.
SAVE_REFERENCE_MODEL = True

MAX_LENGTH = 256

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16

GRADIENT_ACCUMULATION_STEPS = 2

NUM_EPOCHS = 5

# Threshold used during epoch-by-epoch Trainer evaluation. Final thresholds are
# optimised per independent feature on the validation set after training.
DEFAULT_PREDICTION_THRESHOLD = 0.5

# Validation-only threshold search grid.
THRESHOLD_MIN = 0.05
THRESHOLD_MAX = 0.95
THRESHOLD_STEP = 0.01

# Only these POS categories receive feature supervision.
TARGET_POS = {"NOUN", "PROPN"}

# A NOUN/PROPN with no features is considered a failed source annotation rather
# than a valid all-negative target.
EXCLUDE_EMPTY_FEATURE_RECORDS = True

# Optional per-feature switches. Any feature not listed here is enabled.
# Set a feature to False to remove it from:
#   - sentence-level stratification,
#   - neural training targets,
#   - threshold calibration,
#   - hierarchy-expanded gold/predicted feature sets,
#   - per-feature and aggregate evaluation, and
#   - the sentence-cluster bootstrap.
#
# Disabled intermediate hierarchy nodes are skipped but their enabled ancestors
# are still inherited. For example, if Human were disabled, Feminine would still
# inherit Animate/Natural/Concrete/Internal_Control via the hierarchy.
FEATURE_SWITCHES: Dict[str, bool] = {
    "State": False,
    "State_of_Matter": False,
    "Shape": False,
    "Rigidity": False,
}

# Aggregate per-feature test tables are always printed after all repeated runs.
# Set this to True only if you also want the two full per-feature tables printed
# after EVERY run (very verbose for 30 runs).
PRINT_PER_RUN_FEATURE_TABLES = False

EVAL_STRATEGY = "epoch"

SPLIT_PATH = OUTPUT_DIR / "sentence_split.json"
LABELS_PATH = OUTPUT_DIR / "semantic_features.json"
RUNS_DIR = OUTPUT_DIR / "runs"
REFERENCE_MODEL_DIR = OUTPUT_DIR / "reference_model"
REPEATED_RESULTS_PATH = OUTPUT_DIR / "repeated_run_results.json"
BOOTSTRAP_RESULTS_PATH = OUTPUT_DIR / "test_bootstrap_confidence_intervals.json"


# =============================================================================
# Feature hierarchy
# =============================================================================

def is_feature_enabled(feature: str) -> bool:
    """Return whether *feature* participates in this experiment."""

    return bool(FEATURE_SWITCHES.get(feature, True))


def validate_feature_switches(
    train_tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    test_tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> None:
    """Fail early on misspelled feature-switch names and report disabled labels."""

    observed: Set[str] = set()

    for source in (train_tagged_records, test_tagged_records):
        for sentence_records in source.values():
            for record in sentence_records.values():
                raw = record.get("features", [])
                if isinstance(raw, list):
                    observed.update(
                        str(feature).strip()
                        for feature in raw
                        if str(feature).strip()
                    )

    known = set(observed) | set(FEATURE_PARENTS)
    for parents in FEATURE_PARENTS.values():
        known.update(parents)

    unknown = sorted(set(FEATURE_SWITCHES) - known)
    if unknown:
        raise ValueError(
            "FEATURE_SWITCHES contains unknown feature name(s): "
            + ", ".join(unknown)
        )

    disabled = sorted(
        feature for feature, enabled in FEATURE_SWITCHES.items() if not enabled
    )

    print("=" * 80)
    print("FEATURE SWITCHES")
    print("=" * 80)
    if disabled:
        print("Disabled features: " + ", ".join(disabled))
    else:
        print("Disabled features: none")
    print("All unlisted features are enabled.")
    print()


# child -> direct parent(s)
FEATURE_PARENTS: Dict[str, Set[str]] = {
    "Organisation": {
        "Internal_Control",
        "Abstract",
    },
    "Natural_Force": {
        "Internal_Control",
    },
    "Animate": {
        "Internal_Control",
        "Natural",
    },
    "Human": {
        "Animate",
    },
    "Feminine": {
        "Human",
    },
    "Masculine": {
        "Human",
    },
    "Unspecified_Gender_or_Neuter": {
        "Human",
    },
    "Animal": {
        "Animate",
    },
    "Plant": {
        "Animate",
    },
    "Body_Part": {
        "Animate",
    },
    "Machine": {
        "Internal_Control",
        "Artefact",
    },
    "Vehicle": {
        "Machine",
    },
    "Natural": {
        "Concrete",
    },
    "Artefact": {
        "Concrete",
    },
    "Comestible": {
        "Concrete",
    },
    "Substance": {
        "Concrete",
    },
    "State_of_Matter": {
        "Concrete",
    },
    "Inanimate": {
        "Concrete",
    },
    "Garment": {
        "Artefact",
    },
    "Solid": {
        "State_of_Matter",
    },
    "Fluid": {
        "State_of_Matter",
    },
    "Rigidity": {
        "Solid",
    },
    "Shape": {
        "Solid",
    },
    "Rigid": {
        "Rigidity",
    },
    "Non_Rigid": {
        "Rigidity",
    },
    "Elongated": {
        "Shape",
    },
    "Pointy": {
        "Shape",
    },
    "Point_in_Time": {
        "Time",
    },
    "Duration": {
        "Time",
    },
    "Time": {
        "Abstract",
    },
    "Language": {
        "Abstract",
    },
    "Sound": {
        "Abstract",
    },
    "Communication": {
        "Abstract",
    },
    "Eventive": {
        "Abstract",
    },
    "Statement": {
        "Communication",
    },
    "Question": {
        "Communication",
    },
}


def validate_feature_hierarchy() -> None:
    """Fail early if FEATURE_PARENTS contains a cycle or invalid name."""

    all_features: Set[str] = set(FEATURE_PARENTS)

    for parents in FEATURE_PARENTS.values():
        all_features.update(parents)

    for feature in all_features:
        if not isinstance(feature, str) or not feature.strip():
            raise ValueError(
                "FEATURE_PARENTS contains an empty or non-string feature name."
            )

    visiting: Set[str] = set()
    visited: Set[str] = set()

    def visit(feature: str) -> None:
        if feature in visited:
            return

        if feature in visiting:
            raise ValueError(
                f"Cycle detected in FEATURE_PARENTS involving {feature!r}."
            )

        visiting.add(feature)

        for parent in FEATURE_PARENTS.get(feature, set()):
            visit(parent)

        visiting.remove(feature)
        visited.add(feature)

    for feature in sorted(all_features):
        visit(feature)


@lru_cache(maxsize=None)
def get_feature_ancestors(feature: str) -> frozenset:
    """Return every direct and indirect parent of *feature*."""

    ancestors: Set[str] = set()

    for parent in FEATURE_PARENTS.get(feature, set()):
        if is_feature_enabled(parent):
            ancestors.add(parent)
        # Traverse even through a disabled intermediate node so enabled
        # grandparents remain inherited.
        ancestors.update(get_feature_ancestors(parent))

    return frozenset(ancestors)


def expand_feature_hierarchy(features: Iterable[str]) -> Set[str]:
    """Add every deterministic ancestor required by FEATURE_PARENTS."""

    expanded: Set[str] = {
        feature for feature in features if is_feature_enabled(feature)
    }

    for feature in list(expanded):
        expanded.update(get_feature_ancestors(feature))

    return expanded


def reduce_to_independent_features(features: Iterable[str]) -> Set[str]:
    """
    Reduce a complete feature set to the features that must be predicted.

    A feature is redundant when it is an ancestor of another feature present
    for the same token. Such an ancestor can be reconstructed deterministically.

    Examples:
        {Human, Animate, Natural, Concrete, Internal_Control}
            -> {Human}

        {Feminine, Human, Animate, Natural, Concrete, Internal_Control}
            -> {Feminine}

        {Statement, Communication, Abstract, Time}
            -> {Statement, Time}

    This is deliberately PER TOKEN. It is not the same as training only labels
    that are global terminal leaves in the ontology. A generic Human can still
    be a direct neural target if no more specific Human child is present.
    """

    feature_set = set(features)

    inherited_from_other_features: Set[str] = set()

    for feature in feature_set:
        inherited_from_other_features.update(
            get_feature_ancestors(feature)
        )

    return feature_set - inherited_from_other_features


def missing_hierarchy_features(features: Iterable[str]) -> Set[str]:
    """Return ancestors required by the hierarchy but absent from *features*."""

    feature_set = set(features)
    return expand_feature_hierarchy(feature_set) - feature_set


# =============================================================================
# CUDA
# =============================================================================

def check_cuda() -> None:
    """Require CUDA rather than silently falling back to the CPU."""

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. This script is configured to require "
            "a CUDA-capable GPU."
        )

    print("=" * 80)
    print("CUDA")
    print("=" * 80)
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"CUDA runtime: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    props = torch.cuda.get_device_properties(0)

    print(
        "VRAM: "
        f"{props.total_memory / (1024 ** 3):.2f} GiB"
    )
    print()


# =============================================================================
# Load CoNLL-U
# =============================================================================

def load_conllu(path: Path) -> List[Any]:
    """Load all CoNLL-U sentences."""

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find CoNLL-U file:\n{path.resolve()}"
        )

    sentences: List[Any] = []

    with path.open("r", encoding="utf-8") as handle:
        for sentence in parse_incr(handle):
            sentences.append(sentence)

    return sentences


def get_sentence_words(sentence: Any) -> Tuple[List[str], List[int]]:
    """
    Return ordinary integer-ID tokens and their original CoNLL-U token IDs.

    Multiword-token placeholder rows and empty nodes are excluded.
    """

    words: List[str] = []
    token_ids: List[int] = []

    for token in sentence:
        token_id = token.get("id")

        if not isinstance(token_id, int):
            continue

        form = token.get("form")

        if form is None:
            continue

        words.append(str(form))
        token_ids.append(token_id)

    return words, token_ids


# =============================================================================
# Load tagged JSONL
# =============================================================================

def load_tagged_records(
    path: Path,
) -> Dict[int, Dict[int, Dict[str, Any]]]:
    """
    Return:

        sentence_index
            -> token_id
                -> JSONL record

    Only NOUN and PROPN records are retained.
    """

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find tagged JSONL:\n{path.resolve()}"
        )

    records: Dict[int, Dict[int, Dict[str, Any]]] = defaultdict(dict)

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                print(
                    "Warning: invalid JSON on line "
                    f"{line_number}: {exc}"
                )
                continue

            if record.get("upos") not in TARGET_POS:
                continue

            sentence_index = record.get("sentence_index")
            token_id = record.get("token_id")

            if not isinstance(sentence_index, int):
                continue

            if not isinstance(token_id, int):
                continue

            records[sentence_index][token_id] = record

    return dict(records)


# =============================================================================
# Labels
# =============================================================================

def clean_features(record: Dict[str, Any]) -> List[str]:
    """Return unique, non-empty COMPLETE feature names from a JSONL record."""

    raw = record.get("features", [])

    if not isinstance(raw, list):
        return []

    return sorted(
        {
            str(feature).strip()
            for feature in raw
            if (
                str(feature).strip()
                and is_feature_enabled(str(feature).strip())
            )
        }
    )


def get_independent_record_features(record: Dict[str, Any]) -> List[str]:
    """Return the per-token minimal feature set that the model must predict."""

    return sorted(
        reduce_to_independent_features(
            clean_features(record)
        )
    )


def build_full_label_vocabulary(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    training_sentence_indices: Sequence[int],
) -> List[str]:
    """
    Build the COMPLETE evaluation vocabulary from training data only.

    Deterministic ancestors are included even if a training record omitted one,
    although hierarchy closure is separately verified and should normally make
    this addition unnecessary.
    """

    labels: Set[str] = set()

    for sentence_index in training_sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            features = clean_features(record)
            labels.update(features)
            labels.update(expand_feature_hierarchy(features))

    if not labels:
        raise RuntimeError(
            "No semantic features were found in the training split."
        )

    return sorted(labels)


def build_prediction_label_vocabulary(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    training_sentence_indices: Sequence[int],
) -> List[str]:
    """Build the neural-output vocabulary from independent training targets."""

    labels: Set[str] = set()

    for sentence_index in training_sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            labels.update(
                get_independent_record_features(record)
            )

    if not labels:
        raise RuntimeError(
            "No independent semantic features were found in the training split."
        )

    return sorted(labels)


# =============================================================================
# Sentence selection and splitting
# =============================================================================

def get_eligible_sentence_indices(
    sentences: Sequence[Any],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> List[int]:
    """Eligible sentences contain at least one non-empty NOUN/PROPN target."""

    eligible: List[int] = []

    for sentence_index in range(len(sentences)):
        token_records = tagged_records.get(sentence_index, {})

        valid_records = [
            record
            for record in token_records.values()
            if clean_features(record)
        ]

        if valid_records:
            eligible.append(sentence_index)

    return eligible


def get_sentence_independent_feature_counts(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_index: int,
) -> Counter:
    """Count independent positive token targets in one source sentence."""

    counts: Counter = Counter()

    for record in tagged_records.get(sentence_index, {}).values():
        if not clean_features(record):
            continue

        for feature in get_independent_record_features(record):
            counts[feature] += 1

    return counts


def build_sentence_multilabel_matrix(
    eligible_indices: Sequence[int],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> Tuple[np.ndarray, List[str], Dict[int, Counter]]:
    """
    Build one multilabel stratification row per SENTENCE.

    y_sentence[sentence, feature] is 1 when at least one eligible NOUN/PROPN
    token in that sentence has the independent feature. Token counts are kept
    separately for the post-split training-coverage constraint.
    """

    sentence_feature_counts: Dict[int, Counter] = {
        sentence_index: get_sentence_independent_feature_counts(
            tagged_records,
            sentence_index,
        )
        for sentence_index in eligible_indices
    }

    stratification_labels = sorted(
        {
            feature
            for counts in sentence_feature_counts.values()
            for feature, count in counts.items()
            if count > 0
        }
    )

    if not stratification_labels:
        raise RuntimeError(
            "No independent semantic features were found in eligible "
            "training sentences."
        )

    label2column = {
        label: column
        for column, label in enumerate(stratification_labels)
    }

    matrix = np.zeros(
        (len(eligible_indices), len(stratification_labels)),
        dtype=np.int8,
    )

    for row, sentence_index in enumerate(eligible_indices):
        for feature in sentence_feature_counts[sentence_index]:
            matrix[row, label2column[feature]] = 1

    return matrix, stratification_labels, sentence_feature_counts


def _sentence_prevalence_error(
    y_all: np.ndarray,
    validation_rows: np.ndarray,
    validation_fraction: float,
) -> float:
    """
    Score how closely validation sentence-level label prevalences match target.

    Lower is better. The score is used only to choose among deterministic
    iterative-stratification attempts that satisfy the training-coverage rule.
    """

    total_support = y_all.sum(axis=0).astype(np.float64)
    validation_support = y_all[validation_rows].sum(axis=0).astype(np.float64)

    target_support = total_support * float(validation_fraction)

    # Scale by at least one expected example so rare and common labels both
    # contribute without allowing extremely common labels to dominate.
    scale = np.maximum(target_support, 1.0)

    return float(
        np.mean(
            np.abs(validation_support - target_support) / scale
        )
    )


def make_train_validation_split(
    eligible_indices: Sequence[int],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> Dict[str, Any]:
    """
    Create a SENTENCE-LEVEL multilabel-stratified split from TRAINING only.

    Each sentence receives a binary vector indicating which independent feature
    labels occur anywhere among its eligible NOUN/PROPN tokens. Iterative
    multilabel stratification then assigns whole sentences to train/validation.

    The split is retried deterministically when necessary so that every
    independent feature retains at least MIN_TRAIN_POSITIVES_PER_FEATURE
    positive TOKEN targets in the training subset.

    The official test corpus is deliberately not accepted by this function, so
    test labels cannot influence the split.
    """

    if not 0.0 < VALIDATION_FRACTION < 1.0:
        raise ValueError(
            "VALIDATION_FRACTION must be strictly between 0 and 1."
        )

    if MIN_TRAIN_POSITIVES_PER_FEATURE < 1:
        raise ValueError(
            "MIN_TRAIN_POSITIVES_PER_FEATURE must be at least 1."
        )

    if MAX_STRATIFICATION_ATTEMPTS < 1:
        raise ValueError(
            "MAX_STRATIFICATION_ATTEMPTS must be at least 1."
        )

    eligible_indices = list(eligible_indices)

    if len(eligible_indices) < 2:
        raise RuntimeError(
            "Need at least two eligible training sentences to create a "
            "train/validation split."
        )

    (
        sentence_matrix,
        stratification_labels,
        sentence_feature_counts,
    ) = build_sentence_multilabel_matrix(
        eligible_indices,
        tagged_records,
    )

    all_token_counts: Counter = Counter()

    for counts in sentence_feature_counts.values():
        all_token_counts.update(counts)

    x_dummy = np.zeros((len(eligible_indices), 1), dtype=np.int8)

    best_candidate: Optional[Dict[str, Any]] = None
    rejected_for_coverage = 0

    for attempt in range(MAX_STRATIFICATION_ATTEMPTS):
        attempt_seed = SPLIT_SEED + attempt

        splitter = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=VALIDATION_FRACTION,
            random_state=attempt_seed,
        )

        train_rows, validation_rows = next(
            splitter.split(x_dummy, sentence_matrix)
        )

        train_indices = [
            eligible_indices[int(row)]
            for row in train_rows
        ]
        validation_indices = [
            eligible_indices[int(row)]
            for row in validation_rows
        ]

        train_token_counts: Counter = Counter()

        for sentence_index in train_indices:
            train_token_counts.update(
                sentence_feature_counts[sentence_index]
            )

        missing_or_undercovered = sorted(
            feature
            for feature, total_count in all_token_counts.items()
            if (
                total_count > 0
                and train_token_counts[feature]
                < MIN_TRAIN_POSITIVES_PER_FEATURE
            )
        )

        if missing_or_undercovered:
            rejected_for_coverage += 1
            continue

        prevalence_error = _sentence_prevalence_error(
            sentence_matrix,
            validation_rows,
            VALIDATION_FRACTION,
        )

        candidate = {
            "train": train_indices,
            "validation": validation_indices,
            "split_seed_used": attempt_seed,
            "attempt": attempt + 1,
            "prevalence_error": prevalence_error,
            "stratification_labels": stratification_labels,
        }

        if (
            best_candidate is None
            or prevalence_error < best_candidate["prevalence_error"]
        ):
            best_candidate = candidate

        # Iterative stratification is already designed to balance multilabel
        # prevalence. Once a valid candidate is found, further attempts are not
        # needed unless coverage forced earlier candidates to be rejected.
        if rejected_for_coverage == 0:
            break

    if best_candidate is None:
        raise RuntimeError(
            "Could not create a sentence-level multilabel-stratified "
            "train/validation split that preserved the required training "
            f"support after {MAX_STRATIFICATION_ATTEMPTS} attempts."
        )

    train_indices = best_candidate["train"]
    validation_indices = best_candidate["validation"]

    train_set = set(train_indices)
    validation_set = set(validation_indices)

    if train_set & validation_set:
        raise RuntimeError(
            "Internal split error: at least one sentence occurs in both "
            "training and validation."
        )

    if train_set | validation_set != set(eligible_indices):
        raise RuntimeError(
            "Internal split error: sentence-level split does not exactly cover "
            "the eligible training sentences."
        )

    print()
    print("Sentence-level multilabel-stratified train/validation split:")
    print(
        f"  Eligible training sentences : {len(eligible_indices):,}"
    )
    print(
        f"  Target validation sentences : "
        f"{round(len(eligible_indices) * VALIDATION_FRACTION):,}"
    )
    print(
        f"  Actual validation sentences : {len(validation_indices):,}"
    )
    print(
        f"  Training sentences retained : {len(train_indices):,}"
    )
    print(
        f"  Split seed used             : "
        f"{best_candidate['split_seed_used']}"
    )
    print(
        f"  Stratification labels       : "
        f"{len(stratification_labels):,}"
    )
    print(
        f"  Coverage-rejected attempts  : {rejected_for_coverage:,}"
    )
    print(
        f"  Mean relative prevalence error: "
        f"{best_candidate['prevalence_error']:.6f}"
    )

    # Show sentence-level prevalence agreement, which is the quantity actually
    # controlled by the stratifier.
    validation_fraction_actual = (
        len(validation_indices) / len(eligible_indices)
    )

    label2column = {
        label: column
        for column, label in enumerate(stratification_labels)
    }

    row_by_sentence = {
        sentence_index: row
        for row, sentence_index in enumerate(eligible_indices)
    }

    train_rows_lookup = [
        row_by_sentence[index]
        for index in train_indices
    ]
    validation_rows_lookup = [
        row_by_sentence[index]
        for index in validation_indices
    ]

    print()
    print(
        f"{'Feature':35s} "
        f"{'All sent.':>10s} "
        f"{'Train':>10s} "
        f"{'Valid':>10s} "
        f"{'Valid %':>10s}"
    )
    print("-" * 80)

    for feature in stratification_labels:
        column = label2column[feature]
        all_support = int(sentence_matrix[:, column].sum())
        train_support = int(
            sentence_matrix[train_rows_lookup, column].sum()
        )
        validation_support = int(
            sentence_matrix[validation_rows_lookup, column].sum()
        )

        print(
            f"{feature:35s} "
            f"{all_support:10d} "
            f"{train_support:10d} "
            f"{validation_support:10d} "
            f"{validation_support / max(all_support, 1):10.3f}"
        )

    print(
        f"\nActual validation sentence fraction: "
        f"{validation_fraction_actual:.4f}"
    )
    print()

    return best_candidate


# =============================================================================
# Dataset
# =============================================================================

class SemanticFeatureDataset(Dataset):
    """
    RoBERTa multi-label token-classification dataset.

    IMPORTANT:
        labels contain only INDEPENDENT neural targets.

    Labels have shape:
        [sequence_length, number_of_prediction_features]

    For supervised tokens:
        0 = independent feature absent
        1 = independent feature present

    For ignored tokens:
        -100 across all labels

    Only the first RoBERTa subword corresponding to each source token receives
    supervision.
    """

    def __init__(
        self,
        sentences: Sequence[Any],
        sentence_indices: Sequence[int],
        tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
        tokenizer: Any,
        label2id: Dict[str, int],
        max_length: int,
    ) -> None:
        self.sentences = sentences
        self.sentence_indices = list(sentence_indices)
        self.tagged_records = tagged_records
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.num_labels = len(label2id)
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.sentence_indices)

    def __getitem__(
        self,
        dataset_index: int,
    ) -> Dict[str, Any]:
        sentence_index = self.sentence_indices[dataset_index]
        sentence = self.sentences[sentence_index]

        words, token_ids = get_sentence_words(sentence)

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=True,
        )

        word_ids = encoding.word_ids()
        labels: List[List[float]] = []
        previous_word_id: Optional[int] = None
        sentence_records = self.tagged_records.get(sentence_index, {})

        for word_id in word_ids:
            if word_id is None:
                labels.append([-100.0] * self.num_labels)
                previous_word_id = word_id
                continue

            # Ignore all but the first subword.
            if word_id == previous_word_id:
                labels.append([-100.0] * self.num_labels)
                continue

            previous_word_id = word_id

            if not 0 <= word_id < len(token_ids):
                labels.append([-100.0] * self.num_labels)
                continue

            conllu_token_id = token_ids[word_id]
            record = sentence_records.get(conllu_token_id)

            if record is None:
                labels.append([-100.0] * self.num_labels)
                continue

            full_features = clean_features(record)

            if not full_features and EXCLUDE_EMPTY_FEATURE_RECORDS:
                labels.append([-100.0] * self.num_labels)
                continue

            independent_features = reduce_to_independent_features(
                full_features
            )

            target = [0.0] * self.num_labels

            for feature in independent_features:
                label_id = self.label2id.get(feature)

                if label_id is not None:
                    target[label_id] = 1.0

            labels.append(target)

        encoding["labels"] = labels
        encoding["sentence_index"] = sentence_index

        return encoding


# =============================================================================
# Data collator
# =============================================================================

class MultiLabelTokenCollator:
    """Dynamically pad inputs and 3-D multi-label token targets."""

    def __init__(
        self,
        tokenizer: Any,
        num_labels: int,
    ) -> None:
        self.tokenizer = tokenizer
        self.num_labels = num_labels

    def __call__(
        self,
        features: List[Dict[str, Any]],
    ) -> Dict[str, torch.Tensor]:
        clean_features_list: List[Dict[str, Any]] = []
        labels_list: List[List[List[float]]] = []

        for item in features:
            item = dict(item)
            item.pop("sentence_index", None)

            labels = item.pop("labels")
            labels_list.append(labels)
            clean_features_list.append(item)

        batch = self.tokenizer.pad(
            clean_features_list,
            padding=True,
            return_tensors="pt",
        )

        batch_size = len(labels_list)
        max_seq_length = batch["input_ids"].shape[1]

        padded_labels = torch.full(
            (
                batch_size,
                max_seq_length,
                self.num_labels,
            ),
            -100.0,
            dtype=torch.float32,
        )

        for batch_index, labels in enumerate(labels_list):
            label_tensor = torch.tensor(
                labels,
                dtype=torch.float32,
            )

            sequence_length = label_tensor.shape[0]

            padded_labels[
                batch_index,
                :sequence_length,
                :,
            ] = label_tensor

        batch["labels"] = padded_labels

        return batch


# =============================================================================
# Model
# =============================================================================

class RobertaMultiLabelModel(nn.Module):
    """RoBERTa token classifier with BCE loss for independent feature labels."""

    def __init__(
        self,
        model_name: str,
        num_labels: int,
        id2label: Dict[int, str],
        label2id: Dict[str, int],
    ) -> None:
        super().__init__()

        self.model = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            id2label=id2label,
            label2id=label2id,
        )

        self.config = self.model.config

        self.loss_function = nn.BCEWithLogitsLoss(
            reduction="mean"
        )

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        **kwargs,
    ):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs,
        )

        logits = outputs.logits
        loss = None

        if labels is not None:
            valid_mask = labels[:, :, 0] != -100.0

            valid_logits = logits[valid_mask]
            valid_labels = labels[valid_mask]

            if valid_logits.numel() == 0:
                loss = logits.sum() * 0.0
            else:
                loss = self.loss_function(
                    valid_logits,
                    valid_labels,
                )

        return {
            "loss": loss,
            "logits": logits,
        }

    def save_pretrained(
        self,
        save_directory,
        **kwargs,
    ):
        return self.model.save_pretrained(
            save_directory,
            **kwargs,
        )


# =============================================================================
# Hierarchy matrices and array helpers
# =============================================================================

def build_hierarchy_expansion_matrix(
    prediction_labels: Sequence[str],
    full_labels: Sequence[str],
) -> np.ndarray:
    """
    Build a boolean matrix mapping neural outputs to complete feature sets.

    Shape:
        [number_of_prediction_features, number_of_full_features]
    """

    full_label2id = {
        label: index
        for index, label in enumerate(full_labels)
    }

    matrix = np.zeros(
        (len(prediction_labels), len(full_labels)),
        dtype=np.int8,
    )

    for prediction_id, feature in enumerate(prediction_labels):
        expanded = expand_feature_hierarchy({feature})

        for expanded_feature in expanded:
            full_id = full_label2id.get(expanded_feature)

            if full_id is None:
                raise RuntimeError(
                    "Hierarchy expansion produced a feature absent from the "
                    f"full vocabulary: {expanded_feature!r}."
                )

            matrix[prediction_id, full_id] = 1

    return matrix


def expand_binary_feature_matrix(
    independent_matrix: np.ndarray,
    hierarchy_matrix: np.ndarray,
) -> np.ndarray:
    """Expand [tokens, independent] binary values to [tokens, complete]."""

    expanded = independent_matrix.astype(np.int32) @ hierarchy_matrix.astype(
        np.int32
    )

    return (expanded > 0).astype(np.int32)


def extract_valid_token_arrays(
    logits: np.ndarray,
    labels: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return probabilities and true independent labels for supervised tokens."""

    valid_token_mask = labels[:, :, 0] != -100.0

    probabilities = sigmoid_numpy(logits)[
        valid_token_mask
    ]

    true_labels = labels[
        valid_token_mask
    ].astype(np.int32)

    return probabilities, true_labels


def build_full_truth_matrix_for_dataset(
    dataset: "SemanticFeatureDataset",
    full_labels: Sequence[str],
) -> np.ndarray:
    """Build complete source truth for every supervised token in dataset order.

    This is intentionally independent of the neural-output vocabulary.  It is
    therefore safe and necessary for a held-out test corpus that may contain a
    feature such as ``Human`` as a direct independent target even when that
    feature never occurred independently in the training corpus.

    The model is still trained only on training-derived independent output
    heads.  This function merely preserves the complete gold annotations for
    fair hierarchy-expanded evaluation.
    """

    full_label2id = {
        label: index
        for index, label in enumerate(full_labels)
    }

    rows: List[np.ndarray] = []

    for sentence_index in dataset.sentence_indices:
        sentence = dataset.sentences[sentence_index]
        words, token_ids = get_sentence_words(sentence)

        encoding = dataset.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=dataset.max_length,
            add_special_tokens=True,
        )

        word_ids = encoding.word_ids()
        previous_word_id: Optional[int] = None
        sentence_records = dataset.tagged_records.get(sentence_index, {})

        for word_id in word_ids:
            if word_id is None:
                previous_word_id = word_id
                continue

            # Match SemanticFeatureDataset.__getitem__: only the first subword
            # for each source token receives supervision.
            if word_id == previous_word_id:
                continue

            previous_word_id = word_id

            if not 0 <= word_id < len(token_ids):
                continue

            conllu_token_id = token_ids[word_id]
            record = sentence_records.get(conllu_token_id)

            if record is None:
                continue

            features = clean_features(record)

            if not features and EXCLUDE_EMPTY_FEATURE_RECORDS:
                continue

            expanded_features = expand_feature_hierarchy(features)
            row = np.zeros(len(full_labels), dtype=np.int32)

            for feature in expanded_features:
                label_id = full_label2id.get(feature)

                if label_id is None:
                    raise RuntimeError(
                        "Complete source truth contains a feature absent from "
                        f"the training-derived full vocabulary: {feature!r}."
                    )

                row[label_id] = 1

            rows.append(row)

    if not rows:
        return np.zeros((0, len(full_labels)), dtype=np.int32)

    return np.stack(rows, axis=0)


# =============================================================================
# Metrics and thresholds
# =============================================================================

def sigmoid_numpy(x: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid."""

    x = np.clip(x, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-x))


def make_threshold_vector(
    prediction_labels: Sequence[str],
    thresholds: Dict[str, float],
) -> np.ndarray:
    """Return thresholds in neural-output column order."""

    return np.asarray(
        [
            float(
                thresholds.get(
                    label,
                    DEFAULT_PREDICTION_THRESHOLD,
                )
            )
            for label in prediction_labels
        ],
        dtype=np.float32,
    )


def threshold_probabilities(
    probabilities: np.ndarray,
    threshold_vector: np.ndarray,
) -> np.ndarray:
    """Apply one threshold per independent feature."""

    return (
        probabilities >= threshold_vector.reshape(1, -1)
    ).astype(np.int32)


def calculate_average_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    prefix: str = "",
) -> Dict[str, float]:
    """Calculate micro and macro precision/recall/F1."""

    if y_true.shape[0] == 0:
        return {
            f"{prefix}precision_micro": 0.0,
            f"{prefix}recall_micro": 0.0,
            f"{prefix}f1_micro": 0.0,
            f"{prefix}precision_macro": 0.0,
            f"{prefix}recall_macro": 0.0,
            f"{prefix}f1_macro": 0.0,
        }

    (
        precision_micro,
        recall_micro,
        f1_micro,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="micro",
        zero_division=0,
    )

    (
        precision_macro,
        recall_macro,
        f1_macro,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    return {
        f"{prefix}precision_micro": float(precision_micro),
        f"{prefix}recall_micro": float(recall_micro),
        f"{prefix}f1_micro": float(f1_micro),
        f"{prefix}precision_macro": float(precision_macro),
        f"{prefix}recall_macro": float(recall_macro),
        f"{prefix}f1_macro": float(f1_macro),
    }


def calculate_hierarchical_metrics_from_probabilities(
    probabilities: np.ndarray,
    y_true_independent: np.ndarray,
    threshold_vector: np.ndarray,
    hierarchy_matrix: np.ndarray,
    y_true_full: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    """
    Report both direct neural-output metrics and complete hierarchy metrics.

    Unprefixed metrics are the COMPLETE hierarchy metrics.  During ordinary
    training/validation, complete truth can be reconstructed from the
    independent targets.  For a genuinely held-out test corpus, ``y_true_full``
    may instead be supplied directly from the source annotations so an unseen
    direct independent label is not silently erased from the gold truth.
    """

    y_pred_independent = threshold_probabilities(
        probabilities,
        threshold_vector,
    )

    if y_true_full is None:
        y_true_full = expand_binary_feature_matrix(
            y_true_independent,
            hierarchy_matrix,
        )

    if y_true_full.shape[0] != y_true_independent.shape[0]:
        raise RuntimeError(
            "Complete and independent truth matrices contain different "
            f"numbers of supervised tokens: {y_true_full.shape[0]} vs "
            f"{y_true_independent.shape[0]}."
        )

    y_pred_full = expand_binary_feature_matrix(
        y_pred_independent,
        hierarchy_matrix,
    )

    metrics = calculate_average_metrics(
        y_true_full,
        y_pred_full,
        prefix="",
    )

    metrics.update(
        calculate_average_metrics(
            y_true_independent,
            y_pred_independent,
            prefix="direct_",
        )
    )

    return metrics


def make_compute_metrics(
    prediction_labels: Sequence[str],
    hierarchy_matrix: np.ndarray,
):
    """Create Trainer metrics using the fixed default 0.5 thresholds."""

    default_thresholds = np.full(
        len(prediction_labels),
        DEFAULT_PREDICTION_THRESHOLD,
        dtype=np.float32,
    )

    def compute_metrics(eval_prediction) -> Dict[str, float]:
        logits, labels = eval_prediction

        probabilities, y_true_independent = extract_valid_token_arrays(
            logits,
            labels,
        )

        return calculate_hierarchical_metrics_from_probabilities(
            probabilities=probabilities,
            y_true_independent=y_true_independent,
            threshold_vector=default_thresholds,
            hierarchy_matrix=hierarchy_matrix,
        )

    return compute_metrics


def build_threshold_grid() -> np.ndarray:
    """Build an inclusive, stable validation threshold search grid."""

    if THRESHOLD_STEP <= 0:
        raise ValueError("THRESHOLD_STEP must be > 0.")

    if not 0.0 < THRESHOLD_MIN < 1.0:
        raise ValueError("THRESHOLD_MIN must be between 0 and 1.")

    if not 0.0 < THRESHOLD_MAX < 1.0:
        raise ValueError("THRESHOLD_MAX must be between 0 and 1.")

    if THRESHOLD_MIN > THRESHOLD_MAX:
        raise ValueError("THRESHOLD_MIN cannot exceed THRESHOLD_MAX.")

    count = int(
        round(
            (THRESHOLD_MAX - THRESHOLD_MIN)
            / THRESHOLD_STEP
        )
    ) + 1

    return np.linspace(
        THRESHOLD_MIN,
        THRESHOLD_MAX,
        count,
        dtype=np.float64,
    )


def binary_metrics_for_column(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> Tuple[float, float, float, int]:
    """Return binary precision, recall, F1 and positive support."""

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    support = int(y_true.sum())

    return (
        float(precision),
        float(recall),
        float(f1),
        support,
    )


def optimise_feature_thresholds(
    probabilities: np.ndarray,
    y_true_independent: np.ndarray,
    prediction_labels: Sequence[str],
) -> Tuple[Dict[str, float], Dict[str, Dict[str, float]]]:
    """
    Optimise one sigmoid threshold per independent feature on VALIDATION ONLY.

    Objective:
        maximise that feature's binary F1.

    Stability rules:
    - If validation has no positives or no negatives for a feature, keep 0.5.
    - If several thresholds tie for best F1, choose the one closest to 0.5.
      This avoids unnecessarily extreme thresholds for flat validation curves.
    """

    grid = build_threshold_grid()

    thresholds: Dict[str, float] = {}
    diagnostics: Dict[str, Dict[str, float]] = {}

    for label_id, feature in enumerate(prediction_labels):
        y_true = y_true_independent[:, label_id]
        probs = probabilities[:, label_id]

        positive_support = int(y_true.sum())
        negative_support = int((1 - y_true).sum())

        default_pred = (
            probs >= DEFAULT_PREDICTION_THRESHOLD
        ).astype(np.int32)

        (
            default_precision,
            default_recall,
            default_f1,
            _,
        ) = binary_metrics_for_column(
            y_true,
            default_pred,
        )

        if positive_support == 0 or negative_support == 0:
            best_threshold = DEFAULT_PREDICTION_THRESHOLD
        else:
            best_f1 = -1.0
            best_candidates: List[float] = []

            for threshold in grid:
                y_pred = (probs >= threshold).astype(np.int32)

                _, _, f1, _ = binary_metrics_for_column(
                    y_true,
                    y_pred,
                )

                if f1 > best_f1 + 1e-12:
                    best_f1 = f1
                    best_candidates = [float(threshold)]
                elif abs(f1 - best_f1) <= 1e-12:
                    best_candidates.append(float(threshold))

            best_threshold = min(
                best_candidates,
                key=lambda value: (
                    abs(value - DEFAULT_PREDICTION_THRESHOLD),
                    value,
                ),
            )

        best_pred = (
            probs >= best_threshold
        ).astype(np.int32)

        (
            best_precision,
            best_recall,
            best_f1,
            _,
        ) = binary_metrics_for_column(
            y_true,
            best_pred,
        )

        thresholds[feature] = float(best_threshold)

        diagnostics[feature] = {
            "threshold": float(best_threshold),
            "support": positive_support,
            "negative_support": negative_support,
            "precision_default": default_precision,
            "recall_default": default_recall,
            "f1_default": default_f1,
            "precision_optimised": best_precision,
            "recall_optimised": best_recall,
            "f1_optimised": best_f1,
            "f1_change": best_f1 - default_f1,
        }

    return thresholds, diagnostics


def calculate_per_feature_metrics_from_binary(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    labels: Sequence[str],
) -> Dict[str, Dict[str, float]]:
    """Calculate binary precision/recall/F1 independently for each column."""

    results: Dict[str, Dict[str, float]] = {}

    for label_id, feature in enumerate(labels):
        precision, recall, f1, support = binary_metrics_for_column(
            y_true[:, label_id],
            y_pred[:, label_id],
        )

        results[feature] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        }

    return results


# =============================================================================
# Dataset diagnostics and validation
# =============================================================================

def count_feature_occurrences(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_indices: Sequence[int],
    independent_only: bool = False,
) -> Counter:
    """Count complete or independent positive labels in a split."""

    counts: Counter = Counter()

    for sentence_index in sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            if independent_only:
                features = get_independent_record_features(record)
            else:
                features = clean_features(record)

            for feature in features:
                counts[feature] += 1

    return counts


def count_empty_records(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_indices: Sequence[int],
) -> int:
    """Count NOUN/PROPN records having no complete feature tags."""

    count = 0

    for sentence_index in sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            if not clean_features(record):
                count += 1

    return count


def verify_full_label_coverage(
    full_labels: Sequence[str],
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    """Ensure validation/test contain no complete label unseen in training."""

    training_labels = set(full_labels)

    for split_name in ("validation", "test"):
        tagged_records, sentence_indices = split_sources[split_name]
        observed: Set[str] = set()

        for sentence_index in sentence_indices:
            for record in tagged_records.get(sentence_index, {}).values():
                observed.update(clean_features(record))

        unknown = observed - training_labels

        if unknown:
            raise RuntimeError(
                f"{split_name} contains full labels unseen in training: "
                f"{sorted(unknown)}"
            )


def verify_prediction_label_coverage(
    prediction_labels: Sequence[str],
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
    training_file_prediction_labels: Optional[Sequence[str]] = None,
) -> Dict[str, Dict[str, int]]:
    """Check independent-target coverage without leaking test labels.

    Validation remains strict: every validation independent target must have a
    training-derived neural output head.

    Test is different.  A genuine held-out corpus may contain a direct
    independent target that never occurred independently in training.  Such a
    label must NOT be added to the model from the test set.  Instead, report it
    and let complete-feature evaluation use the source gold matrix directly.
    """

    training_labels = set(prediction_labels)
    training_file_labels = (
        set(training_file_prediction_labels)
        if training_file_prediction_labels is not None
        else set(training_labels)
    )

    unseen_test_support: Dict[str, Dict[str, int]] = {}

    for split_name in ("validation", "test"):
        tagged_records, sentence_indices = split_sources[split_name]
        observed_counts: Counter = Counter()

        for sentence_index in sentence_indices:
            for record in tagged_records.get(sentence_index, {}).values():
                for feature in get_independent_record_features(record):
                    observed_counts[feature] += 1

        observed = set(observed_counts)
        unknown = observed - training_labels

        if not unknown:
            continue

        lost_due_to_split = unknown & training_file_labels
        absent_from_training_file = unknown - training_file_labels

        if split_name == "validation":
            details: List[str] = []

            if lost_due_to_split:
                details.append(
                    "present somewhere in the training file but absent from "
                    f"the training subset: {sorted(lost_due_to_split)}"
                )

            if absent_from_training_file:
                details.append(
                    "never occur as independent targets anywhere in the "
                    f"training file: {sorted(absent_from_training_file)}"
                )

            raise RuntimeError(
                "validation contains independent labels with no learnable "
                "training output. "
                + "; ".join(details)
            )

        # For the official test set, do not modify the model vocabulary.
        unseen_test_support = {
            feature: {
                "support": int(observed_counts[feature]),
                "present_in_training_file_as_independent": int(
                    feature in training_file_labels
                ),
            }
            for feature in sorted(unknown)
        }

        print()
        print("=" * 80)
        print("HELD-OUT TEST LABEL COVERAGE WARNING")
        print("=" * 80)
        print(
            "The test corpus contains direct independent targets that have no "
            "training-derived neural output head. They will NOT be added to "
            "the model vocabulary. Complete test truth will be read directly "
            "from the source annotations instead."
        )

        for feature in sorted(unknown):
            print(
                f"  {feature:35s} support={observed_counts[feature]:,}"
            )

        print()

    return unseen_test_support


def verify_hierarchy_closure(
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    """
    Ensure source feature sets already contain all ancestors required by the
    hierarchy. This prevents evaluation against inconsistent source targets.
    """

    problems: List[str] = []
    max_examples = 20

    for split_name in ("train", "validation", "test"):
        tagged_records, sentence_indices = split_sources[split_name]
        problem_count = 0

        for sentence_index in sentence_indices:
            for token_id, record in tagged_records.get(
                sentence_index,
                {},
            ).items():
                features = clean_features(record)

                if not features:
                    continue

                missing = missing_hierarchy_features(features)

                if not missing:
                    continue

                problem_count += 1

                if len(problems) < max_examples:
                    problems.append(
                        f"{split_name}: sentence={sentence_index}, "
                        f"token={token_id}, "
                        f"form={record.get('form')!r}, "
                        f"missing={sorted(missing)}"
                    )

        if problem_count:
            print(
                f"Hierarchy closure problems in {split_name}: "
                f"{problem_count:,} token(s)"
            )

    if problems:
        example_text = "\n".join(
            f"  - {problem}"
            for problem in problems
        )

        raise RuntimeError(
            "The input dataset is not hierarchy-closed. Fix the source tags "
            "before training so deterministic reduction/expansion is exact.\n"
            f"Examples:\n{example_text}"
        )


def print_split_statistics(
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    print("=" * 80)
    print("DATASET SPLIT")
    print("=" * 80)

    for split_name in ("train", "validation", "test"):
        tagged_records, indices = split_sources[split_name]

        full_counts = count_feature_occurrences(
            tagged_records,
            indices,
            independent_only=False,
        )

        independent_counts = count_feature_occurrences(
            tagged_records,
            indices,
            independent_only=True,
        )

        empty_count = count_empty_records(
            tagged_records,
            indices,
        )

        noun_count = sum(
            len(tagged_records.get(index, {}))
            for index in indices
        )

        source_name = (
            "training file"
            if split_name in {"train", "validation"}
            else "test file"
        )

        print(
            f"{split_name:12s}: "
            f"{len(indices):,} sentences | "
            f"{noun_count:,} NOUN/PROPN tokens | "
            f"{sum(full_counts.values()):,} complete positives -> "
            f"{sum(independent_counts.values()):,} independent positives | "
            f"{empty_count:,} empty-feature tokens excluded | "
            f"source={source_name}"
        )

    print()


def print_prediction_label_statistics(
    prediction_labels: Sequence[str],
    full_labels: Sequence[str],
    train_tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    train_sentence_indices: Sequence[int],
) -> None:
    """Show which labels are neural outputs versus deterministic-only parents."""

    train_independent_counts = count_feature_occurrences(
        train_tagged_records,
        train_sentence_indices,
        independent_only=True,
    )

    prediction_set = set(prediction_labels)
    deterministic_only = [
        feature
        for feature in full_labels
        if feature not in prediction_set
    ]

    print("=" * 80)
    print("NEURAL OUTPUT FEATURES")
    print("=" * 80)
    print(
        f"Complete evaluation features: {len(full_labels)}"
    )
    print(
        f"Independent neural outputs: {len(prediction_labels)}"
    )
    print(
        f"Deterministic-only features in this training split: "
        f"{len(deterministic_only)}"
    )
    print()

    print(
        f"{'Feature':35s} "
        f"{'Independent train support':>26s}"
    )
    print("-" * 64)

    for feature in prediction_labels:
        print(
            f"{feature:35s} "
            f"{train_independent_counts[feature]:26d}"
        )

    if deterministic_only:
        print()
        print("Deterministic-only in this split:")

        for feature in deterministic_only:
            print(f"  - {feature}")

    print()


# =============================================================================
# Repeated-run and bootstrap helpers
# =============================================================================

def get_supervised_token_count_for_sentence(
    dataset: "SemanticFeatureDataset",
    sentence_index: int,
) -> int:
    """
    Count supervised first-subword targets for one source sentence.

    This mirrors SemanticFeatureDataset.__getitem__ exactly and is used to map
    flat prediction rows back to sentence clusters for test bootstrapping.
    """

    sentence = dataset.sentences[sentence_index]
    words, token_ids = get_sentence_words(sentence)

    encoding = dataset.tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=dataset.max_length,
        add_special_tokens=True,
    )

    word_ids = encoding.word_ids()
    previous_word_id: Optional[int] = None
    sentence_records = dataset.tagged_records.get(sentence_index, {})

    count = 0

    for word_id in word_ids:
        if word_id is None:
            previous_word_id = word_id
            continue

        if word_id == previous_word_id:
            continue

        previous_word_id = word_id

        if not 0 <= word_id < len(token_ids):
            continue

        conllu_token_id = token_ids[word_id]
        record = sentence_records.get(conllu_token_id)

        if record is None:
            continue

        features = clean_features(record)

        if not features and EXCLUDE_EMPTY_FEATURE_RECORDS:
            continue

        count += 1

    return count


def build_sentence_row_slices(
    dataset: "SemanticFeatureDataset",
) -> List[slice]:
    """
    Map every dataset sentence to its contiguous flat supervised-token rows.
    """

    slices: List[slice] = []
    start = 0

    for sentence_index in dataset.sentence_indices:
        count = get_supervised_token_count_for_sentence(
            dataset,
            sentence_index,
        )
        stop = start + count
        slices.append(slice(start, stop))
        start = stop

    return slices


def summarise_repeated_metrics(
    metric_dicts: Sequence[Dict[str, float]],
) -> Dict[str, Dict[str, float]]:
    """Return mean/SD/min/max across repeated seeded runs."""

    if not metric_dicts:
        return {}

    common_keys = sorted(
        set.intersection(
            *[
                {
                    key
                    for key, value in metrics.items()
                    if isinstance(
                        value,
                        (float, int, np.floating, np.integer),
                    )
                }
                for metrics in metric_dicts
            ]
        )
    )

    summary: Dict[str, Dict[str, float]] = {}

    for key in common_keys:
        values = np.asarray(
            [float(metrics[key]) for metrics in metric_dicts],
            dtype=np.float64,
        )

        summary[key] = {
            "mean": float(values.mean()),
            "sd": float(values.std(ddof=1))
            if len(values) > 1
            else 0.0,
            "minimum": float(values.min()),
            "maximum": float(values.max()),
            "n_runs": int(len(values)),
        }

    return summary


def _metrics_from_confusion_counts(
    tp_by_label: np.ndarray,
    fp_by_label: np.ndarray,
    fn_by_label: np.ndarray,
) -> Dict[str, float]:
    """Calculate six headline metrics directly from labelwise counts."""

    tp_by_label = tp_by_label.astype(np.float64, copy=False)
    fp_by_label = fp_by_label.astype(np.float64, copy=False)
    fn_by_label = fn_by_label.astype(np.float64, copy=False)

    tp = float(tp_by_label.sum())
    fp = float(fp_by_label.sum())
    fn = float(fn_by_label.sum())

    precision_micro = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_micro = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    if precision_micro + recall_micro > 0:
        f1_micro = (
            2.0
            * precision_micro
            * recall_micro
            / (precision_micro + recall_micro)
        )
    else:
        f1_micro = 0.0

    precision_den = tp_by_label + fp_by_label
    recall_den = tp_by_label + fn_by_label

    precision_by_label = np.divide(
        tp_by_label,
        precision_den,
        out=np.zeros_like(tp_by_label, dtype=np.float64),
        where=precision_den > 0,
    )
    recall_by_label = np.divide(
        tp_by_label,
        recall_den,
        out=np.zeros_like(tp_by_label, dtype=np.float64),
        where=recall_den > 0,
    )

    f1_den = precision_by_label + recall_by_label
    f1_by_label = np.divide(
        2.0 * precision_by_label * recall_by_label,
        f1_den,
        out=np.zeros_like(precision_by_label, dtype=np.float64),
        where=f1_den > 0,
    )

    return {
        "precision_micro": float(precision_micro),
        "recall_micro": float(recall_micro),
        "f1_micro": float(f1_micro),
        "precision_macro": float(precision_by_label.mean()),
        "recall_macro": float(recall_by_label.mean()),
        "f1_macro": float(f1_by_label.mean()),
    }


def build_sentence_confusion_arrays(
    y_true_full: np.ndarray,
    run_predictions_full: Sequence[np.ndarray],
    sentence_slices: Sequence[slice],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Precompute TP/FP/FN by [run, sentence, label] for fast cluster bootstrap.
    """

    n_runs = len(run_predictions_full)
    n_sentences = len(sentence_slices)
    n_labels = y_true_full.shape[1]

    tp = np.zeros(
        (n_runs, n_sentences, n_labels),
        dtype=np.int32,
    )
    fp = np.zeros_like(tp)
    fn = np.zeros_like(tp)

    for sentence_pos, row_slice in enumerate(sentence_slices):
        y_true_sentence = y_true_full[row_slice]

        for run_pos, y_pred_full in enumerate(run_predictions_full):
            y_pred_sentence = y_pred_full[row_slice]

            tp[run_pos, sentence_pos] = (
                (y_true_sentence == 1)
                & (y_pred_sentence == 1)
            ).sum(axis=0)

            fp[run_pos, sentence_pos] = (
                (y_true_sentence == 0)
                & (y_pred_sentence == 1)
            ).sum(axis=0)

            fn[run_pos, sentence_pos] = (
                (y_true_sentence == 1)
                & (y_pred_sentence == 0)
            ).sum(axis=0)

    return tp, fp, fn


def bootstrap_mean_test_metrics(
    y_true_full: np.ndarray,
    run_predictions_full: Sequence[np.ndarray],
    sentence_slices: Sequence[slice],
) -> Dict[str, Any]:
    """
    Sentence-cluster bootstrap CIs for the MEAN performance across run seeds.

    Each bootstrap replicate:
      1. samples test sentences with replacement;
      2. applies the SAME sampled sentence multiplicities to every trained run;
      3. computes each run's six hierarchy-expanded metrics;
      4. averages each metric across runs.

    Thus the interval quantifies held-out TEST-SAMPLE uncertainty around the
    mean performance of the repeated-run procedure. Training-seed variability is
    reported separately via mean and SD across runs.
    """

    if TEST_BOOTSTRAP_SAMPLES < 1:
        raise ValueError("TEST_BOOTSTRAP_SAMPLES must be at least 1.")

    if not 0.0 < TEST_BOOTSTRAP_CONFIDENCE < 1.0:
        raise ValueError(
            "TEST_BOOTSTRAP_CONFIDENCE must be strictly between 0 and 1."
        )

    if not run_predictions_full:
        raise ValueError(
            "At least one run prediction matrix is required for bootstrap."
        )

    n_sentences = len(sentence_slices)

    if n_sentences == 0:
        raise ValueError("No test sentences are available for bootstrap.")

    expected_rows = sum(
        row_slice.stop - row_slice.start
        for row_slice in sentence_slices
    )

    if expected_rows != y_true_full.shape[0]:
        raise RuntimeError(
            "Sentence slices do not cover the complete test truth matrix: "
            f"{expected_rows:,} rows vs {y_true_full.shape[0]:,}."
        )

    for run_pos, y_pred_full in enumerate(run_predictions_full):
        if y_pred_full.shape != y_true_full.shape:
            raise RuntimeError(
                f"Run {run_pos} prediction shape {y_pred_full.shape} does "
                f"not match test truth shape {y_true_full.shape}."
            )

    tp, fp, fn = build_sentence_confusion_arrays(
        y_true_full,
        run_predictions_full,
        sentence_slices,
    )

    rng = np.random.default_rng(TEST_BOOTSTRAP_SEED)

    metric_names = [
        "precision_micro",
        "recall_micro",
        "f1_micro",
        "precision_macro",
        "recall_macro",
        "f1_macro",
    ]

    bootstrap_values = {
        metric: np.empty(TEST_BOOTSTRAP_SAMPLES, dtype=np.float64)
        for metric in metric_names
    }

    for bootstrap_index in range(TEST_BOOTSTRAP_SAMPLES):
        sampled_sentences = rng.integers(
            0,
            n_sentences,
            size=n_sentences,
        )

        multiplicities = np.bincount(
            sampled_sentences,
            minlength=n_sentences,
        ).astype(np.int32)

        run_metric_values = {
            metric: []
            for metric in metric_names
        }

        for run_pos in range(len(run_predictions_full)):
            tp_by_label = np.tensordot(
                multiplicities,
                tp[run_pos],
                axes=(0, 0),
            )
            fp_by_label = np.tensordot(
                multiplicities,
                fp[run_pos],
                axes=(0, 0),
            )
            fn_by_label = np.tensordot(
                multiplicities,
                fn[run_pos],
                axes=(0, 0),
            )

            metrics = _metrics_from_confusion_counts(
                tp_by_label,
                fp_by_label,
                fn_by_label,
            )

            for metric in metric_names:
                run_metric_values[metric].append(metrics[metric])

        for metric in metric_names:
            bootstrap_values[metric][bootstrap_index] = float(
                np.mean(run_metric_values[metric])
            )

    alpha = 1.0 - TEST_BOOTSTRAP_CONFIDENCE
    lower_q = alpha / 2.0
    upper_q = 1.0 - lower_q

    intervals: Dict[str, Dict[str, float]] = {}

    for metric in metric_names:
        values = bootstrap_values[metric]

        intervals[metric] = {
            "bootstrap_mean": float(values.mean()),
            "bootstrap_sd": float(values.std(ddof=1))
            if len(values) > 1
            else 0.0,
            "ci_lower": float(np.quantile(values, lower_q)),
            "ci_upper": float(np.quantile(values, upper_q)),
            "confidence": float(TEST_BOOTSTRAP_CONFIDENCE),
            "bootstrap_samples": int(TEST_BOOTSTRAP_SAMPLES),
        }

    return {
        "bootstrap_unit": "sentence",
        "statistic": "mean_metric_across_repeated_training_runs",
        "seed": TEST_BOOTSTRAP_SEED,
        "n_test_sentences": n_sentences,
        "n_runs": len(run_predictions_full),
        "intervals": intervals,
    }


def make_json_serialisable(value: Any) -> Any:
    """Recursively convert NumPy scalars/arrays to ordinary JSON values."""

    if isinstance(value, dict):
        return {
            str(key): make_json_serialisable(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            make_json_serialisable(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, (np.floating, np.integer)):
        return value.item()

    return value


# =============================================================================
# Reporting helpers
# =============================================================================

def print_metric_dictionary(
    metrics: Dict[str, float],
) -> None:
    for key, value in sorted(metrics.items()):
        print(f"{key}: {value}")


def print_threshold_table(
    prediction_labels: Sequence[str],
    diagnostics: Dict[str, Dict[str, float]],
) -> None:
    print()
    print("=" * 100)
    print("VALIDATION-OPTIMISED THRESHOLDS")
    print("=" * 100)

    heading = (
        f"{'Feature':35s} "
        f"{'Threshold':>10s} "
        f"{'Support':>9s} "
        f"{'F1@0.5':>10s} "
        f"{'F1 opt':>10s} "
        f"{'Change':>10s}"
    )

    print(heading)
    print("-" * len(heading))

    for feature in prediction_labels:
        result = diagnostics[feature]

        print(
            f"{feature:35s} "
            f"{result['threshold']:10.2f} "
            f"{int(result['support']):9d} "
            f"{result['f1_default']:10.4f} "
            f"{result['f1_optimised']:10.4f} "
            f"{result['f1_change']:+10.4f}"
        )


def print_per_feature_table(
    title: str,
    labels: Sequence[str],
    results: Dict[str, Dict[str, float]],
) -> None:
    print()
    print("=" * 80)
    print(title)
    print("=" * 80)

    heading = (
        f"{'Feature':35s} "
        f"{'Precision':>10s} "
        f"{'Recall':>10s} "
        f"{'F1':>10s} "
        f"{'Support':>10s}"
    )

    print(heading)
    print("-" * len(heading))

    for feature in labels:
        result = results[feature]

        print(
            f"{feature:35s} "
            f"{result['precision']:10.4f} "
            f"{result['recall']:10.4f} "
            f"{result['f1']:10.4f} "
            f"{result['support']:10d}"
        )


def summarise_repeated_per_feature_metrics(
    run_documents: Sequence[Dict[str, Any]],
    result_key: str,
    labels: Sequence[str],
) -> Dict[str, Dict[str, float]]:
    """Summarise per-feature test precision/recall/F1 across repeated runs."""

    summary: Dict[str, Dict[str, float]] = {}

    for feature in labels:
        precision_values = np.asarray(
            [
                run_document[result_key][feature]["precision"]
                for run_document in run_documents
            ],
            dtype=np.float64,
        )
        recall_values = np.asarray(
            [
                run_document[result_key][feature]["recall"]
                for run_document in run_documents
            ],
            dtype=np.float64,
        )
        f1_values = np.asarray(
            [
                run_document[result_key][feature]["f1"]
                for run_document in run_documents
            ],
            dtype=np.float64,
        )
        supports = {
            int(run_document[result_key][feature]["support"])
            for run_document in run_documents
        }

        if len(supports) != 1:
            raise RuntimeError(
                f"Test support changed across runs for feature {feature!r}: "
                f"{sorted(supports)}"
            )

        summary[feature] = {
            "precision_mean": float(precision_values.mean()),
            "precision_sd": float(precision_values.std(ddof=1))
            if len(precision_values) > 1
            else 0.0,
            "recall_mean": float(recall_values.mean()),
            "recall_sd": float(recall_values.std(ddof=1))
            if len(recall_values) > 1
            else 0.0,
            "f1_mean": float(f1_values.mean()),
            "f1_sd": float(f1_values.std(ddof=1))
            if len(f1_values) > 1
            else 0.0,
            "f1_min": float(f1_values.min()),
            "f1_max": float(f1_values.max()),
            "support": int(next(iter(supports))),
        }

    return summary


def print_repeated_per_feature_table(
    title: str,
    labels: Sequence[str],
    summary: Dict[str, Dict[str, float]],
) -> None:
    """Print repeated-run mean/SD test performance for each feature."""

    print()
    print("=" * 118)
    print(title)
    print("=" * 118)

    heading = (
        f"{'Feature':35s} "
        f"{'P mean':>8s} "
        f"{'P SD':>8s} "
        f"{'R mean':>8s} "
        f"{'R SD':>8s} "
        f"{'F1 mean':>8s} "
        f"{'F1 SD':>8s} "
        f"{'F1 min':>8s} "
        f"{'F1 max':>8s} "
        f"{'Support':>8s}"
    )

    print(heading)
    print("-" * len(heading))

    for feature in labels:
        result = summary[feature]
        print(
            f"{feature:35s} "
            f"{result['precision_mean']:8.4f} "
            f"{result['precision_sd']:8.4f} "
            f"{result['recall_mean']:8.4f} "
            f"{result['recall_sd']:8.4f} "
            f"{result['f1_mean']:8.4f} "
            f"{result['f1_sd']:8.4f} "
            f"{result['f1_min']:8.4f} "
            f"{result['f1_max']:8.4f} "
            f"{int(result['support']):8d}"
        )


# =============================================================================
# Main
# =============================================================================

def main() -> None:
    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )
    RUNS_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    validate_feature_hierarchy()
    check_cuda()

    if not RUN_SEEDS:
        raise ValueError("RUN_SEEDS must contain at least one integer seed.")

    if len(set(RUN_SEEDS)) != len(RUN_SEEDS):
        raise ValueError("RUN_SEEDS contains duplicate seeds.")

    # -------------------------------------------------------------------------
    # Load independent TRAIN and held-out TEST corpora
    # -------------------------------------------------------------------------

    print("=" * 80)
    print("LOADING DATA")
    print("=" * 80)

    train_sentences = load_conllu(TRAIN_CONLLU_PATH)
    test_sentences = load_conllu(TEST_CONLLU_PATH)

    train_tagged_records = load_tagged_records(TRAIN_JSONL_PATH)
    test_tagged_records = load_tagged_records(TEST_JSONL_PATH)

    validate_feature_switches(
        train_tagged_records,
        test_tagged_records,
    )

    print(
        "Training CoNLL-U sentences: "
        f"{len(train_sentences):,}"
    )
    print(
        "Test CoNLL-U sentences: "
        f"{len(test_sentences):,}"
    )
    print(
        "Training sentences represented in tagged JSONL: "
        f"{len(train_tagged_records):,}"
    )
    print(
        "Test sentences represented in tagged JSONL: "
        f"{len(test_tagged_records):,}"
    )

    train_target_record_count = sum(
        len(records)
        for records in train_tagged_records.values()
    )
    test_target_record_count = sum(
        len(records)
        for records in test_tagged_records.values()
    )

    print(
        "Training tagged NOUN/PROPN records: "
        f"{train_target_record_count:,}"
    )
    print(
        "Test tagged NOUN/PROPN records: "
        f"{test_target_record_count:,}"
    )
    print()

    # -------------------------------------------------------------------------
    # Fixed sentence-level TRAIN/VALIDATION split; TEST remains untouched
    # -------------------------------------------------------------------------

    eligible_train_indices = get_eligible_sentence_indices(
        train_sentences,
        train_tagged_records,
    )
    eligible_test_indices = get_eligible_sentence_indices(
        test_sentences,
        test_tagged_records,
    )

    print(
        "Eligible training-file sentences with at least one non-empty "
        "feature target: "
        f"{len(eligible_train_indices):,}"
    )
    print(
        "Eligible test-file sentences with at least one non-empty "
        "feature target: "
        f"{len(eligible_test_indices):,}"
    )

    train_validation_split = make_train_validation_split(
        eligible_train_indices,
        train_tagged_records,
    )

    train_indices = train_validation_split["train"]
    validation_indices = train_validation_split["validation"]
    test_indices = list(eligible_test_indices)

    split_sources = {
        "train": (train_tagged_records, train_indices),
        "validation": (train_tagged_records, validation_indices),
        "test": (test_tagged_records, test_indices),
    }

    split_document = {
        "split_unit": "sentence",
        "stratification": "iterative_multilabel_sentence_presence",
        "split_seed_requested": SPLIT_SEED,
        "split_seed_used": train_validation_split["split_seed_used"],
        "split_attempt": train_validation_split["attempt"],
        "validation_fraction": VALIDATION_FRACTION,
        "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
        "prevalence_error": train_validation_split["prevalence_error"],
        "stratification_labels": (
            train_validation_split["stratification_labels"]
        ),
        "train_jsonl": str(TRAIN_JSONL_PATH),
        "test_jsonl": str(TEST_JSONL_PATH),
        "train_conllu": str(TRAIN_CONLLU_PATH),
        "test_conllu": str(TEST_CONLLU_PATH),
        "index_spaces": {
            "train": "training corpus",
            "validation": "training corpus",
            "test": "test corpus",
        },
        "train": train_indices,
        "validation": validation_indices,
        "test": test_indices,
    }

    with SPLIT_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            make_json_serialisable(split_document),
            handle,
            ensure_ascii=False,
            indent=2,
        )

    # The reduction is exact only when source data are hierarchy-closed.
    verify_hierarchy_closure(split_sources)

    # -------------------------------------------------------------------------
    # Separate complete and neural-output vocabularies
    # -------------------------------------------------------------------------

    training_file_prediction_labels = build_prediction_label_vocabulary(
        train_tagged_records,
        eligible_train_indices,
    )

    # Vocabularies are derived from TRAIN only, never validation/test.
    full_labels = build_full_label_vocabulary(
        train_tagged_records,
        train_indices,
    )

    prediction_labels = build_prediction_label_vocabulary(
        train_tagged_records,
        train_indices,
    )

    verify_full_label_coverage(
        full_labels,
        split_sources,
    )

    unseen_test_independent_labels = verify_prediction_label_coverage(
        prediction_labels,
        split_sources,
        training_file_prediction_labels=training_file_prediction_labels,
    )

    prediction_label2id = {
        label: index
        for index, label in enumerate(prediction_labels)
    }

    prediction_id2label = {
        index: label
        for label, index in prediction_label2id.items()
    }

    full_label2id = {
        label: index
        for index, label in enumerate(full_labels)
    }

    hierarchy_matrix = build_hierarchy_expansion_matrix(
        prediction_labels,
        full_labels,
    )

    label_document = {
        "labels": prediction_labels,
        "label2id": prediction_label2id,
        "id2label": {
            str(key): value
            for key, value in prediction_id2label.items()
        },
        "prediction_labels": prediction_labels,
        "prediction_label2id": prediction_label2id,
        "full_labels": full_labels,
        "full_label2id": full_label2id,
        "feature_parents": {
            child: sorted(parents)
            for child, parents in sorted(FEATURE_PARENTS.items())
        },
        "target_representation": "per_token_independent_features",
        "split_unit": "sentence",
        "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
        "disabled_features": sorted(
            feature
            for feature, enabled in FEATURE_SWITCHES.items()
            if not enabled
        ),
    }

    with LABELS_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            label_document,
            handle,
            ensure_ascii=False,
            indent=2,
        )

    print_split_statistics(split_sources)

    print_prediction_label_statistics(
        prediction_labels,
        full_labels,
        train_tagged_records,
        train_indices,
    )

    # -------------------------------------------------------------------------
    # Tokeniser and fixed datasets
    # -------------------------------------------------------------------------

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        use_fast=True,
        add_prefix_space=True,
    )

    train_dataset = SemanticFeatureDataset(
        sentences=train_sentences,
        sentence_indices=train_indices,
        tagged_records=train_tagged_records,
        tokenizer=tokenizer,
        label2id=prediction_label2id,
        max_length=MAX_LENGTH,
    )

    validation_dataset = SemanticFeatureDataset(
        sentences=train_sentences,
        sentence_indices=validation_indices,
        tagged_records=train_tagged_records,
        tokenizer=tokenizer,
        label2id=prediction_label2id,
        max_length=MAX_LENGTH,
    )

    test_dataset = SemanticFeatureDataset(
        sentences=test_sentences,
        sentence_indices=test_indices,
        tagged_records=test_tagged_records,
        tokenizer=tokenizer,
        label2id=prediction_label2id,
        max_length=MAX_LENGTH,
    )

    data_collator = MultiLabelTokenCollator(
        tokenizer=tokenizer,
        num_labels=len(prediction_labels),
    )

    # Complete held-out gold truth is fixed across all training runs.
    test_true_full = build_full_truth_matrix_for_dataset(
        test_dataset,
        full_labels,
    )

    test_sentence_slices = build_sentence_row_slices(
        test_dataset,
    )

    test_slice_rows = sum(
        row_slice.stop - row_slice.start
        for row_slice in test_sentence_slices
    )

    if test_slice_rows != test_true_full.shape[0]:
        raise RuntimeError(
            "Test sentence-cluster mapping does not align with source truth: "
            f"{test_slice_rows:,} mapped rows vs "
            f"{test_true_full.shape[0]:,} truth rows."
        )

    # -------------------------------------------------------------------------
    # Repeated fine-tuning runs
    # -------------------------------------------------------------------------

    print("=" * 80)
    print("REPEATED FINE-TUNING")
    print("=" * 80)
    print(f"Runs: {len(RUN_SEEDS)}")
    print(f"Seeds: {RUN_SEEDS}")
    print(
        "Train/validation split is fixed across all runs; only the "
        "fine-tuning seed changes."
    )
    print()

    use_bf16 = torch.cuda.is_bf16_supported()

    all_run_documents: List[Dict[str, Any]] = []
    all_test_metric_dicts: List[Dict[str, float]] = []
    all_validation_metric_dicts: List[Dict[str, float]] = []
    all_test_predictions_full: List[np.ndarray] = []

    work_root = OUTPUT_DIR / "_trainer_work"
    work_root.mkdir(parents=True, exist_ok=True)

    for run_number, run_seed in enumerate(RUN_SEEDS, start=1):
        print()
        print("=" * 100)
        print(
            f"RUN {run_number}/{len(RUN_SEEDS)} | "
            f"SEED {run_seed}"
        )
        print("=" * 100)

        set_seed(run_seed)

        run_work_dir = work_root / f"seed_{run_seed}"
        run_result_path = RUNS_DIR / f"seed_{run_seed}.json"

        if run_work_dir.exists():
            shutil.rmtree(run_work_dir)

        model = RobertaMultiLabelModel(
            model_name=MODEL_NAME,
            num_labels=len(prediction_labels),
            id2label=prediction_id2label,
            label2id=prediction_label2id,
        )

        training_args = TrainingArguments(
            output_dir=str(run_work_dir),

            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,

            per_device_train_batch_size=TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=EVAL_BATCH_SIZE,

            gradient_accumulation_steps=(
                GRADIENT_ACCUMULATION_STEPS
            ),

            num_train_epochs=NUM_EPOCHS,

            eval_strategy=EVAL_STRATEGY,
            save_strategy="epoch",
            logging_strategy="steps",
            logging_steps=50,

            load_best_model_at_end=True,

            # Model selection is performed on VALIDATION only, using complete
            # hierarchy-expanded macro F1 at the fixed 0.5 threshold.
            metric_for_best_model="f1_macro",
            greater_is_better=True,

            save_total_limit=2,

            bf16=use_bf16,
            fp16=not use_bf16,

            dataloader_num_workers=4,
            dataloader_pin_memory=True,

            optim="adamw_torch",
            report_to="none",

            seed=run_seed,
            data_seed=run_seed,

            remove_unused_columns=False,
            eval_accumulation_steps=16,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=validation_dataset,
            processing_class=tokenizer,
            data_collator=data_collator,
            compute_metrics=make_compute_metrics(
                prediction_labels,
                hierarchy_matrix,
            ),
        )

        trainer.train()

        # ---------------------------------------------------------------------
        # Validation: first report 0.5, then calibrate thresholds on validation
        # ---------------------------------------------------------------------

        validation_output = trainer.predict(
            validation_dataset,
            metric_key_prefix="validation_default",
        )

        (
            validation_probabilities,
            validation_true_independent,
        ) = extract_valid_token_arrays(
            validation_output.predictions,
            validation_output.label_ids,
        )

        default_threshold_vector = np.full(
            len(prediction_labels),
            DEFAULT_PREDICTION_THRESHOLD,
            dtype=np.float32,
        )

        validation_default_metrics = (
            calculate_hierarchical_metrics_from_probabilities(
                probabilities=validation_probabilities,
                y_true_independent=validation_true_independent,
                threshold_vector=default_threshold_vector,
                hierarchy_matrix=hierarchy_matrix,
            )
        )

        (
            optimised_thresholds,
            threshold_diagnostics,
        ) = optimise_feature_thresholds(
            probabilities=validation_probabilities,
            y_true_independent=validation_true_independent,
            prediction_labels=prediction_labels,
        )

        threshold_vector = make_threshold_vector(
            prediction_labels,
            optimised_thresholds,
        )

        validation_optimised_metrics = (
            calculate_hierarchical_metrics_from_probabilities(
                probabilities=validation_probabilities,
                y_true_independent=validation_true_independent,
                threshold_vector=threshold_vector,
                hierarchy_matrix=hierarchy_matrix,
            )
        )

        print_threshold_table(
            prediction_labels,
            threshold_diagnostics,
        )

        print()
        print("VALIDATION @ 0.5")
        print_metric_dictionary(validation_default_metrics)

        print()
        print("VALIDATION WITH VALIDATION-CALIBRATED THRESHOLDS")
        print_metric_dictionary(validation_optimised_metrics)

        # ---------------------------------------------------------------------
        # Held-out TEST: thresholds are frozen
        # ---------------------------------------------------------------------

        test_output = trainer.predict(
            test_dataset,
            metric_key_prefix="test_default",
        )

        test_probabilities, test_true_independent = (
            extract_valid_token_arrays(
                test_output.predictions,
                test_output.label_ids,
            )
        )

        if test_true_full.shape[0] != test_probabilities.shape[0]:
            raise RuntimeError(
                "Test source-truth alignment failed: predictions contain "
                f"{test_probabilities.shape[0]:,} supervised tokens but "
                f"source truth contains {test_true_full.shape[0]:,}."
            )

        # Explicit test-at-0.5 diagnostic.
        test_default_metrics = (
            calculate_hierarchical_metrics_from_probabilities(
                probabilities=test_probabilities,
                y_true_independent=test_true_independent,
                threshold_vector=default_threshold_vector,
                hierarchy_matrix=hierarchy_matrix,
                y_true_full=test_true_full,
            )
        )

        # Final per-run test result uses validation-calibrated thresholds.
        test_optimised_metrics = (
            calculate_hierarchical_metrics_from_probabilities(
                probabilities=test_probabilities,
                y_true_independent=test_true_independent,
                threshold_vector=threshold_vector,
                hierarchy_matrix=hierarchy_matrix,
                y_true_full=test_true_full,
            )
        )

        test_pred_independent = threshold_probabilities(
            test_probabilities,
            threshold_vector,
        )

        test_pred_full = expand_binary_feature_matrix(
            test_pred_independent,
            hierarchy_matrix,
        )

        all_test_predictions_full.append(
            test_pred_full.astype(np.int8, copy=True)
        )

        per_feature_full = calculate_per_feature_metrics_from_binary(
            y_true=test_true_full,
            y_pred=test_pred_full,
            labels=full_labels,
        )

        per_feature_independent = (
            calculate_per_feature_metrics_from_binary(
                y_true=test_true_independent,
                y_pred=test_pred_independent,
                labels=prediction_labels,
            )
        )

        print()
        print("TEST @ 0.5 (DIAGNOSTIC)")
        print_metric_dictionary(test_default_metrics)

        print()
        print("TEST WITH FROZEN VALIDATION-CALIBRATED THRESHOLDS")
        print_metric_dictionary(test_optimised_metrics)

        if PRINT_PER_RUN_FEATURE_TABLES:
            print_per_feature_table(
                title=(
                    f"RUN {run_number} / SEED {run_seed}: TEST PERFORMANCE "
                    "BY COMPLETE FEATURE (AFTER INHERITANCE)"
                ),
                labels=full_labels,
                results=per_feature_full,
            )
            print_per_feature_table(
                title=(
                    f"RUN {run_number} / SEED {run_seed}: TEST PERFORMANCE "
                    "BY INDEPENDENT NEURAL OUTPUT"
                ),
                labels=prediction_labels,
                results=per_feature_independent,
            )

        # ---------------------------------------------------------------------
        # Save a neutral reference model, not a cherry-picked best seed
        # ---------------------------------------------------------------------

        is_reference_run = (
            SAVE_REFERENCE_MODEL
            and run_number == 1
        )

        if is_reference_run:
            if REFERENCE_MODEL_DIR.exists():
                shutil.rmtree(REFERENCE_MODEL_DIR)

            trainer.save_model(str(REFERENCE_MODEL_DIR))
            tokenizer.save_pretrained(str(REFERENCE_MODEL_DIR))

            with (
                REFERENCE_MODEL_DIR / "semantic_features.json"
            ).open(
                "w",
                encoding="utf-8",
            ) as handle:
                json.dump(
                    label_document,
                    handle,
                    ensure_ascii=False,
                    indent=2,
                )

            reference_threshold_document = {
                "seed": run_seed,
                "selection_policy": (
                    "first_predeclared_seed_not_best_test_run"
                ),
                "optimised_on": "validation",
                "objective": "per_feature_binary_f1",
                "thresholds": optimised_thresholds,
                "diagnostics": threshold_diagnostics,
            }

            with (
                REFERENCE_MODEL_DIR / "prediction_thresholds.json"
            ).open(
                "w",
                encoding="utf-8",
            ) as handle:
                json.dump(
                    make_json_serialisable(
                        reference_threshold_document
                    ),
                    handle,
                    ensure_ascii=False,
                    indent=2,
                )

        run_document = {
            "run_number": run_number,
            "seed": run_seed,
            "is_reference_model": is_reference_run,
            "model": MODEL_NAME,
            "target_representation": (
                "per_token_independent_features"
            ),
            "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
            "disabled_features": sorted(
                feature
                for feature, enabled in FEATURE_SWITCHES.items()
                if not enabled
            ),
            "split": {
                "unit": "sentence",
                "split_seed": (
                    train_validation_split["split_seed_used"]
                ),
                "validation_fraction": VALIDATION_FRACTION,
                "train_sentences": len(train_indices),
                "validation_sentences": len(validation_indices),
                "test_sentences": len(test_indices),
            },
            "threshold_calibration": {
                "data": "validation_only",
                "objective": "per_feature_binary_f1",
                "thresholds": optimised_thresholds,
                "diagnostics": threshold_diagnostics,
            },
            "validation_metrics_at_0_5": (
                validation_default_metrics
            ),
            "validation_metrics_optimised": (
                validation_optimised_metrics
            ),
            "test_metrics_at_0_5": test_default_metrics,
            "test_metrics_optimised": test_optimised_metrics,
            "per_feature_complete": per_feature_full,
            "per_feature_independent": (
                per_feature_independent
            ),
            "unseen_test_independent_labels": (
                unseen_test_independent_labels
            ),
        }

        with run_result_path.open(
            "w",
            encoding="utf-8",
        ) as handle:
            json.dump(
                make_json_serialisable(run_document),
                handle,
                ensure_ascii=False,
                indent=2,
            )

        all_run_documents.append(run_document)
        all_test_metric_dicts.append(test_optimised_metrics)
        all_validation_metric_dicts.append(
            validation_optimised_metrics
        )

        # Release run-specific GPU state before initialising the next model.
        del test_output
        del validation_output
        del trainer
        del model

        gc.collect()
        torch.cuda.empty_cache()

        # Checkpoints were needed only to recover the best validation epoch.
        # Remove them after each run to avoid storing dozens of RoBERTa-large
        # copies. The neutral first-seed reference model was saved separately.
        if run_work_dir.exists():
            shutil.rmtree(run_work_dir)

    if work_root.exists() and not any(work_root.iterdir()):
        work_root.rmdir()

    # -------------------------------------------------------------------------
    # Repeated-run summary: stochastic training variability
    # -------------------------------------------------------------------------

    validation_summary = summarise_repeated_metrics(
        all_validation_metric_dicts
    )
    test_summary = summarise_repeated_metrics(
        all_test_metric_dicts
    )

    print()
    print("=" * 100)
    print("REPEATED-RUN TEST SUMMARY")
    print("=" * 100)

    headline_metrics = [
        "precision_micro",
        "recall_micro",
        "f1_micro",
        "precision_macro",
        "recall_macro",
        "f1_macro",
    ]

    print(
        f"{'Metric':25s} "
        f"{'Mean':>10s} "
        f"{'SD':>10s} "
        f"{'Min':>10s} "
        f"{'Max':>10s}"
    )
    print("-" * 70)

    for metric in headline_metrics:
        result = test_summary[metric]

        print(
            f"{metric:25s} "
            f"{result['mean']:10.4f} "
            f"{result['sd']:10.4f} "
            f"{result['minimum']:10.4f} "
            f"{result['maximum']:10.4f}"
        )

    # -------------------------------------------------------------------------
    # Repeated-run per-feature test summaries
    # -------------------------------------------------------------------------

    per_feature_complete_summary = summarise_repeated_per_feature_metrics(
        all_run_documents,
        result_key="per_feature_complete",
        labels=full_labels,
    )
    per_feature_independent_summary = summarise_repeated_per_feature_metrics(
        all_run_documents,
        result_key="per_feature_independent",
        labels=prediction_labels,
    )

    print_repeated_per_feature_table(
        title=(
            "REPEATED-RUN TEST PERFORMANCE BY COMPLETE FEATURE "
            "(AFTER INHERITANCE)"
        ),
        labels=full_labels,
        summary=per_feature_complete_summary,
    )

    print_repeated_per_feature_table(
        title=(
            "REPEATED-RUN TEST PERFORMANCE BY INDEPENDENT NEURAL OUTPUT"
        ),
        labels=prediction_labels,
        summary=per_feature_independent_summary,
    )

    # -------------------------------------------------------------------------
    # Sentence-cluster bootstrap: held-out test-sample uncertainty
    # -------------------------------------------------------------------------

    print()
    print("=" * 100)
    print("SENTENCE-CLUSTER BOOTSTRAP ON HELD-OUT TEST SET")
    print("=" * 100)

    bootstrap_results = bootstrap_mean_test_metrics(
        y_true_full=test_true_full,
        run_predictions_full=all_test_predictions_full,
        sentence_slices=test_sentence_slices,
    )

    print(
        f"Bootstrap samples: {TEST_BOOTSTRAP_SAMPLES:,}"
    )
    print(
        f"Confidence level : "
        f"{TEST_BOOTSTRAP_CONFIDENCE:.1%}"
    )
    print(
        "Bootstrap unit   : sentence (all eligible NOUN/PROPN targets "
        "from a sampled sentence move together)"
    )
    print()

    ci_percent = int(round(TEST_BOOTSTRAP_CONFIDENCE * 100))

    print(
        f"{'Metric':25s} "
        f"{'Run mean':>10s} "
        f"{'Run SD':>10s} "
        f"{str(ci_percent) + '% CI low':>12s} "
        f"{str(ci_percent) + '% CI high':>12s}"
    )
    print("-" * 78)

    for metric in headline_metrics:
        repeated = test_summary[metric]
        interval = bootstrap_results["intervals"][metric]

        print(
            f"{metric:25s} "
            f"{repeated['mean']:10.4f} "
            f"{repeated['sd']:10.4f} "
            f"{interval['ci_lower']:12.4f} "
            f"{interval['ci_upper']:12.4f}"
        )

    # -------------------------------------------------------------------------
    # Save experiment-level results
    # -------------------------------------------------------------------------

    bootstrap_document = {
        "model": MODEL_NAME,
        "run_seeds": RUN_SEEDS,
        "split_seed": train_validation_split["split_seed_used"],
        "test_bootstrap": bootstrap_results,
    }

    with BOOTSTRAP_RESULTS_PATH.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            make_json_serialisable(bootstrap_document),
            handle,
            ensure_ascii=False,
            indent=2,
        )

    repeated_document = {
        "model": MODEL_NAME,
        "experimental_design": {
            "test_set": "fixed_external_held_out_file",
            "train_validation_split_unit": "sentence",
            "train_validation_stratification": (
                "iterative_multilabel_sentence_presence"
            ),
            "split_seed": (
                train_validation_split["split_seed_used"]
            ),
            "validation_fraction": VALIDATION_FRACTION,
            "number_of_runs": len(RUN_SEEDS),
            "run_seeds": RUN_SEEDS,
            "same_split_for_every_run": True,
            "threshold_calibration": (
                "per_feature_binary_f1_on_validation_only"
            ),
            "test_bootstrap_unit": "sentence",
            "test_bootstrap_samples": TEST_BOOTSTRAP_SAMPLES,
            "test_bootstrap_confidence": (
                TEST_BOOTSTRAP_CONFIDENCE
            ),
        },
        "data_sources": {
            "train_jsonl": str(TRAIN_JSONL_PATH),
            "test_jsonl": str(TEST_JSONL_PATH),
            "train_conllu": str(TRAIN_CONLLU_PATH),
            "test_conllu": str(TEST_CONLLU_PATH),
        },
        "split_sizes": {
            "train_sentences": len(train_indices),
            "validation_sentences": len(validation_indices),
            "test_sentences": len(test_indices),
        },
        "prediction_labels": prediction_labels,
        "full_labels": full_labels,
        "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
        "disabled_features": sorted(
            feature
            for feature, enabled in FEATURE_SWITCHES.items()
            if not enabled
        ),
        "unseen_test_independent_labels": (
            unseen_test_independent_labels
        ),
        "validation_repeated_run_summary": validation_summary,
        "test_repeated_run_summary": test_summary,
        "test_per_feature_complete_repeated_run_summary": (
            per_feature_complete_summary
        ),
        "test_per_feature_independent_repeated_run_summary": (
            per_feature_independent_summary
        ),
        "test_bootstrap": bootstrap_results,
        "runs": [
            {
                "run_number": run_document["run_number"],
                "seed": run_document["seed"],
                "is_reference_model": (
                    run_document["is_reference_model"]
                ),
                "result_file": str(
                    RUNS_DIR
                    / f"seed_{run_document['seed']}.json"
                ),
                "validation_metrics_optimised": (
                    run_document[
                        "validation_metrics_optimised"
                    ]
                ),
                "test_metrics_at_0_5": (
                    run_document["test_metrics_at_0_5"]
                ),
                "test_metrics_optimised": (
                    run_document["test_metrics_optimised"]
                ),
            }
            for run_document in all_run_documents
        ],
    }

    with REPEATED_RESULTS_PATH.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            make_json_serialisable(repeated_document),
            handle,
            ensure_ascii=False,
            indent=2,
        )

    print()
    print("=" * 100)
    print("OUTPUTS")
    print("=" * 100)
    print(
        "Fixed sentence split:\n"
        f"  {SPLIT_PATH.resolve()}"
    )
    print(
        "Semantic feature metadata:\n"
        f"  {LABELS_PATH.resolve()}"
    )
    print(
        "Per-run results:\n"
        f"  {RUNS_DIR.resolve()}"
    )
    print(
        "Repeated-run summary:\n"
        f"  {REPEATED_RESULTS_PATH.resolve()}"
    )
    print(
        "Test bootstrap confidence intervals:\n"
        f"  {BOOTSTRAP_RESULTS_PATH.resolve()}"
    )

    if SAVE_REFERENCE_MODEL:
        print(
            "Neutral reference model (first predeclared seed; not "
            "best-test selected):\n"
            f"  {REFERENCE_MODEL_DIR.resolve()}"
        )


if __name__ == "__main__":
    main()


/root/.pyenv/versions/3.10.16/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA
PyTorch version: 2.13.0+cu130
CUDA available: True
CUDA runtime: 13.0
GPU: NVIDIA GeForce RTX 5070 Ti
VRAM: 15.92 GiB

LOADING DATA
FEATURE SWITCHES
Disabled features: State
All unlisted features are enabled.

Training CoNLL-U sentences: 12,544
Test CoNLL-U sentences: 2,077
Training sentences represented in tagged JSONL: 11,511
Test sentences represented in tagged JSONL: 1,815
Training tagged NOUN/PROPN records: 47,371
Test tagged NOUN/PROPN records: 6,198

Eligible training-file sentences with at least one non-empty feature target: 11,198
Eligible test-file sentences with at least one non-empty feature target: 1,759

Sentence-level multilabel-stratified train/validation split:
  Eligible training sentences : 11,198
  Target validation sentences : 2,240
  Actual validation sentences : 2,313
  Training sentences retained : 8,885
  Split seed used             : 42
  Stratification labels       : 37
  Coverage-rejected attempts  : 0
  Mean relative prevalence error: 0.008441

Feature

Loading weights: 100%|███████████████████████████████████████████████████████████████| 389/389 [00:01<00:00, 339.04it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision Micro,Recall Micro,F1 Micro,Precision Macro,Recall Macro,F1 Macro,Direct Precision Micro,Direct Recall Micro,Direct F1 Micro,Direct Precision Macro,Direct Recall Macro,Direct F1 Macro
1,0.108096,0.044646,0.928032,0.855264,0.890163,0.815721,0.695057,0.733560,0.869723,0.787912,0.826799,0.713310,0.583491,0.621079
2,0.069943,0.034927,0.934325,0.900181,0.916935,0.876600,0.808073,0.833712,0.879206,0.853700,0.866265,0.811668,0.725233,0.755578
3,0.052070,0.032319,0.937223,0.914166,0.925551,0.888740,0.863020,0.873116,0.887744,0.864982,0.876215,0.851691,0.790847,0.808698
4,0.036753,0.031239,0.939192,0.921302,0.930161,0.897576,0.866265,0.877630,0.886424,0.875385,0.880870,0.851568,0.795422,0.810738
5,0.032900,0.031188,0.940321,0.924438,0.932312,0.902132,0.870060,0.882209,0.890273,0.879707,0.884958,0.886224,0.803335,0.821461



VALIDATION-OPTIMISED THRESHOLDS
Feature                              Threshold   Support     F1@0.5     F1 opt     Change
-----------------------------------------------------------------------------------------
Abstract                                  0.57      1405     0.8888     0.8934    +0.0046
Animal                                    0.82       184     0.9389     0.9408    +0.0020
Animate                                   0.23        13     0.6957     0.7500    +0.0543
Artefact                                  0.36       503     0.8837     0.8854    +0.0017
Body_Part                                 0.16        74     0.9028     0.9189    +0.0161
Comestible                                0.53       157     0.9524     0.9554    +0.0030
Communication                             0.39       534     0.7870     0.7959    +0.0089
Currency                                  0.50        45     1.0000     1.0000    +0.0000
Duration                                  0.70       340     0.8567

...


TEST @ 0.5 (DIAGNOSTIC)
direct_f1_macro: 0.7782294293233665
direct_f1_micro: 0.8606902836313931
direct_precision_macro: 0.8229572422714615
direct_precision_micro: 0.8652238635062407
direct_recall_macro: 0.7627515266551811
direct_recall_micro: 0.8562039660056657
f1_macro: 0.8523023629544918
f1_micro: 0.9135588705380927
precision_macro: 0.8747249094851296
precision_micro: 0.9195602627698083
recall_macro: 0.8370260484550888
recall_micro: 0.9076353050152177

TEST WITH FROZEN VALIDATION-CALIBRATED THRESHOLDS
direct_f1_macro: 0.7745598556550345
direct_f1_micro: 0.8583229602806383
direct_precision_macro: 0.8041237758876258
direct_precision_micro: 0.8571590010170641
direct_recall_macro: 0.7705902965169668
direct_recall_micro: 0.8594900849858357
f1_macro: 0.8478144167648416
f1_micro: 0.9126723282232271
precision_macro: 0.868233016995317
precision_micro: 0.9128132721496647
recall_macro: 0.8382502804657206
recall_micro: 0.9125314278152706

RUN 30/30 | SEED 1031


Loading weights: 100%|███████████████████████████████████████████████████████████████| 389/389 [00:00<00:00, 958.03it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision Micro,Recall Micro,F1 Micro,Precision Macro,Recall Macro,F1 Macro,Direct Precision Micro,Direct Recall Micro,Direct F1 Micro,Direct Precision Macro,Direct Recall Macro,Direct F1 Macro
1,0.099379,0.042761,0.929388,0.874486,0.901102,0.797098,0.724024,0.747078,0.876295,0.805421,0.839365,0.697219,0.609297,0.635778
2,0.068093,0.036310,0.934312,0.895894,0.914700,0.881169,0.809012,0.835176,0.881681,0.837436,0.858989,0.817502,0.721356,0.753179
3,0.051345,0.031732,0.940854,0.912238,0.926325,0.908329,0.848435,0.872362,0.890585,0.867619,0.878952,0.872929,0.775779,0.806152
4,0.039006,0.030320,0.942523,0.920582,0.931423,0.909988,0.867905,0.885347,0.894294,0.879487,0.886829,0.875182,0.798426,0.822193
5,0.030658,0.030485,0.941638,0.924812,0.933149,0.908257,0.873428,0.887640,0.891500,0.883663,0.887564,0.899846,0.808473,0.831164



VALIDATION-OPTIMISED THRESHOLDS
Feature                              Threshold   Support     F1@0.5     F1 opt     Change
-----------------------------------------------------------------------------------------
Abstract                                  0.70      1405     0.8887     0.8938    +0.0050
Animal                                    0.74       184     0.9474     0.9524    +0.0050
Animate                                   0.21        13     0.7000     0.9167    +0.2167
Artefact                                  0.54       503     0.8965     0.8990    +0.0025
Body_Part                                 0.16        74     0.9028     0.9139    +0.0111
Comestible                                0.50       157     0.9651     0.9651    +0.0000
Communication                             0.48       534     0.7993     0.8004    +0.0011
Currency                                  0.73        45     0.9783     0.9890    +0.0108
Duration                                  0.66       340     0.8382


TEST @ 0.5 (DIAGNOSTIC)
direct_f1_macro: 0.7671789154532318
direct_f1_micro: 0.8606426618049225
direct_precision_macro: 0.8584839600832023
direct_precision_micro: 0.8654749627592528
direct_recall_macro: 0.7388069680634745
direct_recall_micro: 0.8558640226628895
f1_macro: 0.8491299303625598
f1_micro: 0.9136465522980981
precision_macro: 0.9055152211366365
precision_micro: 0.9194211184563159
recall_macro: 0.8238549037098073
recall_micro: 0.9079440695161219

TEST WITH FROZEN VALIDATION-CALIBRATED THRESHOLDS
direct_f1_macro: 0.764976315755845
direct_f1_micro: 0.8579905288982712
direct_precision_macro: 0.8069887816537362
direct_precision_micro: 0.8640542404045047
direct_recall_macro: 0.7472435576494125
direct_recall_micro: 0.8520113314447592
f1_macro: 0.8416102672649476
f1_micro: 0.9108191345578623
precision_macro: 0.8847283867831143
precision_micro: 0.915056539934111
recall_macro: 0.818811561303086
recall_micro: 0.9066207930836752

REPEATED-RUN TEST SUMMARY
Metric                          

<h1>Using learning_rate scheduler</h1>

In [1]:
#!/usr/bin/env python3
"""
Fine-tune RoBERTa for hierarchical multi-label semantic feature tagging of
NOUN/PROPN tokens.

Compatible with Python 3.9.21.

Key design changes
------------------
1. The model is trained only on independent semantic features. For each token,
   any feature that can be inherited from a more specific present feature is
   removed from the neural target. This is a per-token reduction rather than a
   global "terminal leaf only" rule, so features such as Human, Time, Shape,
   Communication, etc. can still be predicted directly when they are the most
   specific available feature for that token.

2. All inherited features are reconstructed deterministically from
   FEATURE_PARENTS for validation, testing, reporting, and later inference.

3. A separate sigmoid threshold is optimised for every neural output using the
   validation split only. The selected thresholds are frozen before the test
   split is evaluated and are saved to prediction_thresholds.json.

4. Train/validation splitting is performed at SENTENCE level with iterative
   multilabel stratification over sentence-level independent-feature presence.
   The split is fixed across repeated fine-tuning runs.

5. Fine-tuning is repeated over a predeclared list of random seeds. The same
   split is used for every run, and results are summarised with mean and SD.

6. The official test corpus may contain a direct independent target that never
   occurred independently in training. Such a label is never added from test
   data. Complete test truth is read directly from source annotations.

7. Held-out test uncertainty is estimated with a SENTENCE-CLUSTER bootstrap.
   All supervised noun/proper-noun targets from a sampled sentence move together.

Tagged data:
    Training/validation:
        ewt_train_wordnet_wsd_features_wordnet_fixed.jsonl
    Test:
        ewt_test_wordnet_wsd_features_wordnet_fixed.jsonl

Sentence/token sources:
    Training/validation:
        UD_English-EWT-master/
        UD_English-EWT-master/
        en_ewt-ud-train.conllu
    Test:
        UD_English-EWT-master/
        UD_English-EWT-master/
        en_ewt-ud-test.conllu

The validation split is created only from the training corpus. The official
test corpus remains untouched until final evaluation.

Loss:
    BCEWithLogitsLoss over INDEPENDENT feature labels only.

Headline metrics:
    Computed after deterministic hierarchy expansion, so they describe the
    complete semantic feature sets rather than only the neural outputs.

Requirements:
    python -m pip install torch transformers datasets accelerate conllu scikit-learn iterative-stratification

CUDA is required.
"""

from __future__ import annotations

import gc
import json
import shutil
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Set, Tuple

import numpy as np
import torch
import torch.nn as nn

from conllu import parse_incr
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import precision_recall_fscore_support
from torch.utils.data import Dataset

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)


# =============================================================================
# Configuration
# =============================================================================

TRAIN_JSONL_PATH = Path(
    "../data/ewt_train_wordnet_wsd_features_wordnet_fixed.jsonl"
    #"ewt_train_wordnet_wsd_features_fully_tagged_llm_reviewed.jsonl"
)

TEST_JSONL_PATH = Path(
    "../data/ewt_test_wordnet_wsd_features_wordnet_fixed.jsonl"
    #"ewt_test_wordnet_wsd_features_fully_tagged_llm_reviewed.jsonl"
)

TRAIN_CONLLU_PATH = Path(
    "../data/UD_English-EWT-master/"
    "UD_English-EWT-master/"
    "en_ewt-ud-train.conllu"
)

TEST_CONLLU_PATH = Path(
    "../data/UD_English-EWT-master/"
    "UD_English-EWT-master/"
    "en_ewt-ud-test.conllu"
)

# MODEL_NAME = "FacebookAI/roberta-base"
MODEL_NAME = "FacebookAI/roberta-large"

OUTPUT_DIR = Path("../models/roberta_semantic_features")

# Validation is sampled only from eligible TRAINING sentences.
# Stratification is performed at SENTENCE level using the multilabel feature
# presence vector for each sentence. The separate official test file is never
# split and is used only for final evaluation.
VALIDATION_FRACTION = 0.20

# Fixed seed used ONLY to create the sentence-level train/validation split.
# Keeping this fixed across model runs means repeated runs measure stochastic
# fine-tuning variation rather than variation in the validation partition.
SPLIT_SEED = 42

# Repeated fine-tuning seeds. Use the same list when comparing model variants.
# Thirty runs follow the experiment-design recommendation; reduce this list if
# RoBERTa-large makes 30 runs computationally impractical, and report the actual
# number used.
RUN_SEEDS = [
    11, 23, 37, 53, 71, 89, 107, 127, 149, 173,
    197, 223, 251, 281, 313, 347, 383, 421, 461, 503,
    547, 593, 641, 691, 743, 797, 853, 911, 971, 1031,
]

# The multilabel stratifier is retried with deterministic derived seeds until
# every independent feature still has at least this many positive token targets
# in the TRAIN subset. This prevents a rare direct target from disappearing.
MIN_TRAIN_POSITIVES_PER_FEATURE = 1
MAX_STRATIFICATION_ATTEMPTS = 200

# Sentence-cluster bootstrap for uncertainty in the held-out test sample.
# The same sampled test sentences are used for every run within each bootstrap
# replicate, and the statistic is the mean metric across run seeds.
TEST_BOOTSTRAP_SAMPLES = 2000
TEST_BOOTSTRAP_CONFIDENCE = 0.95
TEST_BOOTSTRAP_SEED = 20260926

# Saving 30 RoBERTa-large models would use a great deal of disk space. The first
# run is therefore saved as a neutral REFERENCE model; it is not selected for
# having the best test result. Metrics and thresholds are saved for every run.
SAVE_REFERENCE_MODEL = True

MAX_LENGTH = 256

LEARNING_RATE = 3.5e-5
WEIGHT_DECAY = 0.01

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16

GRADIENT_ACCUMULATION_STEPS = 2

NUM_EPOCHS = 5

# Threshold used during epoch-by-epoch Trainer evaluation. Final thresholds are
# optimised per independent feature on the validation set after training.
DEFAULT_PREDICTION_THRESHOLD = 0.5

# Validation-only threshold search grid.
THRESHOLD_MIN = 0.05
THRESHOLD_MAX = 0.95
THRESHOLD_STEP = 0.01

# Only these POS categories receive feature supervision.
TARGET_POS = {"NOUN", "PROPN"}

# A NOUN/PROPN with no features is considered a failed source annotation rather
# than a valid all-negative target.
EXCLUDE_EMPTY_FEATURE_RECORDS = True

# Optional per-feature switches. Any feature not listed here is enabled.
# Set a feature to False to remove it from:
#   - sentence-level stratification,
#   - neural training targets,
#   - threshold calibration,
#   - hierarchy-expanded gold/predicted feature sets,
#   - per-feature and aggregate evaluation, and
#   - the sentence-cluster bootstrap.
#
# Disabled intermediate hierarchy nodes are skipped but their enabled ancestors
# are still inherited. For example, if Human were disabled, Feminine would still
# inherit Animate/Natural/Concrete/Internal_Control via the hierarchy.
FEATURE_SWITCHES: Dict[str, bool] = {
    "State": False,
    "State_of_Matter": False,
    "Shape": False,
    "Rigidity": False,
}

# Aggregate per-feature test tables are always printed after all repeated runs.
# Set this to True only if you also want the two full per-feature tables printed
# after EVERY run (very verbose for 30 runs).
PRINT_PER_RUN_FEATURE_TABLES = False

EVAL_STRATEGY = "epoch"

SPLIT_PATH = OUTPUT_DIR / "sentence_split.json"
LABELS_PATH = OUTPUT_DIR / "semantic_features.json"
RUNS_DIR = OUTPUT_DIR / "runs"
REFERENCE_MODEL_DIR = OUTPUT_DIR / "reference_model"
REPEATED_RESULTS_PATH = OUTPUT_DIR / "repeated_run_results.json"
BOOTSTRAP_RESULTS_PATH = OUTPUT_DIR / "test_bootstrap_confidence_intervals.json"


# =============================================================================
# Feature hierarchy
# =============================================================================

def is_feature_enabled(feature: str) -> bool:
    """Return whether *feature* participates in this experiment."""

    return bool(FEATURE_SWITCHES.get(feature, True))


def validate_feature_switches(
    train_tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    test_tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> None:
    """Fail early on misspelled feature-switch names and report disabled labels."""

    observed: Set[str] = set()

    for source in (train_tagged_records, test_tagged_records):
        for sentence_records in source.values():
            for record in sentence_records.values():
                raw = record.get("features", [])
                if isinstance(raw, list):
                    observed.update(
                        str(feature).strip()
                        for feature in raw
                        if str(feature).strip()
                    )

    known = set(observed) | set(FEATURE_PARENTS)
    for parents in FEATURE_PARENTS.values():
        known.update(parents)

    unknown = sorted(set(FEATURE_SWITCHES) - known)
    if unknown:
        raise ValueError(
            "FEATURE_SWITCHES contains unknown feature name(s): "
            + ", ".join(unknown)
        )

    disabled = sorted(
        feature for feature, enabled in FEATURE_SWITCHES.items() if not enabled
    )

    print("=" * 80)
    print("FEATURE SWITCHES")
    print("=" * 80)
    if disabled:
        print("Disabled features: " + ", ".join(disabled))
    else:
        print("Disabled features: none")
    print("All unlisted features are enabled.")
    print()


# child -> direct parent(s)
FEATURE_PARENTS: Dict[str, Set[str]] = {
    "Organisation": {
        "Internal_Control",
        "Abstract",
    },
    "Natural_Force": {
        "Internal_Control",
    },
    "Animate": {
        "Internal_Control",
        "Natural",
    },
    "Human": {
        "Animate",
    },
    "Feminine": {
        "Human",
    },
    "Masculine": {
        "Human",
    },
    "Unspecified_Gender_or_Neuter": {
        "Human",
    },
    "Animal": {
        "Animate",
    },
    "Plant": {
        "Animate",
    },
    "Body_Part": {
        "Animate",
    },
    "Machine": {
        "Internal_Control",
        "Artefact",
    },
    "Vehicle": {
        "Machine",
    },
    "Natural": {
        "Concrete",
    },
    "Artefact": {
        "Concrete",
    },
    "Comestible": {
        "Concrete",
    },
    "Substance": {
        "Concrete",
    },
    "State_of_Matter": {
        "Concrete",
    },
    "Inanimate": {
        "Concrete",
    },
    "Garment": {
        "Artefact",
    },
    "Solid": {
        "State_of_Matter",
    },
    "Fluid": {
        "State_of_Matter",
    },
    "Rigidity": {
        "Solid",
    },
    "Shape": {
        "Solid",
    },
    "Rigid": {
        "Rigidity",
    },
    "Non_Rigid": {
        "Rigidity",
    },
    "Elongated": {
        "Shape",
    },
    "Pointy": {
        "Shape",
    },
    "Point_in_Time": {
        "Time",
    },
    "Duration": {
        "Time",
    },
    "Time": {
        "Abstract",
    },
    "Language": {
        "Abstract",
    },
    "Sound": {
        "Abstract",
    },
    "Communication": {
        "Abstract",
    },
    "Eventive": {
        "Abstract",
    },
    "Statement": {
        "Communication",
    },
    "Question": {
        "Communication",
    },
}


def validate_feature_hierarchy() -> None:
    """Fail early if FEATURE_PARENTS contains a cycle or invalid name."""

    all_features: Set[str] = set(FEATURE_PARENTS)

    for parents in FEATURE_PARENTS.values():
        all_features.update(parents)

    for feature in all_features:
        if not isinstance(feature, str) or not feature.strip():
            raise ValueError(
                "FEATURE_PARENTS contains an empty or non-string feature name."
            )

    visiting: Set[str] = set()
    visited: Set[str] = set()

    def visit(feature: str) -> None:
        if feature in visited:
            return

        if feature in visiting:
            raise ValueError(
                f"Cycle detected in FEATURE_PARENTS involving {feature!r}."
            )

        visiting.add(feature)

        for parent in FEATURE_PARENTS.get(feature, set()):
            visit(parent)

        visiting.remove(feature)
        visited.add(feature)

    for feature in sorted(all_features):
        visit(feature)


@lru_cache(maxsize=None)
def get_feature_ancestors(feature: str) -> frozenset:
    """Return every direct and indirect parent of *feature*."""

    ancestors: Set[str] = set()

    for parent in FEATURE_PARENTS.get(feature, set()):
        if is_feature_enabled(parent):
            ancestors.add(parent)
        # Traverse even through a disabled intermediate node so enabled
        # grandparents remain inherited.
        ancestors.update(get_feature_ancestors(parent))

    return frozenset(ancestors)


def expand_feature_hierarchy(features: Iterable[str]) -> Set[str]:
    """Add every deterministic ancestor required by FEATURE_PARENTS."""

    expanded: Set[str] = {
        feature for feature in features if is_feature_enabled(feature)
    }

    for feature in list(expanded):
        expanded.update(get_feature_ancestors(feature))

    return expanded


def reduce_to_independent_features(features: Iterable[str]) -> Set[str]:
    """
    Reduce a complete feature set to the features that must be predicted.

    A feature is redundant when it is an ancestor of another feature present
    for the same token. Such an ancestor can be reconstructed deterministically.

    Examples:
        {Human, Animate, Natural, Concrete, Internal_Control}
            -> {Human}

        {Feminine, Human, Animate, Natural, Concrete, Internal_Control}
            -> {Feminine}

        {Statement, Communication, Abstract, Time}
            -> {Statement, Time}

    This is deliberately PER TOKEN. It is not the same as training only labels
    that are global terminal leaves in the ontology. A generic Human can still
    be a direct neural target if no more specific Human child is present.
    """

    feature_set = set(features)

    inherited_from_other_features: Set[str] = set()

    for feature in feature_set:
        inherited_from_other_features.update(
            get_feature_ancestors(feature)
        )

    return feature_set - inherited_from_other_features


def missing_hierarchy_features(features: Iterable[str]) -> Set[str]:
    """Return ancestors required by the hierarchy but absent from *features*."""

    feature_set = set(features)
    return expand_feature_hierarchy(feature_set) - feature_set


# =============================================================================
# CUDA
# =============================================================================

def check_cuda() -> None:
    """Require CUDA rather than silently falling back to the CPU."""

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. This script is configured to require "
            "a CUDA-capable GPU."
        )

    print("=" * 80)
    print("CUDA")
    print("=" * 80)
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"CUDA runtime: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    props = torch.cuda.get_device_properties(0)

    print(
        "VRAM: "
        f"{props.total_memory / (1024 ** 3):.2f} GiB"
    )
    print()


# =============================================================================
# Load CoNLL-U
# =============================================================================

def load_conllu(path: Path) -> List[Any]:
    """Load all CoNLL-U sentences."""

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find CoNLL-U file:\n{path.resolve()}"
        )

    sentences: List[Any] = []

    with path.open("r", encoding="utf-8") as handle:
        for sentence in parse_incr(handle):
            sentences.append(sentence)

    return sentences


def get_sentence_words(sentence: Any) -> Tuple[List[str], List[int]]:
    """
    Return ordinary integer-ID tokens and their original CoNLL-U token IDs.

    Multiword-token placeholder rows and empty nodes are excluded.
    """

    words: List[str] = []
    token_ids: List[int] = []

    for token in sentence:
        token_id = token.get("id")

        if not isinstance(token_id, int):
            continue

        form = token.get("form")

        if form is None:
            continue

        words.append(str(form))
        token_ids.append(token_id)

    return words, token_ids


# =============================================================================
# Load tagged JSONL
# =============================================================================

def load_tagged_records(
    path: Path,
) -> Dict[int, Dict[int, Dict[str, Any]]]:
    """
    Return:

        sentence_index
            -> token_id
                -> JSONL record

    Only NOUN and PROPN records are retained.
    """

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find tagged JSONL:\n{path.resolve()}"
        )

    records: Dict[int, Dict[int, Dict[str, Any]]] = defaultdict(dict)

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                print(
                    "Warning: invalid JSON on line "
                    f"{line_number}: {exc}"
                )
                continue

            if record.get("upos") not in TARGET_POS:
                continue

            sentence_index = record.get("sentence_index")
            token_id = record.get("token_id")

            if not isinstance(sentence_index, int):
                continue

            if not isinstance(token_id, int):
                continue

            records[sentence_index][token_id] = record

    return dict(records)


# =============================================================================
# Labels
# =============================================================================

def clean_features(record: Dict[str, Any]) -> List[str]:
    """Return unique, non-empty COMPLETE feature names from a JSONL record."""

    raw = record.get("features", [])

    if not isinstance(raw, list):
        return []

    return sorted(
        {
            str(feature).strip()
            for feature in raw
            if (
                str(feature).strip()
                and is_feature_enabled(str(feature).strip())
            )
        }
    )


def get_independent_record_features(record: Dict[str, Any]) -> List[str]:
    """Return the per-token minimal feature set that the model must predict."""

    return sorted(
        reduce_to_independent_features(
            clean_features(record)
        )
    )


def build_full_label_vocabulary(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    training_sentence_indices: Sequence[int],
) -> List[str]:
    """
    Build the COMPLETE evaluation vocabulary from training data only.

    Deterministic ancestors are included even if a training record omitted one,
    although hierarchy closure is separately verified and should normally make
    this addition unnecessary.
    """

    labels: Set[str] = set()

    for sentence_index in training_sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            features = clean_features(record)
            labels.update(features)
            labels.update(expand_feature_hierarchy(features))

    if not labels:
        raise RuntimeError(
            "No semantic features were found in the training split."
        )

    return sorted(labels)


def build_prediction_label_vocabulary(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    training_sentence_indices: Sequence[int],
) -> List[str]:
    """Build the neural-output vocabulary from independent training targets."""

    labels: Set[str] = set()

    for sentence_index in training_sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            labels.update(
                get_independent_record_features(record)
            )

    if not labels:
        raise RuntimeError(
            "No independent semantic features were found in the training split."
        )

    return sorted(labels)


# =============================================================================
# Sentence selection and splitting
# =============================================================================

def get_eligible_sentence_indices(
    sentences: Sequence[Any],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> List[int]:
    """Eligible sentences contain at least one non-empty NOUN/PROPN target."""

    eligible: List[int] = []

    for sentence_index in range(len(sentences)):
        token_records = tagged_records.get(sentence_index, {})

        valid_records = [
            record
            for record in token_records.values()
            if clean_features(record)
        ]

        if valid_records:
            eligible.append(sentence_index)

    return eligible


def get_sentence_independent_feature_counts(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_index: int,
) -> Counter:
    """Count independent positive token targets in one source sentence."""

    counts: Counter = Counter()

    for record in tagged_records.get(sentence_index, {}).values():
        if not clean_features(record):
            continue

        for feature in get_independent_record_features(record):
            counts[feature] += 1

    return counts


def build_sentence_multilabel_matrix(
    eligible_indices: Sequence[int],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> Tuple[np.ndarray, List[str], Dict[int, Counter]]:
    """
    Build one multilabel stratification row per SENTENCE.

    y_sentence[sentence, feature] is 1 when at least one eligible NOUN/PROPN
    token in that sentence has the independent feature. Token counts are kept
    separately for the post-split training-coverage constraint.
    """

    sentence_feature_counts: Dict[int, Counter] = {
        sentence_index: get_sentence_independent_feature_counts(
            tagged_records,
            sentence_index,
        )
        for sentence_index in eligible_indices
    }

    stratification_labels = sorted(
        {
            feature
            for counts in sentence_feature_counts.values()
            for feature, count in counts.items()
            if count > 0
        }
    )

    if not stratification_labels:
        raise RuntimeError(
            "No independent semantic features were found in eligible "
            "training sentences."
        )

    label2column = {
        label: column
        for column, label in enumerate(stratification_labels)
    }

    matrix = np.zeros(
        (len(eligible_indices), len(stratification_labels)),
        dtype=np.int8,
    )

    for row, sentence_index in enumerate(eligible_indices):
        for feature in sentence_feature_counts[sentence_index]:
            matrix[row, label2column[feature]] = 1

    return matrix, stratification_labels, sentence_feature_counts


def _sentence_prevalence_error(
    y_all: np.ndarray,
    validation_rows: np.ndarray,
    validation_fraction: float,
) -> float:
    """
    Score how closely validation sentence-level label prevalences match target.

    Lower is better. The score is used only to choose among deterministic
    iterative-stratification attempts that satisfy the training-coverage rule.
    """

    total_support = y_all.sum(axis=0).astype(np.float64)
    validation_support = y_all[validation_rows].sum(axis=0).astype(np.float64)

    target_support = total_support * float(validation_fraction)

    # Scale by at least one expected example so rare and common labels both
    # contribute without allowing extremely common labels to dominate.
    scale = np.maximum(target_support, 1.0)

    return float(
        np.mean(
            np.abs(validation_support - target_support) / scale
        )
    )


def make_train_validation_split(
    eligible_indices: Sequence[int],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> Dict[str, Any]:
    """
    Create a SENTENCE-LEVEL multilabel-stratified split from TRAINING only.

    Each sentence receives a binary vector indicating which independent feature
    labels occur anywhere among its eligible NOUN/PROPN tokens. Iterative
    multilabel stratification then assigns whole sentences to train/validation.

    The split is retried deterministically when necessary so that every
    independent feature retains at least MIN_TRAIN_POSITIVES_PER_FEATURE
    positive TOKEN targets in the training subset.

    The official test corpus is deliberately not accepted by this function, so
    test labels cannot influence the split.
    """

    if not 0.0 < VALIDATION_FRACTION < 1.0:
        raise ValueError(
            "VALIDATION_FRACTION must be strictly between 0 and 1."
        )

    if MIN_TRAIN_POSITIVES_PER_FEATURE < 1:
        raise ValueError(
            "MIN_TRAIN_POSITIVES_PER_FEATURE must be at least 1."
        )

    if MAX_STRATIFICATION_ATTEMPTS < 1:
        raise ValueError(
            "MAX_STRATIFICATION_ATTEMPTS must be at least 1."
        )

    eligible_indices = list(eligible_indices)

    if len(eligible_indices) < 2:
        raise RuntimeError(
            "Need at least two eligible training sentences to create a "
            "train/validation split."
        )

    (
        sentence_matrix,
        stratification_labels,
        sentence_feature_counts,
    ) = build_sentence_multilabel_matrix(
        eligible_indices,
        tagged_records,
    )

    all_token_counts: Counter = Counter()

    for counts in sentence_feature_counts.values():
        all_token_counts.update(counts)

    x_dummy = np.zeros((len(eligible_indices), 1), dtype=np.int8)

    best_candidate: Optional[Dict[str, Any]] = None
    rejected_for_coverage = 0

    for attempt in range(MAX_STRATIFICATION_ATTEMPTS):
        attempt_seed = SPLIT_SEED + attempt

        splitter = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=VALIDATION_FRACTION,
            random_state=attempt_seed,
        )

        train_rows, validation_rows = next(
            splitter.split(x_dummy, sentence_matrix)
        )

        train_indices = [
            eligible_indices[int(row)]
            for row in train_rows
        ]
        validation_indices = [
            eligible_indices[int(row)]
            for row in validation_rows
        ]

        train_token_counts: Counter = Counter()

        for sentence_index in train_indices:
            train_token_counts.update(
                sentence_feature_counts[sentence_index]
            )

        missing_or_undercovered = sorted(
            feature
            for feature, total_count in all_token_counts.items()
            if (
                total_count > 0
                and train_token_counts[feature]
                < MIN_TRAIN_POSITIVES_PER_FEATURE
            )
        )

        if missing_or_undercovered:
            rejected_for_coverage += 1
            continue

        prevalence_error = _sentence_prevalence_error(
            sentence_matrix,
            validation_rows,
            VALIDATION_FRACTION,
        )

        candidate = {
            "train": train_indices,
            "validation": validation_indices,
            "split_seed_used": attempt_seed,
            "attempt": attempt + 1,
            "prevalence_error": prevalence_error,
            "stratification_labels": stratification_labels,
        }

        if (
            best_candidate is None
            or prevalence_error < best_candidate["prevalence_error"]
        ):
            best_candidate = candidate

        # Iterative stratification is already designed to balance multilabel
        # prevalence. Once a valid candidate is found, further attempts are not
        # needed unless coverage forced earlier candidates to be rejected.
        if rejected_for_coverage == 0:
            break

    if best_candidate is None:
        raise RuntimeError(
            "Could not create a sentence-level multilabel-stratified "
            "train/validation split that preserved the required training "
            f"support after {MAX_STRATIFICATION_ATTEMPTS} attempts."
        )

    train_indices = best_candidate["train"]
    validation_indices = best_candidate["validation"]

    train_set = set(train_indices)
    validation_set = set(validation_indices)

    if train_set & validation_set:
        raise RuntimeError(
            "Internal split error: at least one sentence occurs in both "
            "training and validation."
        )

    if train_set | validation_set != set(eligible_indices):
        raise RuntimeError(
            "Internal split error: sentence-level split does not exactly cover "
            "the eligible training sentences."
        )

    print()
    print("Sentence-level multilabel-stratified train/validation split:")
    print(
        f"  Eligible training sentences : {len(eligible_indices):,}"
    )
    print(
        f"  Target validation sentences : "
        f"{round(len(eligible_indices) * VALIDATION_FRACTION):,}"
    )
    print(
        f"  Actual validation sentences : {len(validation_indices):,}"
    )
    print(
        f"  Training sentences retained : {len(train_indices):,}"
    )
    print(
        f"  Split seed used             : "
        f"{best_candidate['split_seed_used']}"
    )
    print(
        f"  Stratification labels       : "
        f"{len(stratification_labels):,}"
    )
    print(
        f"  Coverage-rejected attempts  : {rejected_for_coverage:,}"
    )
    print(
        f"  Mean relative prevalence error: "
        f"{best_candidate['prevalence_error']:.6f}"
    )

    # Show sentence-level prevalence agreement, which is the quantity actually
    # controlled by the stratifier.
    validation_fraction_actual = (
        len(validation_indices) / len(eligible_indices)
    )

    label2column = {
        label: column
        for column, label in enumerate(stratification_labels)
    }

    row_by_sentence = {
        sentence_index: row
        for row, sentence_index in enumerate(eligible_indices)
    }

    train_rows_lookup = [
        row_by_sentence[index]
        for index in train_indices
    ]
    validation_rows_lookup = [
        row_by_sentence[index]
        for index in validation_indices
    ]

    print()
    print(
        f"{'Feature':35s} "
        f"{'All sent.':>10s} "
        f"{'Train':>10s} "
        f"{'Valid':>10s} "
        f"{'Valid %':>10s}"
    )
    print("-" * 80)

    for feature in stratification_labels:
        column = label2column[feature]
        all_support = int(sentence_matrix[:, column].sum())
        train_support = int(
            sentence_matrix[train_rows_lookup, column].sum()
        )
        validation_support = int(
            sentence_matrix[validation_rows_lookup, column].sum()
        )

        print(
            f"{feature:35s} "
            f"{all_support:10d} "
            f"{train_support:10d} "
            f"{validation_support:10d} "
            f"{validation_support / max(all_support, 1):10.3f}"
        )

    print(
        f"\nActual validation sentence fraction: "
        f"{validation_fraction_actual:.4f}"
    )
    print()

    return best_candidate


# =============================================================================
# Dataset
# =============================================================================

class SemanticFeatureDataset(Dataset):
    """
    RoBERTa multi-label token-classification dataset.

    IMPORTANT:
        labels contain only INDEPENDENT neural targets.

    Labels have shape:
        [sequence_length, number_of_prediction_features]

    For supervised tokens:
        0 = independent feature absent
        1 = independent feature present

    For ignored tokens:
        -100 across all labels

    Only the first RoBERTa subword corresponding to each source token receives
    supervision.
    """

    def __init__(
        self,
        sentences: Sequence[Any],
        sentence_indices: Sequence[int],
        tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
        tokenizer: Any,
        label2id: Dict[str, int],
        max_length: int,
    ) -> None:
        self.sentences = sentences
        self.sentence_indices = list(sentence_indices)
        self.tagged_records = tagged_records
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.num_labels = len(label2id)
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.sentence_indices)

    def __getitem__(
        self,
        dataset_index: int,
    ) -> Dict[str, Any]:
        sentence_index = self.sentence_indices[dataset_index]
        sentence = self.sentences[sentence_index]

        words, token_ids = get_sentence_words(sentence)

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=True,
        )

        word_ids = encoding.word_ids()
        labels: List[List[float]] = []
        previous_word_id: Optional[int] = None
        sentence_records = self.tagged_records.get(sentence_index, {})

        for word_id in word_ids:
            if word_id is None:
                labels.append([-100.0] * self.num_labels)
                previous_word_id = word_id
                continue

            # Ignore all but the first subword.
            if word_id == previous_word_id:
                labels.append([-100.0] * self.num_labels)
                continue

            previous_word_id = word_id

            if not 0 <= word_id < len(token_ids):
                labels.append([-100.0] * self.num_labels)
                continue

            conllu_token_id = token_ids[word_id]
            record = sentence_records.get(conllu_token_id)

            if record is None:
                labels.append([-100.0] * self.num_labels)
                continue

            full_features = clean_features(record)

            if not full_features and EXCLUDE_EMPTY_FEATURE_RECORDS:
                labels.append([-100.0] * self.num_labels)
                continue

            independent_features = reduce_to_independent_features(
                full_features
            )

            target = [0.0] * self.num_labels

            for feature in independent_features:
                label_id = self.label2id.get(feature)

                if label_id is not None:
                    target[label_id] = 1.0

            labels.append(target)

        encoding["labels"] = labels
        encoding["sentence_index"] = sentence_index

        return encoding


# =============================================================================
# Data collator
# =============================================================================

class MultiLabelTokenCollator:
    """Dynamically pad inputs and 3-D multi-label token targets."""

    def __init__(
        self,
        tokenizer: Any,
        num_labels: int,
    ) -> None:
        self.tokenizer = tokenizer
        self.num_labels = num_labels

    def __call__(
        self,
        features: List[Dict[str, Any]],
    ) -> Dict[str, torch.Tensor]:
        clean_features_list: List[Dict[str, Any]] = []
        labels_list: List[List[List[float]]] = []

        for item in features:
            item = dict(item)
            item.pop("sentence_index", None)

            labels = item.pop("labels")
            labels_list.append(labels)
            clean_features_list.append(item)

        batch = self.tokenizer.pad(
            clean_features_list,
            padding=True,
            return_tensors="pt",
        )

        batch_size = len(labels_list)
        max_seq_length = batch["input_ids"].shape[1]

        padded_labels = torch.full(
            (
                batch_size,
                max_seq_length,
                self.num_labels,
            ),
            -100.0,
            dtype=torch.float32,
        )

        for batch_index, labels in enumerate(labels_list):
            label_tensor = torch.tensor(
                labels,
                dtype=torch.float32,
            )

            sequence_length = label_tensor.shape[0]

            padded_labels[
                batch_index,
                :sequence_length,
                :,
            ] = label_tensor

        batch["labels"] = padded_labels

        return batch


# =============================================================================
# Model
# =============================================================================

class RobertaMultiLabelModel(nn.Module):
    """RoBERTa token classifier with BCE loss for independent feature labels."""

    def __init__(
        self,
        model_name: str,
        num_labels: int,
        id2label: Dict[int, str],
        label2id: Dict[str, int],
    ) -> None:
        super().__init__()

        self.model = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            id2label=id2label,
            label2id=label2id,
        )

        self.config = self.model.config

        self.loss_function = nn.BCEWithLogitsLoss(
            reduction="mean"
        )

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        **kwargs,
    ):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs,
        )

        logits = outputs.logits
        loss = None

        if labels is not None:
            valid_mask = labels[:, :, 0] != -100.0

            valid_logits = logits[valid_mask]
            valid_labels = labels[valid_mask]

            if valid_logits.numel() == 0:
                loss = logits.sum() * 0.0
            else:
                loss = self.loss_function(
                    valid_logits,
                    valid_labels,
                )

        return {
            "loss": loss,
            "logits": logits,
        }

    def save_pretrained(
        self,
        save_directory,
        **kwargs,
    ):
        return self.model.save_pretrained(
            save_directory,
            **kwargs,
        )


# =============================================================================
# Hierarchy matrices and array helpers
# =============================================================================

def build_hierarchy_expansion_matrix(
    prediction_labels: Sequence[str],
    full_labels: Sequence[str],
) -> np.ndarray:
    """
    Build a boolean matrix mapping neural outputs to complete feature sets.

    Shape:
        [number_of_prediction_features, number_of_full_features]
    """

    full_label2id = {
        label: index
        for index, label in enumerate(full_labels)
    }

    matrix = np.zeros(
        (len(prediction_labels), len(full_labels)),
        dtype=np.int8,
    )

    for prediction_id, feature in enumerate(prediction_labels):
        expanded = expand_feature_hierarchy({feature})

        for expanded_feature in expanded:
            full_id = full_label2id.get(expanded_feature)

            if full_id is None:
                raise RuntimeError(
                    "Hierarchy expansion produced a feature absent from the "
                    f"full vocabulary: {expanded_feature!r}."
                )

            matrix[prediction_id, full_id] = 1

    return matrix


def expand_binary_feature_matrix(
    independent_matrix: np.ndarray,
    hierarchy_matrix: np.ndarray,
) -> np.ndarray:
    """Expand [tokens, independent] binary values to [tokens, complete]."""

    expanded = independent_matrix.astype(np.int32) @ hierarchy_matrix.astype(
        np.int32
    )

    return (expanded > 0).astype(np.int32)


def extract_valid_token_arrays(
    logits: np.ndarray,
    labels: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return probabilities and true independent labels for supervised tokens."""

    valid_token_mask = labels[:, :, 0] != -100.0

    probabilities = sigmoid_numpy(logits)[
        valid_token_mask
    ]

    true_labels = labels[
        valid_token_mask
    ].astype(np.int32)

    return probabilities, true_labels


def build_full_truth_matrix_for_dataset(
    dataset: "SemanticFeatureDataset",
    full_labels: Sequence[str],
) -> np.ndarray:
    """Build complete source truth for every supervised token in dataset order.

    This is intentionally independent of the neural-output vocabulary.  It is
    therefore safe and necessary for a held-out test corpus that may contain a
    feature such as ``Human`` as a direct independent target even when that
    feature never occurred independently in the training corpus.

    The model is still trained only on training-derived independent output
    heads.  This function merely preserves the complete gold annotations for
    fair hierarchy-expanded evaluation.
    """

    full_label2id = {
        label: index
        for index, label in enumerate(full_labels)
    }

    rows: List[np.ndarray] = []

    for sentence_index in dataset.sentence_indices:
        sentence = dataset.sentences[sentence_index]
        words, token_ids = get_sentence_words(sentence)

        encoding = dataset.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=dataset.max_length,
            add_special_tokens=True,
        )

        word_ids = encoding.word_ids()
        previous_word_id: Optional[int] = None
        sentence_records = dataset.tagged_records.get(sentence_index, {})

        for word_id in word_ids:
            if word_id is None:
                previous_word_id = word_id
                continue

            # Match SemanticFeatureDataset.__getitem__: only the first subword
            # for each source token receives supervision.
            if word_id == previous_word_id:
                continue

            previous_word_id = word_id

            if not 0 <= word_id < len(token_ids):
                continue

            conllu_token_id = token_ids[word_id]
            record = sentence_records.get(conllu_token_id)

            if record is None:
                continue

            features = clean_features(record)

            if not features and EXCLUDE_EMPTY_FEATURE_RECORDS:
                continue

            expanded_features = expand_feature_hierarchy(features)
            row = np.zeros(len(full_labels), dtype=np.int32)

            for feature in expanded_features:
                label_id = full_label2id.get(feature)

                if label_id is None:
                    raise RuntimeError(
                        "Complete source truth contains a feature absent from "
                        f"the training-derived full vocabulary: {feature!r}."
                    )

                row[label_id] = 1

            rows.append(row)

    if not rows:
        return np.zeros((0, len(full_labels)), dtype=np.int32)

    return np.stack(rows, axis=0)


# =============================================================================
# Metrics and thresholds
# =============================================================================

def sigmoid_numpy(x: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid."""

    x = np.clip(x, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-x))


def make_threshold_vector(
    prediction_labels: Sequence[str],
    thresholds: Dict[str, float],
) -> np.ndarray:
    """Return thresholds in neural-output column order."""

    return np.asarray(
        [
            float(
                thresholds.get(
                    label,
                    DEFAULT_PREDICTION_THRESHOLD,
                )
            )
            for label in prediction_labels
        ],
        dtype=np.float32,
    )


def threshold_probabilities(
    probabilities: np.ndarray,
    threshold_vector: np.ndarray,
) -> np.ndarray:
    """Apply one threshold per independent feature."""

    return (
        probabilities >= threshold_vector.reshape(1, -1)
    ).astype(np.int32)


def calculate_average_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    prefix: str = "",
) -> Dict[str, float]:
    """Calculate micro and macro precision/recall/F1."""

    if y_true.shape[0] == 0:
        return {
            f"{prefix}precision_micro": 0.0,
            f"{prefix}recall_micro": 0.0,
            f"{prefix}f1_micro": 0.0,
            f"{prefix}precision_macro": 0.0,
            f"{prefix}recall_macro": 0.0,
            f"{prefix}f1_macro": 0.0,
        }

    (
        precision_micro,
        recall_micro,
        f1_micro,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="micro",
        zero_division=0,
    )

    (
        precision_macro,
        recall_macro,
        f1_macro,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    return {
        f"{prefix}precision_micro": float(precision_micro),
        f"{prefix}recall_micro": float(recall_micro),
        f"{prefix}f1_micro": float(f1_micro),
        f"{prefix}precision_macro": float(precision_macro),
        f"{prefix}recall_macro": float(recall_macro),
        f"{prefix}f1_macro": float(f1_macro),
    }


def calculate_hierarchical_metrics_from_probabilities(
    probabilities: np.ndarray,
    y_true_independent: np.ndarray,
    threshold_vector: np.ndarray,
    hierarchy_matrix: np.ndarray,
    y_true_full: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    """
    Report both direct neural-output metrics and complete hierarchy metrics.

    Unprefixed metrics are the COMPLETE hierarchy metrics.  During ordinary
    training/validation, complete truth can be reconstructed from the
    independent targets.  For a genuinely held-out test corpus, ``y_true_full``
    may instead be supplied directly from the source annotations so an unseen
    direct independent label is not silently erased from the gold truth.
    """

    y_pred_independent = threshold_probabilities(
        probabilities,
        threshold_vector,
    )

    if y_true_full is None:
        y_true_full = expand_binary_feature_matrix(
            y_true_independent,
            hierarchy_matrix,
        )

    if y_true_full.shape[0] != y_true_independent.shape[0]:
        raise RuntimeError(
            "Complete and independent truth matrices contain different "
            f"numbers of supervised tokens: {y_true_full.shape[0]} vs "
            f"{y_true_independent.shape[0]}."
        )

    y_pred_full = expand_binary_feature_matrix(
        y_pred_independent,
        hierarchy_matrix,
    )

    metrics = calculate_average_metrics(
        y_true_full,
        y_pred_full,
        prefix="",
    )

    metrics.update(
        calculate_average_metrics(
            y_true_independent,
            y_pred_independent,
            prefix="direct_",
        )
    )

    return metrics


def make_compute_metrics(
    prediction_labels: Sequence[str],
    hierarchy_matrix: np.ndarray,
):
    """Create Trainer metrics using the fixed default 0.5 thresholds."""

    default_thresholds = np.full(
        len(prediction_labels),
        DEFAULT_PREDICTION_THRESHOLD,
        dtype=np.float32,
    )

    def compute_metrics(eval_prediction) -> Dict[str, float]:
        logits, labels = eval_prediction

        probabilities, y_true_independent = extract_valid_token_arrays(
            logits,
            labels,
        )

        return calculate_hierarchical_metrics_from_probabilities(
            probabilities=probabilities,
            y_true_independent=y_true_independent,
            threshold_vector=default_thresholds,
            hierarchy_matrix=hierarchy_matrix,
        )

    return compute_metrics


def build_threshold_grid() -> np.ndarray:
    """Build an inclusive, stable validation threshold search grid."""

    if THRESHOLD_STEP <= 0:
        raise ValueError("THRESHOLD_STEP must be > 0.")

    if not 0.0 < THRESHOLD_MIN < 1.0:
        raise ValueError("THRESHOLD_MIN must be between 0 and 1.")

    if not 0.0 < THRESHOLD_MAX < 1.0:
        raise ValueError("THRESHOLD_MAX must be between 0 and 1.")

    if THRESHOLD_MIN > THRESHOLD_MAX:
        raise ValueError("THRESHOLD_MIN cannot exceed THRESHOLD_MAX.")

    count = int(
        round(
            (THRESHOLD_MAX - THRESHOLD_MIN)
            / THRESHOLD_STEP
        )
    ) + 1

    return np.linspace(
        THRESHOLD_MIN,
        THRESHOLD_MAX,
        count,
        dtype=np.float64,
    )


def binary_metrics_for_column(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> Tuple[float, float, float, int]:
    """Return binary precision, recall, F1 and positive support."""

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    support = int(y_true.sum())

    return (
        float(precision),
        float(recall),
        float(f1),
        support,
    )


def optimise_feature_thresholds(
    probabilities: np.ndarray,
    y_true_independent: np.ndarray,
    prediction_labels: Sequence[str],
) -> Tuple[Dict[str, float], Dict[str, Dict[str, float]]]:
    """
    Optimise one sigmoid threshold per independent feature on VALIDATION ONLY.

    Objective:
        maximise that feature's binary F1.

    Stability rules:
    - If validation has no positives or no negatives for a feature, keep 0.5.
    - If several thresholds tie for best F1, choose the one closest to 0.5.
      This avoids unnecessarily extreme thresholds for flat validation curves.
    """

    grid = build_threshold_grid()

    thresholds: Dict[str, float] = {}
    diagnostics: Dict[str, Dict[str, float]] = {}

    for label_id, feature in enumerate(prediction_labels):
        y_true = y_true_independent[:, label_id]
        probs = probabilities[:, label_id]

        positive_support = int(y_true.sum())
        negative_support = int((1 - y_true).sum())

        default_pred = (
            probs >= DEFAULT_PREDICTION_THRESHOLD
        ).astype(np.int32)

        (
            default_precision,
            default_recall,
            default_f1,
            _,
        ) = binary_metrics_for_column(
            y_true,
            default_pred,
        )

        if positive_support == 0 or negative_support == 0:
            best_threshold = DEFAULT_PREDICTION_THRESHOLD
        else:
            best_f1 = -1.0
            best_candidates: List[float] = []

            for threshold in grid:
                y_pred = (probs >= threshold).astype(np.int32)

                _, _, f1, _ = binary_metrics_for_column(
                    y_true,
                    y_pred,
                )

                if f1 > best_f1 + 1e-12:
                    best_f1 = f1
                    best_candidates = [float(threshold)]
                elif abs(f1 - best_f1) <= 1e-12:
                    best_candidates.append(float(threshold))

            best_threshold = min(
                best_candidates,
                key=lambda value: (
                    abs(value - DEFAULT_PREDICTION_THRESHOLD),
                    value,
                ),
            )

        best_pred = (
            probs >= best_threshold
        ).astype(np.int32)

        (
            best_precision,
            best_recall,
            best_f1,
            _,
        ) = binary_metrics_for_column(
            y_true,
            best_pred,
        )

        thresholds[feature] = float(best_threshold)

        diagnostics[feature] = {
            "threshold": float(best_threshold),
            "support": positive_support,
            "negative_support": negative_support,
            "precision_default": default_precision,
            "recall_default": default_recall,
            "f1_default": default_f1,
            "precision_optimised": best_precision,
            "recall_optimised": best_recall,
            "f1_optimised": best_f1,
            "f1_change": best_f1 - default_f1,
        }

    return thresholds, diagnostics


def calculate_per_feature_metrics_from_binary(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    labels: Sequence[str],
) -> Dict[str, Dict[str, float]]:
    """Calculate binary precision/recall/F1 independently for each column."""

    results: Dict[str, Dict[str, float]] = {}

    for label_id, feature in enumerate(labels):
        precision, recall, f1, support = binary_metrics_for_column(
            y_true[:, label_id],
            y_pred[:, label_id],
        )

        results[feature] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        }

    return results


# =============================================================================
# Dataset diagnostics and validation
# =============================================================================

def count_feature_occurrences(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_indices: Sequence[int],
    independent_only: bool = False,
) -> Counter:
    """Count complete or independent positive labels in a split."""

    counts: Counter = Counter()

    for sentence_index in sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            if independent_only:
                features = get_independent_record_features(record)
            else:
                features = clean_features(record)

            for feature in features:
                counts[feature] += 1

    return counts


def count_empty_records(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_indices: Sequence[int],
) -> int:
    """Count NOUN/PROPN records having no complete feature tags."""

    count = 0

    for sentence_index in sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            if not clean_features(record):
                count += 1

    return count


def verify_full_label_coverage(
    full_labels: Sequence[str],
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    """Ensure validation/test contain no complete label unseen in training."""

    training_labels = set(full_labels)

    for split_name in ("validation", "test"):
        tagged_records, sentence_indices = split_sources[split_name]
        observed: Set[str] = set()

        for sentence_index in sentence_indices:
            for record in tagged_records.get(sentence_index, {}).values():
                observed.update(clean_features(record))

        unknown = observed - training_labels

        if unknown:
            raise RuntimeError(
                f"{split_name} contains full labels unseen in training: "
                f"{sorted(unknown)}"
            )


def verify_prediction_label_coverage(
    prediction_labels: Sequence[str],
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
    training_file_prediction_labels: Optional[Sequence[str]] = None,
) -> Dict[str, Dict[str, int]]:
    """Check independent-target coverage without leaking test labels.

    Validation remains strict: every validation independent target must have a
    training-derived neural output head.

    Test is different.  A genuine held-out corpus may contain a direct
    independent target that never occurred independently in training.  Such a
    label must NOT be added to the model from the test set.  Instead, report it
    and let complete-feature evaluation use the source gold matrix directly.
    """

    training_labels = set(prediction_labels)
    training_file_labels = (
        set(training_file_prediction_labels)
        if training_file_prediction_labels is not None
        else set(training_labels)
    )

    unseen_test_support: Dict[str, Dict[str, int]] = {}

    for split_name in ("validation", "test"):
        tagged_records, sentence_indices = split_sources[split_name]
        observed_counts: Counter = Counter()

        for sentence_index in sentence_indices:
            for record in tagged_records.get(sentence_index, {}).values():
                for feature in get_independent_record_features(record):
                    observed_counts[feature] += 1

        observed = set(observed_counts)
        unknown = observed - training_labels

        if not unknown:
            continue

        lost_due_to_split = unknown & training_file_labels
        absent_from_training_file = unknown - training_file_labels

        if split_name == "validation":
            details: List[str] = []

            if lost_due_to_split:
                details.append(
                    "present somewhere in the training file but absent from "
                    f"the training subset: {sorted(lost_due_to_split)}"
                )

            if absent_from_training_file:
                details.append(
                    "never occur as independent targets anywhere in the "
                    f"training file: {sorted(absent_from_training_file)}"
                )

            raise RuntimeError(
                "validation contains independent labels with no learnable "
                "training output. "
                + "; ".join(details)
            )

        # For the official test set, do not modify the model vocabulary.
        unseen_test_support = {
            feature: {
                "support": int(observed_counts[feature]),
                "present_in_training_file_as_independent": int(
                    feature in training_file_labels
                ),
            }
            for feature in sorted(unknown)
        }

        print()
        print("=" * 80)
        print("HELD-OUT TEST LABEL COVERAGE WARNING")
        print("=" * 80)
        print(
            "The test corpus contains direct independent targets that have no "
            "training-derived neural output head. They will NOT be added to "
            "the model vocabulary. Complete test truth will be read directly "
            "from the source annotations instead."
        )

        for feature in sorted(unknown):
            print(
                f"  {feature:35s} support={observed_counts[feature]:,}"
            )

        print()

    return unseen_test_support


def verify_hierarchy_closure(
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    """
    Ensure source feature sets already contain all ancestors required by the
    hierarchy. This prevents evaluation against inconsistent source targets.
    """

    problems: List[str] = []
    max_examples = 20

    for split_name in ("train", "validation", "test"):
        tagged_records, sentence_indices = split_sources[split_name]
        problem_count = 0

        for sentence_index in sentence_indices:
            for token_id, record in tagged_records.get(
                sentence_index,
                {},
            ).items():
                features = clean_features(record)

                if not features:
                    continue

                missing = missing_hierarchy_features(features)

                if not missing:
                    continue

                problem_count += 1

                if len(problems) < max_examples:
                    problems.append(
                        f"{split_name}: sentence={sentence_index}, "
                        f"token={token_id}, "
                        f"form={record.get('form')!r}, "
                        f"missing={sorted(missing)}"
                    )

        if problem_count:
            print(
                f"Hierarchy closure problems in {split_name}: "
                f"{problem_count:,} token(s)"
            )

    if problems:
        example_text = "\n".join(
            f"  - {problem}"
            for problem in problems
        )

        raise RuntimeError(
            "The input dataset is not hierarchy-closed. Fix the source tags "
            "before training so deterministic reduction/expansion is exact.\n"
            f"Examples:\n{example_text}"
        )


def print_split_statistics(
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    print("=" * 80)
    print("DATASET SPLIT")
    print("=" * 80)

    for split_name in ("train", "validation", "test"):
        tagged_records, indices = split_sources[split_name]

        full_counts = count_feature_occurrences(
            tagged_records,
            indices,
            independent_only=False,
        )

        independent_counts = count_feature_occurrences(
            tagged_records,
            indices,
            independent_only=True,
        )

        empty_count = count_empty_records(
            tagged_records,
            indices,
        )

        noun_count = sum(
            len(tagged_records.get(index, {}))
            for index in indices
        )

        source_name = (
            "training file"
            if split_name in {"train", "validation"}
            else "test file"
        )

        print(
            f"{split_name:12s}: "
            f"{len(indices):,} sentences | "
            f"{noun_count:,} NOUN/PROPN tokens | "
            f"{sum(full_counts.values()):,} complete positives -> "
            f"{sum(independent_counts.values()):,} independent positives | "
            f"{empty_count:,} empty-feature tokens excluded | "
            f"source={source_name}"
        )

    print()


def print_prediction_label_statistics(
    prediction_labels: Sequence[str],
    full_labels: Sequence[str],
    train_tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    train_sentence_indices: Sequence[int],
) -> None:
    """Show which labels are neural outputs versus deterministic-only parents."""

    train_independent_counts = count_feature_occurrences(
        train_tagged_records,
        train_sentence_indices,
        independent_only=True,
    )

    prediction_set = set(prediction_labels)
    deterministic_only = [
        feature
        for feature in full_labels
        if feature not in prediction_set
    ]

    print("=" * 80)
    print("NEURAL OUTPUT FEATURES")
    print("=" * 80)
    print(
        f"Complete evaluation features: {len(full_labels)}"
    )
    print(
        f"Independent neural outputs: {len(prediction_labels)}"
    )
    print(
        f"Deterministic-only features in this training split: "
        f"{len(deterministic_only)}"
    )
    print()

    print(
        f"{'Feature':35s} "
        f"{'Independent train support':>26s}"
    )
    print("-" * 64)

    for feature in prediction_labels:
        print(
            f"{feature:35s} "
            f"{train_independent_counts[feature]:26d}"
        )

    if deterministic_only:
        print()
        print("Deterministic-only in this split:")

        for feature in deterministic_only:
            print(f"  - {feature}")

    print()


# =============================================================================
# Repeated-run and bootstrap helpers
# =============================================================================

def get_supervised_token_count_for_sentence(
    dataset: "SemanticFeatureDataset",
    sentence_index: int,
) -> int:
    """
    Count supervised first-subword targets for one source sentence.

    This mirrors SemanticFeatureDataset.__getitem__ exactly and is used to map
    flat prediction rows back to sentence clusters for test bootstrapping.
    """

    sentence = dataset.sentences[sentence_index]
    words, token_ids = get_sentence_words(sentence)

    encoding = dataset.tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=dataset.max_length,
        add_special_tokens=True,
    )

    word_ids = encoding.word_ids()
    previous_word_id: Optional[int] = None
    sentence_records = dataset.tagged_records.get(sentence_index, {})

    count = 0

    for word_id in word_ids:
        if word_id is None:
            previous_word_id = word_id
            continue

        if word_id == previous_word_id:
            continue

        previous_word_id = word_id

        if not 0 <= word_id < len(token_ids):
            continue

        conllu_token_id = token_ids[word_id]
        record = sentence_records.get(conllu_token_id)

        if record is None:
            continue

        features = clean_features(record)

        if not features and EXCLUDE_EMPTY_FEATURE_RECORDS:
            continue

        count += 1

    return count


def build_sentence_row_slices(
    dataset: "SemanticFeatureDataset",
) -> List[slice]:
    """
    Map every dataset sentence to its contiguous flat supervised-token rows.
    """

    slices: List[slice] = []
    start = 0

    for sentence_index in dataset.sentence_indices:
        count = get_supervised_token_count_for_sentence(
            dataset,
            sentence_index,
        )
        stop = start + count
        slices.append(slice(start, stop))
        start = stop

    return slices


def summarise_repeated_metrics(
    metric_dicts: Sequence[Dict[str, float]],
) -> Dict[str, Dict[str, float]]:
    """Return mean/SD/min/max across repeated seeded runs."""

    if not metric_dicts:
        return {}

    common_keys = sorted(
        set.intersection(
            *[
                {
                    key
                    for key, value in metrics.items()
                    if isinstance(
                        value,
                        (float, int, np.floating, np.integer),
                    )
                }
                for metrics in metric_dicts
            ]
        )
    )

    summary: Dict[str, Dict[str, float]] = {}

    for key in common_keys:
        values = np.asarray(
            [float(metrics[key]) for metrics in metric_dicts],
            dtype=np.float64,
        )

        summary[key] = {
            "mean": float(values.mean()),
            "sd": float(values.std(ddof=1))
            if len(values) > 1
            else 0.0,
            "minimum": float(values.min()),
            "maximum": float(values.max()),
            "n_runs": int(len(values)),
        }

    return summary


def _metrics_from_confusion_counts(
    tp_by_label: np.ndarray,
    fp_by_label: np.ndarray,
    fn_by_label: np.ndarray,
) -> Dict[str, float]:
    """Calculate six headline metrics directly from labelwise counts."""

    tp_by_label = tp_by_label.astype(np.float64, copy=False)
    fp_by_label = fp_by_label.astype(np.float64, copy=False)
    fn_by_label = fn_by_label.astype(np.float64, copy=False)

    tp = float(tp_by_label.sum())
    fp = float(fp_by_label.sum())
    fn = float(fn_by_label.sum())

    precision_micro = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_micro = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    if precision_micro + recall_micro > 0:
        f1_micro = (
            2.0
            * precision_micro
            * recall_micro
            / (precision_micro + recall_micro)
        )
    else:
        f1_micro = 0.0

    precision_den = tp_by_label + fp_by_label
    recall_den = tp_by_label + fn_by_label

    precision_by_label = np.divide(
        tp_by_label,
        precision_den,
        out=np.zeros_like(tp_by_label, dtype=np.float64),
        where=precision_den > 0,
    )
    recall_by_label = np.divide(
        tp_by_label,
        recall_den,
        out=np.zeros_like(tp_by_label, dtype=np.float64),
        where=recall_den > 0,
    )

    f1_den = precision_by_label + recall_by_label
    f1_by_label = np.divide(
        2.0 * precision_by_label * recall_by_label,
        f1_den,
        out=np.zeros_like(precision_by_label, dtype=np.float64),
        where=f1_den > 0,
    )

    return {
        "precision_micro": float(precision_micro),
        "recall_micro": float(recall_micro),
        "f1_micro": float(f1_micro),
        "precision_macro": float(precision_by_label.mean()),
        "recall_macro": float(recall_by_label.mean()),
        "f1_macro": float(f1_by_label.mean()),
    }


def build_sentence_confusion_arrays(
    y_true_full: np.ndarray,
    run_predictions_full: Sequence[np.ndarray],
    sentence_slices: Sequence[slice],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Precompute TP/FP/FN by [run, sentence, label] for fast cluster bootstrap.
    """

    n_runs = len(run_predictions_full)
    n_sentences = len(sentence_slices)
    n_labels = y_true_full.shape[1]

    tp = np.zeros(
        (n_runs, n_sentences, n_labels),
        dtype=np.int32,
    )
    fp = np.zeros_like(tp)
    fn = np.zeros_like(tp)

    for sentence_pos, row_slice in enumerate(sentence_slices):
        y_true_sentence = y_true_full[row_slice]

        for run_pos, y_pred_full in enumerate(run_predictions_full):
            y_pred_sentence = y_pred_full[row_slice]

            tp[run_pos, sentence_pos] = (
                (y_true_sentence == 1)
                & (y_pred_sentence == 1)
            ).sum(axis=0)

            fp[run_pos, sentence_pos] = (
                (y_true_sentence == 0)
                & (y_pred_sentence == 1)
            ).sum(axis=0)

            fn[run_pos, sentence_pos] = (
                (y_true_sentence == 1)
                & (y_pred_sentence == 0)
            ).sum(axis=0)

    return tp, fp, fn


def bootstrap_mean_test_metrics(
    y_true_full: np.ndarray,
    run_predictions_full: Sequence[np.ndarray],
    sentence_slices: Sequence[slice],
) -> Dict[str, Any]:
    """
    Sentence-cluster bootstrap CIs for the MEAN performance across run seeds.

    Each bootstrap replicate:
      1. samples test sentences with replacement;
      2. applies the SAME sampled sentence multiplicities to every trained run;
      3. computes each run's six hierarchy-expanded metrics;
      4. averages each metric across runs.

    Thus the interval quantifies held-out TEST-SAMPLE uncertainty around the
    mean performance of the repeated-run procedure. Training-seed variability is
    reported separately via mean and SD across runs.
    """

    if TEST_BOOTSTRAP_SAMPLES < 1:
        raise ValueError("TEST_BOOTSTRAP_SAMPLES must be at least 1.")

    if not 0.0 < TEST_BOOTSTRAP_CONFIDENCE < 1.0:
        raise ValueError(
            "TEST_BOOTSTRAP_CONFIDENCE must be strictly between 0 and 1."
        )

    if not run_predictions_full:
        raise ValueError(
            "At least one run prediction matrix is required for bootstrap."
        )

    n_sentences = len(sentence_slices)

    if n_sentences == 0:
        raise ValueError("No test sentences are available for bootstrap.")

    expected_rows = sum(
        row_slice.stop - row_slice.start
        for row_slice in sentence_slices
    )

    if expected_rows != y_true_full.shape[0]:
        raise RuntimeError(
            "Sentence slices do not cover the complete test truth matrix: "
            f"{expected_rows:,} rows vs {y_true_full.shape[0]:,}."
        )

    for run_pos, y_pred_full in enumerate(run_predictions_full):
        if y_pred_full.shape != y_true_full.shape:
            raise RuntimeError(
                f"Run {run_pos} prediction shape {y_pred_full.shape} does "
                f"not match test truth shape {y_true_full.shape}."
            )

    tp, fp, fn = build_sentence_confusion_arrays(
        y_true_full,
        run_predictions_full,
        sentence_slices,
    )

    rng = np.random.default_rng(TEST_BOOTSTRAP_SEED)

    metric_names = [
        "precision_micro",
        "recall_micro",
        "f1_micro",
        "precision_macro",
        "recall_macro",
        "f1_macro",
    ]

    bootstrap_values = {
        metric: np.empty(TEST_BOOTSTRAP_SAMPLES, dtype=np.float64)
        for metric in metric_names
    }

    for bootstrap_index in range(TEST_BOOTSTRAP_SAMPLES):
        sampled_sentences = rng.integers(
            0,
            n_sentences,
            size=n_sentences,
        )

        multiplicities = np.bincount(
            sampled_sentences,
            minlength=n_sentences,
        ).astype(np.int32)

        run_metric_values = {
            metric: []
            for metric in metric_names
        }

        for run_pos in range(len(run_predictions_full)):
            tp_by_label = np.tensordot(
                multiplicities,
                tp[run_pos],
                axes=(0, 0),
            )
            fp_by_label = np.tensordot(
                multiplicities,
                fp[run_pos],
                axes=(0, 0),
            )
            fn_by_label = np.tensordot(
                multiplicities,
                fn[run_pos],
                axes=(0, 0),
            )

            metrics = _metrics_from_confusion_counts(
                tp_by_label,
                fp_by_label,
                fn_by_label,
            )

            for metric in metric_names:
                run_metric_values[metric].append(metrics[metric])

        for metric in metric_names:
            bootstrap_values[metric][bootstrap_index] = float(
                np.mean(run_metric_values[metric])
            )

    alpha = 1.0 - TEST_BOOTSTRAP_CONFIDENCE
    lower_q = alpha / 2.0
    upper_q = 1.0 - lower_q

    intervals: Dict[str, Dict[str, float]] = {}

    for metric in metric_names:
        values = bootstrap_values[metric]

        intervals[metric] = {
            "bootstrap_mean": float(values.mean()),
            "bootstrap_sd": float(values.std(ddof=1))
            if len(values) > 1
            else 0.0,
            "ci_lower": float(np.quantile(values, lower_q)),
            "ci_upper": float(np.quantile(values, upper_q)),
            "confidence": float(TEST_BOOTSTRAP_CONFIDENCE),
            "bootstrap_samples": int(TEST_BOOTSTRAP_SAMPLES),
        }

    return {
        "bootstrap_unit": "sentence",
        "statistic": "mean_metric_across_repeated_training_runs",
        "seed": TEST_BOOTSTRAP_SEED,
        "n_test_sentences": n_sentences,
        "n_runs": len(run_predictions_full),
        "intervals": intervals,
    }


def make_json_serialisable(value: Any) -> Any:
    """Recursively convert NumPy scalars/arrays to ordinary JSON values."""

    if isinstance(value, dict):
        return {
            str(key): make_json_serialisable(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            make_json_serialisable(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, (np.floating, np.integer)):
        return value.item()

    return value


# =============================================================================
# Reporting helpers
# =============================================================================

def print_metric_dictionary(
    metrics: Dict[str, float],
) -> None:
    for key, value in sorted(metrics.items()):
        print(f"{key}: {value}")


def print_threshold_table(
    prediction_labels: Sequence[str],
    diagnostics: Dict[str, Dict[str, float]],
) -> None:
    print()
    print("=" * 100)
    print("VALIDATION-OPTIMISED THRESHOLDS")
    print("=" * 100)

    heading = (
        f"{'Feature':35s} "
        f"{'Threshold':>10s} "
        f"{'Support':>9s} "
        f"{'F1@0.5':>10s} "
        f"{'F1 opt':>10s} "
        f"{'Change':>10s}"
    )

    print(heading)
    print("-" * len(heading))

    for feature in prediction_labels:
        result = diagnostics[feature]

        print(
            f"{feature:35s} "
            f"{result['threshold']:10.2f} "
            f"{int(result['support']):9d} "
            f"{result['f1_default']:10.4f} "
            f"{result['f1_optimised']:10.4f} "
            f"{result['f1_change']:+10.4f}"
        )


def print_per_feature_table(
    title: str,
    labels: Sequence[str],
    results: Dict[str, Dict[str, float]],
) -> None:
    print()
    print("=" * 80)
    print(title)
    print("=" * 80)

    heading = (
        f"{'Feature':35s} "
        f"{'Precision':>10s} "
        f"{'Recall':>10s} "
        f"{'F1':>10s} "
        f"{'Support':>10s}"
    )

    print(heading)
    print("-" * len(heading))

    for feature in labels:
        result = results[feature]

        print(
            f"{feature:35s} "
            f"{result['precision']:10.4f} "
            f"{result['recall']:10.4f} "
            f"{result['f1']:10.4f} "
            f"{result['support']:10d}"
        )


def summarise_repeated_per_feature_metrics(
    run_documents: Sequence[Dict[str, Any]],
    result_key: str,
    labels: Sequence[str],
) -> Dict[str, Dict[str, float]]:
    """Summarise per-feature test precision/recall/F1 across repeated runs."""

    summary: Dict[str, Dict[str, float]] = {}

    for feature in labels:
        precision_values = np.asarray(
            [
                run_document[result_key][feature]["precision"]
                for run_document in run_documents
            ],
            dtype=np.float64,
        )
        recall_values = np.asarray(
            [
                run_document[result_key][feature]["recall"]
                for run_document in run_documents
            ],
            dtype=np.float64,
        )
        f1_values = np.asarray(
            [
                run_document[result_key][feature]["f1"]
                for run_document in run_documents
            ],
            dtype=np.float64,
        )
        supports = {
            int(run_document[result_key][feature]["support"])
            for run_document in run_documents
        }

        if len(supports) != 1:
            raise RuntimeError(
                f"Test support changed across runs for feature {feature!r}: "
                f"{sorted(supports)}"
            )

        summary[feature] = {
            "precision_mean": float(precision_values.mean()),
            "precision_sd": float(precision_values.std(ddof=1))
            if len(precision_values) > 1
            else 0.0,
            "recall_mean": float(recall_values.mean()),
            "recall_sd": float(recall_values.std(ddof=1))
            if len(recall_values) > 1
            else 0.0,
            "f1_mean": float(f1_values.mean()),
            "f1_sd": float(f1_values.std(ddof=1))
            if len(f1_values) > 1
            else 0.0,
            "f1_min": float(f1_values.min()),
            "f1_max": float(f1_values.max()),
            "support": int(next(iter(supports))),
        }

    return summary


def print_repeated_per_feature_table(
    title: str,
    labels: Sequence[str],
    summary: Dict[str, Dict[str, float]],
) -> None:
    """Print repeated-run mean/SD test performance for each feature."""

    print()
    print("=" * 118)
    print(title)
    print("=" * 118)

    heading = (
        f"{'Feature':35s} "
        f"{'P mean':>8s} "
        f"{'P SD':>8s} "
        f"{'R mean':>8s} "
        f"{'R SD':>8s} "
        f"{'F1 mean':>8s} "
        f"{'F1 SD':>8s} "
        f"{'F1 min':>8s} "
        f"{'F1 max':>8s} "
        f"{'Support':>8s}"
    )

    print(heading)
    print("-" * len(heading))

    for feature in labels:
        result = summary[feature]
        print(
            f"{feature:35s} "
            f"{result['precision_mean']:8.4f} "
            f"{result['precision_sd']:8.4f} "
            f"{result['recall_mean']:8.4f} "
            f"{result['recall_sd']:8.4f} "
            f"{result['f1_mean']:8.4f} "
            f"{result['f1_sd']:8.4f} "
            f"{result['f1_min']:8.4f} "
            f"{result['f1_max']:8.4f} "
            f"{int(result['support']):8d}"
        )


# =============================================================================
# Main
# =============================================================================

def main() -> None:
    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )
    RUNS_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    validate_feature_hierarchy()
    check_cuda()

    if not RUN_SEEDS:
        raise ValueError("RUN_SEEDS must contain at least one integer seed.")

    if len(set(RUN_SEEDS)) != len(RUN_SEEDS):
        raise ValueError("RUN_SEEDS contains duplicate seeds.")

    # -------------------------------------------------------------------------
    # Load independent TRAIN and held-out TEST corpora
    # -------------------------------------------------------------------------

    print("=" * 80)
    print("LOADING DATA")
    print("=" * 80)

    train_sentences = load_conllu(TRAIN_CONLLU_PATH)
    test_sentences = load_conllu(TEST_CONLLU_PATH)

    train_tagged_records = load_tagged_records(TRAIN_JSONL_PATH)
    test_tagged_records = load_tagged_records(TEST_JSONL_PATH)

    validate_feature_switches(
        train_tagged_records,
        test_tagged_records,
    )

    print(
        "Training CoNLL-U sentences: "
        f"{len(train_sentences):,}"
    )
    print(
        "Test CoNLL-U sentences: "
        f"{len(test_sentences):,}"
    )
    print(
        "Training sentences represented in tagged JSONL: "
        f"{len(train_tagged_records):,}"
    )
    print(
        "Test sentences represented in tagged JSONL: "
        f"{len(test_tagged_records):,}"
    )

    train_target_record_count = sum(
        len(records)
        for records in train_tagged_records.values()
    )
    test_target_record_count = sum(
        len(records)
        for records in test_tagged_records.values()
    )

    print(
        "Training tagged NOUN/PROPN records: "
        f"{train_target_record_count:,}"
    )
    print(
        "Test tagged NOUN/PROPN records: "
        f"{test_target_record_count:,}"
    )
    print()

    # -------------------------------------------------------------------------
    # Fixed sentence-level TRAIN/VALIDATION split; TEST remains untouched
    # -------------------------------------------------------------------------

    eligible_train_indices = get_eligible_sentence_indices(
        train_sentences,
        train_tagged_records,
    )
    eligible_test_indices = get_eligible_sentence_indices(
        test_sentences,
        test_tagged_records,
    )

    print(
        "Eligible training-file sentences with at least one non-empty "
        "feature target: "
        f"{len(eligible_train_indices):,}"
    )
    print(
        "Eligible test-file sentences with at least one non-empty "
        "feature target: "
        f"{len(eligible_test_indices):,}"
    )

    train_validation_split = make_train_validation_split(
        eligible_train_indices,
        train_tagged_records,
    )

    train_indices = train_validation_split["train"]
    validation_indices = train_validation_split["validation"]
    test_indices = list(eligible_test_indices)

    split_sources = {
        "train": (train_tagged_records, train_indices),
        "validation": (train_tagged_records, validation_indices),
        "test": (test_tagged_records, test_indices),
    }

    split_document = {
        "split_unit": "sentence",
        "stratification": "iterative_multilabel_sentence_presence",
        "split_seed_requested": SPLIT_SEED,
        "split_seed_used": train_validation_split["split_seed_used"],
        "split_attempt": train_validation_split["attempt"],
        "validation_fraction": VALIDATION_FRACTION,
        "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
        "prevalence_error": train_validation_split["prevalence_error"],
        "stratification_labels": (
            train_validation_split["stratification_labels"]
        ),
        "train_jsonl": str(TRAIN_JSONL_PATH),
        "test_jsonl": str(TEST_JSONL_PATH),
        "train_conllu": str(TRAIN_CONLLU_PATH),
        "test_conllu": str(TEST_CONLLU_PATH),
        "index_spaces": {
            "train": "training corpus",
            "validation": "training corpus",
            "test": "test corpus",
        },
        "train": train_indices,
        "validation": validation_indices,
        "test": test_indices,
    }

    with SPLIT_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            make_json_serialisable(split_document),
            handle,
            ensure_ascii=False,
            indent=2,
        )

    # The reduction is exact only when source data are hierarchy-closed.
    verify_hierarchy_closure(split_sources)

    # -------------------------------------------------------------------------
    # Separate complete and neural-output vocabularies
    # -------------------------------------------------------------------------

    training_file_prediction_labels = build_prediction_label_vocabulary(
        train_tagged_records,
        eligible_train_indices,
    )

    # Vocabularies are derived from TRAIN only, never validation/test.
    full_labels = build_full_label_vocabulary(
        train_tagged_records,
        train_indices,
    )

    prediction_labels = build_prediction_label_vocabulary(
        train_tagged_records,
        train_indices,
    )

    verify_full_label_coverage(
        full_labels,
        split_sources,
    )

    unseen_test_independent_labels = verify_prediction_label_coverage(
        prediction_labels,
        split_sources,
        training_file_prediction_labels=training_file_prediction_labels,
    )

    prediction_label2id = {
        label: index
        for index, label in enumerate(prediction_labels)
    }

    prediction_id2label = {
        index: label
        for label, index in prediction_label2id.items()
    }

    full_label2id = {
        label: index
        for index, label in enumerate(full_labels)
    }

    hierarchy_matrix = build_hierarchy_expansion_matrix(
        prediction_labels,
        full_labels,
    )

    label_document = {
        "labels": prediction_labels,
        "label2id": prediction_label2id,
        "id2label": {
            str(key): value
            for key, value in prediction_id2label.items()
        },
        "prediction_labels": prediction_labels,
        "prediction_label2id": prediction_label2id,
        "full_labels": full_labels,
        "full_label2id": full_label2id,
        "feature_parents": {
            child: sorted(parents)
            for child, parents in sorted(FEATURE_PARENTS.items())
        },
        "target_representation": "per_token_independent_features",
        "split_unit": "sentence",
        "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
        "disabled_features": sorted(
            feature
            for feature, enabled in FEATURE_SWITCHES.items()
            if not enabled
        ),
    }

    with LABELS_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            label_document,
            handle,
            ensure_ascii=False,
            indent=2,
        )

    print_split_statistics(split_sources)

    print_prediction_label_statistics(
        prediction_labels,
        full_labels,
        train_tagged_records,
        train_indices,
    )

    # -------------------------------------------------------------------------
    # Tokeniser and fixed datasets
    # -------------------------------------------------------------------------

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        use_fast=True,
        add_prefix_space=True,
    )

    train_dataset = SemanticFeatureDataset(
        sentences=train_sentences,
        sentence_indices=train_indices,
        tagged_records=train_tagged_records,
        tokenizer=tokenizer,
        label2id=prediction_label2id,
        max_length=MAX_LENGTH,
    )

    validation_dataset = SemanticFeatureDataset(
        sentences=train_sentences,
        sentence_indices=validation_indices,
        tagged_records=train_tagged_records,
        tokenizer=tokenizer,
        label2id=prediction_label2id,
        max_length=MAX_LENGTH,
    )

    test_dataset = SemanticFeatureDataset(
        sentences=test_sentences,
        sentence_indices=test_indices,
        tagged_records=test_tagged_records,
        tokenizer=tokenizer,
        label2id=prediction_label2id,
        max_length=MAX_LENGTH,
    )

    data_collator = MultiLabelTokenCollator(
        tokenizer=tokenizer,
        num_labels=len(prediction_labels),
    )

    # Complete held-out gold truth is fixed across all training runs.
    test_true_full = build_full_truth_matrix_for_dataset(
        test_dataset,
        full_labels,
    )

    test_sentence_slices = build_sentence_row_slices(
        test_dataset,
    )

    test_slice_rows = sum(
        row_slice.stop - row_slice.start
        for row_slice in test_sentence_slices
    )

    if test_slice_rows != test_true_full.shape[0]:
        raise RuntimeError(
            "Test sentence-cluster mapping does not align with source truth: "
            f"{test_slice_rows:,} mapped rows vs "
            f"{test_true_full.shape[0]:,} truth rows."
        )

    # -------------------------------------------------------------------------
    # Repeated fine-tuning runs
    # -------------------------------------------------------------------------

    print("=" * 80)
    print("REPEATED FINE-TUNING")
    print("=" * 80)
    print(f"Runs: {len(RUN_SEEDS)}")
    print(f"Seeds: {RUN_SEEDS}")
    print(
        "Train/validation split is fixed across all runs; only the "
        "fine-tuning seed changes."
    )
    print()

    use_bf16 = torch.cuda.is_bf16_supported()

    all_run_documents: List[Dict[str, Any]] = []
    all_test_metric_dicts: List[Dict[str, float]] = []
    all_validation_metric_dicts: List[Dict[str, float]] = []
    all_test_predictions_full: List[np.ndarray] = []

    work_root = OUTPUT_DIR / "_trainer_work"
    work_root.mkdir(parents=True, exist_ok=True)

    for run_number, run_seed in enumerate(RUN_SEEDS, start=1):
        print()
        print("=" * 100)
        print(
            f"RUN {run_number}/{len(RUN_SEEDS)} | "
            f"SEED {run_seed}"
        )
        print("=" * 100)

        set_seed(run_seed)

        run_work_dir = work_root / f"seed_{run_seed}"
        run_result_path = RUNS_DIR / f"seed_{run_seed}.json"

        if run_work_dir.exists():
            shutil.rmtree(run_work_dir)

        model = RobertaMultiLabelModel(
            model_name=MODEL_NAME,
            num_labels=len(prediction_labels),
            id2label=prediction_id2label,
            label2id=prediction_label2id,
        )

        training_args = TrainingArguments(
            output_dir=str(run_work_dir),

            learning_rate=LEARNING_RATE,
            lr_scheduler_type="linear",
            weight_decay=WEIGHT_DECAY,

            per_device_train_batch_size=TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=EVAL_BATCH_SIZE,

            gradient_accumulation_steps=(
                GRADIENT_ACCUMULATION_STEPS
            ),

            num_train_epochs=NUM_EPOCHS,

            eval_strategy=EVAL_STRATEGY,
            save_strategy="epoch",
            logging_strategy="steps",
            logging_steps=50,

            load_best_model_at_end=True,

            # Model selection is performed on VALIDATION only, using complete
            # hierarchy-expanded macro F1 at the fixed 0.5 threshold.
            metric_for_best_model="f1_macro",
            greater_is_better=True,

            save_total_limit=2,

            bf16=use_bf16,
            fp16=not use_bf16,

            dataloader_num_workers=4,
            dataloader_pin_memory=True,

            optim="adamw_torch",
            report_to="none",

            seed=run_seed,
            data_seed=run_seed,

            remove_unused_columns=False,
            eval_accumulation_steps=16,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=validation_dataset,
            processing_class=tokenizer,
            data_collator=data_collator,
            compute_metrics=make_compute_metrics(
                prediction_labels,
                hierarchy_matrix,
            ),
        )

        trainer.train()

        # ---------------------------------------------------------------------
        # Validation: first report 0.5, then calibrate thresholds on validation
        # ---------------------------------------------------------------------

        validation_output = trainer.predict(
            validation_dataset,
            metric_key_prefix="validation_default",
        )

        (
            validation_probabilities,
            validation_true_independent,
        ) = extract_valid_token_arrays(
            validation_output.predictions,
            validation_output.label_ids,
        )

        default_threshold_vector = np.full(
            len(prediction_labels),
            DEFAULT_PREDICTION_THRESHOLD,
            dtype=np.float32,
        )

        validation_default_metrics = (
            calculate_hierarchical_metrics_from_probabilities(
                probabilities=validation_probabilities,
                y_true_independent=validation_true_independent,
                threshold_vector=default_threshold_vector,
                hierarchy_matrix=hierarchy_matrix,
            )
        )

        (
            optimised_thresholds,
            threshold_diagnostics,
        ) = optimise_feature_thresholds(
            probabilities=validation_probabilities,
            y_true_independent=validation_true_independent,
            prediction_labels=prediction_labels,
        )

        threshold_vector = make_threshold_vector(
            prediction_labels,
            optimised_thresholds,
        )

        validation_optimised_metrics = (
            calculate_hierarchical_metrics_from_probabilities(
                probabilities=validation_probabilities,
                y_true_independent=validation_true_independent,
                threshold_vector=threshold_vector,
                hierarchy_matrix=hierarchy_matrix,
            )
        )

        print_threshold_table(
            prediction_labels,
            threshold_diagnostics,
        )

        print()
        print("VALIDATION @ 0.5")
        print_metric_dictionary(validation_default_metrics)

        print()
        print("VALIDATION WITH VALIDATION-CALIBRATED THRESHOLDS")
        print_metric_dictionary(validation_optimised_metrics)

        # ---------------------------------------------------------------------
        # Held-out TEST: thresholds are frozen
        # ---------------------------------------------------------------------

        test_output = trainer.predict(
            test_dataset,
            metric_key_prefix="test_default",
        )

        test_probabilities, test_true_independent = (
            extract_valid_token_arrays(
                test_output.predictions,
                test_output.label_ids,
            )
        )

        if test_true_full.shape[0] != test_probabilities.shape[0]:
            raise RuntimeError(
                "Test source-truth alignment failed: predictions contain "
                f"{test_probabilities.shape[0]:,} supervised tokens but "
                f"source truth contains {test_true_full.shape[0]:,}."
            )

        # Explicit test-at-0.5 diagnostic.
        test_default_metrics = (
            calculate_hierarchical_metrics_from_probabilities(
                probabilities=test_probabilities,
                y_true_independent=test_true_independent,
                threshold_vector=default_threshold_vector,
                hierarchy_matrix=hierarchy_matrix,
                y_true_full=test_true_full,
            )
        )

        # Final per-run test result uses validation-calibrated thresholds.
        test_optimised_metrics = (
            calculate_hierarchical_metrics_from_probabilities(
                probabilities=test_probabilities,
                y_true_independent=test_true_independent,
                threshold_vector=threshold_vector,
                hierarchy_matrix=hierarchy_matrix,
                y_true_full=test_true_full,
            )
        )

        test_pred_independent = threshold_probabilities(
            test_probabilities,
            threshold_vector,
        )

        test_pred_full = expand_binary_feature_matrix(
            test_pred_independent,
            hierarchy_matrix,
        )

        all_test_predictions_full.append(
            test_pred_full.astype(np.int8, copy=True)
        )

        per_feature_full = calculate_per_feature_metrics_from_binary(
            y_true=test_true_full,
            y_pred=test_pred_full,
            labels=full_labels,
        )

        per_feature_independent = (
            calculate_per_feature_metrics_from_binary(
                y_true=test_true_independent,
                y_pred=test_pred_independent,
                labels=prediction_labels,
            )
        )

        print()
        print("TEST @ 0.5 (DIAGNOSTIC)")
        print_metric_dictionary(test_default_metrics)

        print()
        print("TEST WITH FROZEN VALIDATION-CALIBRATED THRESHOLDS")
        print_metric_dictionary(test_optimised_metrics)

        if PRINT_PER_RUN_FEATURE_TABLES:
            print_per_feature_table(
                title=(
                    f"RUN {run_number} / SEED {run_seed}: TEST PERFORMANCE "
                    "BY COMPLETE FEATURE (AFTER INHERITANCE)"
                ),
                labels=full_labels,
                results=per_feature_full,
            )
            print_per_feature_table(
                title=(
                    f"RUN {run_number} / SEED {run_seed}: TEST PERFORMANCE "
                    "BY INDEPENDENT NEURAL OUTPUT"
                ),
                labels=prediction_labels,
                results=per_feature_independent,
            )

        # ---------------------------------------------------------------------
        # Save a neutral reference model, not a cherry-picked best seed
        # ---------------------------------------------------------------------

        is_reference_run = (
            SAVE_REFERENCE_MODEL
            and run_number == 1
        )

        if is_reference_run:
            if REFERENCE_MODEL_DIR.exists():
                shutil.rmtree(REFERENCE_MODEL_DIR)

            trainer.save_model(str(REFERENCE_MODEL_DIR))
            tokenizer.save_pretrained(str(REFERENCE_MODEL_DIR))

            with (
                REFERENCE_MODEL_DIR / "semantic_features.json"
            ).open(
                "w",
                encoding="utf-8",
            ) as handle:
                json.dump(
                    label_document,
                    handle,
                    ensure_ascii=False,
                    indent=2,
                )

            reference_threshold_document = {
                "seed": run_seed,
                "selection_policy": (
                    "first_predeclared_seed_not_best_test_run"
                ),
                "optimised_on": "validation",
                "objective": "per_feature_binary_f1",
                "thresholds": optimised_thresholds,
                "diagnostics": threshold_diagnostics,
            }

            with (
                REFERENCE_MODEL_DIR / "prediction_thresholds.json"
            ).open(
                "w",
                encoding="utf-8",
            ) as handle:
                json.dump(
                    make_json_serialisable(
                        reference_threshold_document
                    ),
                    handle,
                    ensure_ascii=False,
                    indent=2,
                )

        run_document = {
            "run_number": run_number,
            "seed": run_seed,
            "is_reference_model": is_reference_run,
            "model": MODEL_NAME,
            "target_representation": (
                "per_token_independent_features"
            ),
            "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
            "disabled_features": sorted(
                feature
                for feature, enabled in FEATURE_SWITCHES.items()
                if not enabled
            ),
            "split": {
                "unit": "sentence",
                "split_seed": (
                    train_validation_split["split_seed_used"]
                ),
                "validation_fraction": VALIDATION_FRACTION,
                "train_sentences": len(train_indices),
                "validation_sentences": len(validation_indices),
                "test_sentences": len(test_indices),
            },
            "threshold_calibration": {
                "data": "validation_only",
                "objective": "per_feature_binary_f1",
                "thresholds": optimised_thresholds,
                "diagnostics": threshold_diagnostics,
            },
            "validation_metrics_at_0_5": (
                validation_default_metrics
            ),
            "validation_metrics_optimised": (
                validation_optimised_metrics
            ),
            "test_metrics_at_0_5": test_default_metrics,
            "test_metrics_optimised": test_optimised_metrics,
            "per_feature_complete": per_feature_full,
            "per_feature_independent": (
                per_feature_independent
            ),
            "unseen_test_independent_labels": (
                unseen_test_independent_labels
            ),
        }

        with run_result_path.open(
            "w",
            encoding="utf-8",
        ) as handle:
            json.dump(
                make_json_serialisable(run_document),
                handle,
                ensure_ascii=False,
                indent=2,
            )

        all_run_documents.append(run_document)
        all_test_metric_dicts.append(test_optimised_metrics)
        all_validation_metric_dicts.append(
            validation_optimised_metrics
        )

        # Release run-specific GPU state before initialising the next model.
        del test_output
        del validation_output
        del trainer
        del model

        gc.collect()
        torch.cuda.empty_cache()

        # Checkpoints were needed only to recover the best validation epoch.
        # Remove them after each run to avoid storing dozens of RoBERTa-large
        # copies. The neutral first-seed reference model was saved separately.
        if run_work_dir.exists():
            shutil.rmtree(run_work_dir)

    if work_root.exists() and not any(work_root.iterdir()):
        work_root.rmdir()

    # -------------------------------------------------------------------------
    # Repeated-run summary: stochastic training variability
    # -------------------------------------------------------------------------

    validation_summary = summarise_repeated_metrics(
        all_validation_metric_dicts
    )
    test_summary = summarise_repeated_metrics(
        all_test_metric_dicts
    )

    print()
    print("=" * 100)
    print("REPEATED-RUN TEST SUMMARY")
    print("=" * 100)

    headline_metrics = [
        "precision_micro",
        "recall_micro",
        "f1_micro",
        "precision_macro",
        "recall_macro",
        "f1_macro",
    ]

    print(
        f"{'Metric':25s} "
        f"{'Mean':>10s} "
        f"{'SD':>10s} "
        f"{'Min':>10s} "
        f"{'Max':>10s}"
    )
    print("-" * 70)

    for metric in headline_metrics:
        result = test_summary[metric]

        print(
            f"{metric:25s} "
            f"{result['mean']:10.4f} "
            f"{result['sd']:10.4f} "
            f"{result['minimum']:10.4f} "
            f"{result['maximum']:10.4f}"
        )

    # -------------------------------------------------------------------------
    # Repeated-run per-feature test summaries
    # -------------------------------------------------------------------------

    per_feature_complete_summary = summarise_repeated_per_feature_metrics(
        all_run_documents,
        result_key="per_feature_complete",
        labels=full_labels,
    )
    per_feature_independent_summary = summarise_repeated_per_feature_metrics(
        all_run_documents,
        result_key="per_feature_independent",
        labels=prediction_labels,
    )

    print_repeated_per_feature_table(
        title=(
            "REPEATED-RUN TEST PERFORMANCE BY COMPLETE FEATURE "
            "(AFTER INHERITANCE)"
        ),
        labels=full_labels,
        summary=per_feature_complete_summary,
    )

    print_repeated_per_feature_table(
        title=(
            "REPEATED-RUN TEST PERFORMANCE BY INDEPENDENT NEURAL OUTPUT"
        ),
        labels=prediction_labels,
        summary=per_feature_independent_summary,
    )

    # -------------------------------------------------------------------------
    # Sentence-cluster bootstrap: held-out test-sample uncertainty
    # -------------------------------------------------------------------------

    print()
    print("=" * 100)
    print("SENTENCE-CLUSTER BOOTSTRAP ON HELD-OUT TEST SET")
    print("=" * 100)

    bootstrap_results = bootstrap_mean_test_metrics(
        y_true_full=test_true_full,
        run_predictions_full=all_test_predictions_full,
        sentence_slices=test_sentence_slices,
    )

    print(
        f"Bootstrap samples: {TEST_BOOTSTRAP_SAMPLES:,}"
    )
    print(
        f"Confidence level : "
        f"{TEST_BOOTSTRAP_CONFIDENCE:.1%}"
    )
    print(
        "Bootstrap unit   : sentence (all eligible NOUN/PROPN targets "
        "from a sampled sentence move together)"
    )
    print()

    ci_percent = int(round(TEST_BOOTSTRAP_CONFIDENCE * 100))

    print(
        f"{'Metric':25s} "
        f"{'Run mean':>10s} "
        f"{'Run SD':>10s} "
        f"{str(ci_percent) + '% CI low':>12s} "
        f"{str(ci_percent) + '% CI high':>12s}"
    )
    print("-" * 78)

    for metric in headline_metrics:
        repeated = test_summary[metric]
        interval = bootstrap_results["intervals"][metric]

        print(
            f"{metric:25s} "
            f"{repeated['mean']:10.4f} "
            f"{repeated['sd']:10.4f} "
            f"{interval['ci_lower']:12.4f} "
            f"{interval['ci_upper']:12.4f}"
        )

    # -------------------------------------------------------------------------
    # Save experiment-level results
    # -------------------------------------------------------------------------

    bootstrap_document = {
        "model": MODEL_NAME,
        "run_seeds": RUN_SEEDS,
        "split_seed": train_validation_split["split_seed_used"],
        "test_bootstrap": bootstrap_results,
    }

    with BOOTSTRAP_RESULTS_PATH.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            make_json_serialisable(bootstrap_document),
            handle,
            ensure_ascii=False,
            indent=2,
        )

    repeated_document = {
        "model": MODEL_NAME,
        "experimental_design": {
            "test_set": "fixed_external_held_out_file",
            "train_validation_split_unit": "sentence",
            "train_validation_stratification": (
                "iterative_multilabel_sentence_presence"
            ),
            "split_seed": (
                train_validation_split["split_seed_used"]
            ),
            "validation_fraction": VALIDATION_FRACTION,
            "number_of_runs": len(RUN_SEEDS),
            "run_seeds": RUN_SEEDS,
            "same_split_for_every_run": True,
            "threshold_calibration": (
                "per_feature_binary_f1_on_validation_only"
            ),
            "test_bootstrap_unit": "sentence",
            "test_bootstrap_samples": TEST_BOOTSTRAP_SAMPLES,
            "test_bootstrap_confidence": (
                TEST_BOOTSTRAP_CONFIDENCE
            ),
        },
        "data_sources": {
            "train_jsonl": str(TRAIN_JSONL_PATH),
            "test_jsonl": str(TEST_JSONL_PATH),
            "train_conllu": str(TRAIN_CONLLU_PATH),
            "test_conllu": str(TEST_CONLLU_PATH),
        },
        "split_sizes": {
            "train_sentences": len(train_indices),
            "validation_sentences": len(validation_indices),
            "test_sentences": len(test_indices),
        },
        "prediction_labels": prediction_labels,
        "full_labels": full_labels,
        "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
        "disabled_features": sorted(
            feature
            for feature, enabled in FEATURE_SWITCHES.items()
            if not enabled
        ),
        "unseen_test_independent_labels": (
            unseen_test_independent_labels
        ),
        "validation_repeated_run_summary": validation_summary,
        "test_repeated_run_summary": test_summary,
        "test_per_feature_complete_repeated_run_summary": (
            per_feature_complete_summary
        ),
        "test_per_feature_independent_repeated_run_summary": (
            per_feature_independent_summary
        ),
        "test_bootstrap": bootstrap_results,
        "runs": [
            {
                "run_number": run_document["run_number"],
                "seed": run_document["seed"],
                "is_reference_model": (
                    run_document["is_reference_model"]
                ),
                "result_file": str(
                    RUNS_DIR
                    / f"seed_{run_document['seed']}.json"
                ),
                "validation_metrics_optimised": (
                    run_document[
                        "validation_metrics_optimised"
                    ]
                ),
                "test_metrics_at_0_5": (
                    run_document["test_metrics_at_0_5"]
                ),
                "test_metrics_optimised": (
                    run_document["test_metrics_optimised"]
                ),
            }
            for run_document in all_run_documents
        ],
    }

    with REPEATED_RESULTS_PATH.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            make_json_serialisable(repeated_document),
            handle,
            ensure_ascii=False,
            indent=2,
        )

    print()
    print("=" * 100)
    print("OUTPUTS")
    print("=" * 100)
    print(
        "Fixed sentence split:\n"
        f"  {SPLIT_PATH.resolve()}"
    )
    print(
        "Semantic feature metadata:\n"
        f"  {LABELS_PATH.resolve()}"
    )
    print(
        "Per-run results:\n"
        f"  {RUNS_DIR.resolve()}"
    )
    print(
        "Repeated-run summary:\n"
        f"  {REPEATED_RESULTS_PATH.resolve()}"
    )
    print(
        "Test bootstrap confidence intervals:\n"
        f"  {BOOTSTRAP_RESULTS_PATH.resolve()}"
    )

    if SAVE_REFERENCE_MODEL:
        print(
            "Neutral reference model (first predeclared seed; not "
            "best-test selected):\n"
            f"  {REFERENCE_MODEL_DIR.resolve()}"
        )


if __name__ == "__main__":
    main()


/root/.pyenv/versions/3.10.16/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA
PyTorch version: 2.13.0+cu130
CUDA available: True
CUDA runtime: 13.0
GPU: NVIDIA GeForce RTX 5070 Ti
VRAM: 15.92 GiB

LOADING DATA
FEATURE SWITCHES
Disabled features: Rigidity, Shape, State, State_of_Matter
All unlisted features are enabled.

Training CoNLL-U sentences: 12,544
Test CoNLL-U sentences: 2,077
Training sentences represented in tagged JSONL: 11,511
Test sentences represented in tagged JSONL: 1,815
Training tagged NOUN/PROPN records: 47,371
Test tagged NOUN/PROPN records: 6,198

Eligible training-file sentences with at least one non-empty feature target: 11,198
Eligible test-file sentences with at least one non-empty feature target: 1,759

Sentence-level multilabel-stratified train/validation split:
  Eligible training sentences : 11,198
  Target validation sentences : 2,240
  Actual validation sentences : 2,313
  Training sentences retained : 8,885
  Split seed used             : 42
  Stratification labels       : 37
  Coverage-rejected attempts  : 0
  Mean relative p

REPEATED FINE-TUNING
Runs: 30
Seeds: [11, 23, 37, 53, 71, 89, 107, 127, 149, 173, 197, 223, 251, 281, 313, 347, 383, 421, 461, 503, 547, 593, 641, 691, 743, 797, 853, 911, 971, 1031]
Train/validation split is fixed across all runs; only the fine-tuning seed changes.


RUN 1/30 | SEED 11


Loading weights: 100%|███████████████████████████████████████████████████████████████| 389/389 [00:01<00:00, 231.37it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision Micro,Recall Micro,F1 Micro,Precision Macro,Recall Macro,F1 Macro,Direct Precision Micro,Direct Recall Micro,Direct F1 Micro,Direct Precision Macro,Direct Recall Macro,Direct F1 Macro
1,0.093731,0.040096,0.932767,0.880572,0.905918,0.849755,0.763932,0.793560,0.871493,0.819267,0.844574,0.780539,0.670923,0.706957
2,0.063088,0.032259,0.937613,0.913538,0.925419,0.898717,0.831664,0.858990,0.883632,0.863370,0.873383,0.860109,0.760744,0.795172
3,0.040718,0.031801,0.940107,0.923994,0.931981,0.901794,0.889790,0.894861,0.891389,0.873626,0.882418,0.888841,0.827082,0.843016
4,0.025718,0.030608,0.941397,0.931253,0.936297,0.910065,0.883785,0.896098,0.890190,0.885495,0.887836,0.884617,0.830720,0.850336
5,0.022054,0.030745,0.944504,0.934386,0.939418,0.914861,0.897852,0.906111,0.895218,0.888791,0.891993,0.893280,0.841413,0.858392


...


TEST @ 0.5 (DIAGNOSTIC)
direct_f1_macro: 0.7819406501163559
direct_f1_micro: 0.865407520843968
direct_precision_macro: 0.8177276620943066
direct_precision_micro: 0.8663411310470134
direct_recall_macro: 0.7671059876167511
direct_recall_micro: 0.8644759206798867
f1_macro: 0.8551713742253025
f1_micro: 0.9187225743539736
precision_macro: 0.87135120857051
precision_micro: 0.9213279874828868
recall_macro: 0.844117372216991
recall_micro: 0.9161318553092183

TEST WITH FROZEN VALIDATION-CALIBRATED THRESHOLDS
direct_f1_macro: 0.8001702939389782
direct_f1_micro: 0.8635823093040376
direct_precision_macro: 0.8264341525552478
direct_precision_micro: 0.8631424043468418
direct_recall_macro: 0.7901697102994928
direct_recall_micro: 0.8640226628895185
f1_macro: 0.8534493249650104
f1_micro: 0.9174692185785688
precision_macro: 0.8686021466148441
precision_micro: 0.9201838900572211
recall_macro: 0.8449182477348212
recall_micro: 0.9147705173084403

RUN 30/30 | SEED 1031


Loading weights: 100%|██████████████████████████████████████████████████████████████| 389/389 [00:00<00:00, 1020.39it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision Micro,Recall Micro,F1 Micro,Precision Macro,Recall Macro,F1 Macro,Direct Precision Micro,Direct Recall Micro,Direct F1 Micro,Direct Precision Macro,Direct Recall Macro,Direct F1 Macro
1,0.092491,0.040450,0.920814,0.878621,0.899223,0.864304,0.769373,0.804594,0.874851,0.809158,0.840723,0.800220,0.675622,0.717700
2,0.061690,0.033711,0.938657,0.901739,0.919828,0.903772,0.835483,0.862310,0.889955,0.849011,0.869001,0.867308,0.759859,0.794343
3,0.044931,0.030233,0.944829,0.916672,0.930538,0.912096,0.872075,0.890229,0.896809,0.872894,0.884690,0.901826,0.814871,0.841738
4,0.028118,0.030444,0.944291,0.926808,0.935468,0.908956,0.891279,0.899513,0.895101,0.882051,0.888528,0.898833,0.833435,0.849575
5,0.021407,0.031008,0.942061,0.933235,0.937627,0.907316,0.901427,0.904179,0.893456,0.890183,0.891817,0.882966,0.843205,0.853121



VALIDATION-OPTIMISED THRESHOLDS
Feature                              Threshold   Support     F1@0.5     F1 opt     Change
-----------------------------------------------------------------------------------------
Abstract                                  0.64      1405     0.8939     0.8963    +0.0024
Animal                                    0.29       184     0.9448     0.9505    +0.0058
Animate                                   0.38        13     0.8696     0.8800    +0.0104
Artefact                                  0.55       503     0.9178     0.9215    +0.0037
Body_Part                                 0.14        74     0.9262     0.9351    +0.0089
Comestible                                0.50       157     0.9557     0.9557    +0.0000
Communication                             0.58       534     0.8147     0.8207    +0.0060
Currency                                  0.74        45     0.9783     1.0000    +0.0217
Duration                                  0.71       340     0.8488


TEST @ 0.5 (DIAGNOSTIC)
direct_f1_macro: 0.7818946881242113
direct_f1_micro: 0.8616030881017257
direct_precision_macro: 0.8052136441312748
direct_precision_micro: 0.8632692526447503
direct_recall_macro: 0.7716057971203792
direct_recall_micro: 0.859943342776204
f1_macro: 0.8604670067296107
f1_micro: 0.917260220509318
precision_macro: 0.8735361670580911
precision_micro: 0.9204033679263756
recall_macro: 0.8509011968739413
recall_micro: 0.9141384675223648

TEST WITH FROZEN VALIDATION-CALIBRATED THRESHOLDS
direct_f1_macro: 0.7862741408496535
direct_f1_micro: 0.8601858146385679
direct_precision_macro: 0.7929065044531706
direct_precision_micro: 0.8600883652430045
direct_recall_macro: 0.7866523849937622
direct_recall_micro: 0.8602832861189802
f1_macro: 0.8603603170999846
f1_micro: 0.9167294374924068
precision_macro: 0.8696026526805029
precision_micro: 0.916306406955846
recall_macro: 0.8553639242906623
recall_micro: 0.9171528588098017

REPEATED-RUN TEST SUMMARY
Metric                          

<h1>Hyperparameter tuning</h1>

In [6]:
#!/usr/bin/env python3
"""
Fine-tune RoBERTa for hierarchical multi-label semantic feature tagging of
NOUN/PROPN tokens.

Compatible with Python 3.9.21.

Key design changes
------------------
1. The model is trained only on independent semantic features. For each token,
   any feature that can be inherited from a more specific present feature is
   removed from the neural target. This is a per-token reduction rather than a
   global "terminal leaf only" rule, so features such as Human, Time, Shape,
   Communication, etc. can still be predicted directly when they are the most
   specific available feature for that token.

2. All inherited features are reconstructed deterministically from
   FEATURE_PARENTS for validation, testing, reporting, and later inference.

3. A separate sigmoid threshold is optimised for every neural output using the
   validation split only. The selected thresholds are frozen before the test
   split is evaluated and are saved to prediction_thresholds.json.

4. Train/validation splitting is performed at SENTENCE level with iterative
   multilabel stratification over sentence-level independent-feature presence.
   The split is fixed across repeated fine-tuning runs.

5. A single fixed training seed is used for a validation-only grid search over
   learning rate and effective batch size. This selects a candidate setting
   for the later repeated-seed experiment, not a final performance estimate.

6. The official test corpus is not loaded or evaluated during tuning.

Tagged data:
    Training/validation:
        ewt_train_wordnet_wsd_features_wordnet_fixed.jsonl

Sentence/token sources:
    Training/validation:
        UD_English-EWT-master/
        UD_English-EWT-master/
        en_ewt-ud-train.conllu

The validation split is created only from the training corpus. The official
test corpus remains untouched until final evaluation.

Loss:
    BCEWithLogitsLoss over INDEPENDENT feature labels only.

Headline metrics:
    Computed after deterministic hierarchy expansion, so they describe the
    complete semantic feature sets rather than only the neural outputs.

Requirements:
    python -m pip install torch transformers datasets accelerate conllu scikit-learn iterative-stratification

CUDA is required.
"""

from __future__ import annotations

import csv
import gc
import json
import shutil
from itertools import product
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Set, Tuple

import numpy as np
import torch
import torch.nn as nn

from conllu import parse_incr
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import precision_recall_fscore_support
from torch.utils.data import Dataset

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)


# =============================================================================
# Configuration
# =============================================================================

TRAIN_JSONL_PATH = Path(
    "../data/ewt_train_wordnet_wsd_features_wordnet_fixed.jsonl"
    #"ewt_train_wordnet_wsd_features_fully_tagged_llm_reviewed.jsonl"
)

TRAIN_CONLLU_PATH = Path(
    "../data/UD_English-EWT-master/"
    "UD_English-EWT-master/"
    "en_ewt-ud-train.conllu"
)

# MODEL_NAME = "FacebookAI/roberta-base"
MODEL_NAME = "FacebookAI/roberta-large"

OUTPUT_DIR = Path("../models/roberta_semantic_features_grid_search")

# Validation is sampled only from eligible training sentences.
VALIDATION_FRACTION = 0.20

# Fixed seed for the sentence-level train/validation split.
SPLIT_SEED = 42

# All grid candidates use the same random seed and split.
TUNING_SEED = 11

# 3 learning rates x 4 batch configurations = 12 training runs.
LEARNING_RATES = [2.5e-5, 3e-5, 3.5e-5, 4e-5]
PER_DEVICE_TRAIN_BATCH_SIZES = [4, 8]
EFFECTIVE_BATCH_SIZES = [16, 32]
# Gradient accumulation is effective_batch_size // per_device_train_batch_size.
# For a single GPU, the four configurations are (4, 4), (8, 2),
# (4, 8), and (8, 4), where pairs mean (device batch, accumulation).

# The multilabel stratifier is retried with deterministic derived seeds until
# every independent feature still has at least this many positive token targets
# in the TRAIN subset. This prevents a rare direct target from disappearing.
MIN_TRAIN_POSITIVES_PER_FEATURE = 1
MAX_STRATIFICATION_ATTEMPTS = 200

MAX_LENGTH = 256

WEIGHT_DECAY = 0.01

EVAL_BATCH_SIZE = 16

NUM_EPOCHS = 5

# Threshold used during epoch-by-epoch Trainer evaluation. Final thresholds are
# optimised per independent feature on the validation set after training.
DEFAULT_PREDICTION_THRESHOLD = 0.5

# Validation-only threshold search grid.
THRESHOLD_MIN = 0.05
THRESHOLD_MAX = 0.95
THRESHOLD_STEP = 0.01

# Only these POS categories receive feature supervision.
TARGET_POS = {"NOUN", "PROPN"}

# A NOUN/PROPN with no features is considered a failed source annotation rather
# than a valid all-negative target.
EXCLUDE_EMPTY_FEATURE_RECORDS = True

# Optional per-feature switches. Any feature not listed here is enabled.
# Set a feature to False to remove it from:
#   - sentence-level stratification,
#   - neural training targets,
#   - threshold calibration,
#   - hierarchy-expanded gold/predicted feature sets,
#   - per-feature and aggregate evaluation, and
#   - the sentence-cluster bootstrap.
#
# Disabled intermediate hierarchy nodes are skipped but their enabled ancestors
# are still inherited. For example, if Human were disabled, Feminine would still
# inherit Animate/Natural/Concrete/Internal_Control via the hierarchy.
FEATURE_SWITCHES: Dict[str, bool] = {
    "State": False,
    "State_of_Matter": False,
    "Shape": False,
    "Rigidity": False,
}

EVAL_STRATEGY = "epoch"

SPLIT_PATH = OUTPUT_DIR / "sentence_split.json"
LABELS_PATH = OUTPUT_DIR / "semantic_features.json"
GRID_RESULTS_PATH = OUTPUT_DIR / "hyperparameter_grid_results.json"
GRID_CSV_PATH = OUTPUT_DIR / "hyperparameter_grid_results.csv"


# =============================================================================
# Feature hierarchy
# =============================================================================

def is_feature_enabled(feature: str) -> bool:
    """Return whether *feature* participates in this experiment."""

    return bool(FEATURE_SWITCHES.get(feature, True))


def validate_feature_switches(
    train_tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> None:
    """Fail early on misspelled feature-switch names and report disabled labels."""

    observed: Set[str] = set()

    for source in (train_tagged_records,):
        for sentence_records in source.values():
            for record in sentence_records.values():
                raw = record.get("features", [])
                if isinstance(raw, list):
                    observed.update(
                        str(feature).strip()
                        for feature in raw
                        if str(feature).strip()
                    )

    known = set(observed) | set(FEATURE_PARENTS)
    for parents in FEATURE_PARENTS.values():
        known.update(parents)

    unknown = sorted(set(FEATURE_SWITCHES) - known)
    if unknown:
        raise ValueError(
            "FEATURE_SWITCHES contains unknown feature name(s): "
            + ", ".join(unknown)
        )

    disabled = sorted(
        feature for feature, enabled in FEATURE_SWITCHES.items() if not enabled
    )

    print("=" * 80)
    print("FEATURE SWITCHES")
    print("=" * 80)
    if disabled:
        print("Disabled features: " + ", ".join(disabled))
    else:
        print("Disabled features: none")
    print("All unlisted features are enabled.")
    print()


# child -> direct parent(s)
FEATURE_PARENTS: Dict[str, Set[str]] = {
    "Organisation": {
        "Internal_Control",
        "Abstract",
    },
    "Natural_Force": {
        "Internal_Control",
    },
    "Animate": {
        "Internal_Control",
        "Natural",
    },
    "Human": {
        "Animate",
    },
    "Feminine": {
        "Human",
    },
    "Masculine": {
        "Human",
    },
    "Unspecified_Gender_or_Neuter": {
        "Human",
    },
    "Animal": {
        "Animate",
    },
    "Plant": {
        "Animate",
    },
    "Body_Part": {
        "Animate",
    },
    "Machine": {
        "Internal_Control",
        "Artefact",
    },
    "Vehicle": {
        "Machine",
    },
    "Natural": {
        "Concrete",
    },
    "Artefact": {
        "Concrete",
    },
    "Comestible": {
        "Concrete",
    },
    "Substance": {
        "Concrete",
    },
    "State_of_Matter": {
        "Concrete",
    },
    "Inanimate": {
        "Concrete",
    },
    "Garment": {
        "Artefact",
    },
    "Solid": {
        "State_of_Matter",
    },
    "Fluid": {
        "State_of_Matter",
    },
    "Rigidity": {
        "Solid",
    },
    "Shape": {
        "Solid",
    },
    "Rigid": {
        "Rigidity",
    },
    "Non_Rigid": {
        "Rigidity",
    },
    "Elongated": {
        "Shape",
    },
    "Pointy": {
        "Shape",
    },
    "Point_in_Time": {
        "Time",
    },
    "Duration": {
        "Time",
    },
    "Time": {
        "Abstract",
    },
    "Language": {
        "Abstract",
    },
    "Sound": {
        "Abstract",
    },
    "Communication": {
        "Abstract",
    },
    "Eventive": {
        "Abstract",
    },
    "Statement": {
        "Communication",
    },
    "Question": {
        "Communication",
    },
}


def validate_feature_hierarchy() -> None:
    """Fail early if FEATURE_PARENTS contains a cycle or invalid name."""

    all_features: Set[str] = set(FEATURE_PARENTS)

    for parents in FEATURE_PARENTS.values():
        all_features.update(parents)

    for feature in all_features:
        if not isinstance(feature, str) or not feature.strip():
            raise ValueError(
                "FEATURE_PARENTS contains an empty or non-string feature name."
            )

    visiting: Set[str] = set()
    visited: Set[str] = set()

    def visit(feature: str) -> None:
        if feature in visited:
            return

        if feature in visiting:
            raise ValueError(
                f"Cycle detected in FEATURE_PARENTS involving {feature!r}."
            )

        visiting.add(feature)

        for parent in FEATURE_PARENTS.get(feature, set()):
            visit(parent)

        visiting.remove(feature)
        visited.add(feature)

    for feature in sorted(all_features):
        visit(feature)


@lru_cache(maxsize=None)
def get_feature_ancestors(feature: str) -> frozenset:
    """Return every direct and indirect parent of *feature*."""

    ancestors: Set[str] = set()

    for parent in FEATURE_PARENTS.get(feature, set()):
        if is_feature_enabled(parent):
            ancestors.add(parent)
        # Traverse even through a disabled intermediate node so enabled
        # grandparents remain inherited.
        ancestors.update(get_feature_ancestors(parent))

    return frozenset(ancestors)


def expand_feature_hierarchy(features: Iterable[str]) -> Set[str]:
    """Add every deterministic ancestor required by FEATURE_PARENTS."""

    expanded: Set[str] = {
        feature for feature in features if is_feature_enabled(feature)
    }

    for feature in list(expanded):
        expanded.update(get_feature_ancestors(feature))

    return expanded


def reduce_to_independent_features(features: Iterable[str]) -> Set[str]:
    """
    Reduce a complete feature set to the features that must be predicted.

    A feature is redundant when it is an ancestor of another feature present
    for the same token. Such an ancestor can be reconstructed deterministically.

    Examples:
        {Human, Animate, Natural, Concrete, Internal_Control}
            -> {Human}

        {Feminine, Human, Animate, Natural, Concrete, Internal_Control}
            -> {Feminine}

        {Statement, Communication, Abstract, Time}
            -> {Statement, Time}

    This is deliberately PER TOKEN. It is not the same as training only labels
    that are global terminal leaves in the ontology. A generic Human can still
    be a direct neural target if no more specific Human child is present.
    """

    feature_set = set(features)

    inherited_from_other_features: Set[str] = set()

    for feature in feature_set:
        inherited_from_other_features.update(
            get_feature_ancestors(feature)
        )

    return feature_set - inherited_from_other_features


def missing_hierarchy_features(features: Iterable[str]) -> Set[str]:
    """Return ancestors required by the hierarchy but absent from *features*."""

    feature_set = set(features)
    return expand_feature_hierarchy(feature_set) - feature_set


# =============================================================================
# CUDA
# =============================================================================

def check_cuda() -> None:
    """Require CUDA rather than silently falling back to the CPU."""

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. This script is configured to require "
            "a CUDA-capable GPU."
        )

    print("=" * 80)
    print("CUDA")
    print("=" * 80)
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"CUDA runtime: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    props = torch.cuda.get_device_properties(0)

    print(
        "VRAM: "
        f"{props.total_memory / (1024 ** 3):.2f} GiB"
    )
    print()


# =============================================================================
# Load CoNLL-U
# =============================================================================

def load_conllu(path: Path) -> List[Any]:
    """Load all CoNLL-U sentences."""

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find CoNLL-U file:\n{path.resolve()}"
        )

    sentences: List[Any] = []

    with path.open("r", encoding="utf-8") as handle:
        for sentence in parse_incr(handle):
            sentences.append(sentence)

    return sentences


def get_sentence_words(sentence: Any) -> Tuple[List[str], List[int]]:
    """
    Return ordinary integer-ID tokens and their original CoNLL-U token IDs.

    Multiword-token placeholder rows and empty nodes are excluded.
    """

    words: List[str] = []
    token_ids: List[int] = []

    for token in sentence:
        token_id = token.get("id")

        if not isinstance(token_id, int):
            continue

        form = token.get("form")

        if form is None:
            continue

        words.append(str(form))
        token_ids.append(token_id)

    return words, token_ids


# =============================================================================
# Load tagged JSONL
# =============================================================================

def load_tagged_records(
    path: Path,
) -> Dict[int, Dict[int, Dict[str, Any]]]:
    """
    Return:

        sentence_index
            -> token_id
                -> JSONL record

    Only NOUN and PROPN records are retained.
    """

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find tagged JSONL:\n{path.resolve()}"
        )

    records: Dict[int, Dict[int, Dict[str, Any]]] = defaultdict(dict)

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                print(
                    "Warning: invalid JSON on line "
                    f"{line_number}: {exc}"
                )
                continue

            if record.get("upos") not in TARGET_POS:
                continue

            sentence_index = record.get("sentence_index")
            token_id = record.get("token_id")

            if not isinstance(sentence_index, int):
                continue

            if not isinstance(token_id, int):
                continue

            records[sentence_index][token_id] = record

    return dict(records)


# =============================================================================
# Labels
# =============================================================================

def clean_features(record: Dict[str, Any]) -> List[str]:
    """Return unique, non-empty COMPLETE feature names from a JSONL record."""

    raw = record.get("features", [])

    if not isinstance(raw, list):
        return []

    return sorted(
        {
            str(feature).strip()
            for feature in raw
            if (
                str(feature).strip()
                and is_feature_enabled(str(feature).strip())
            )
        }
    )


def get_independent_record_features(record: Dict[str, Any]) -> List[str]:
    """Return the per-token minimal feature set that the model must predict."""

    return sorted(
        reduce_to_independent_features(
            clean_features(record)
        )
    )


def build_full_label_vocabulary(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    training_sentence_indices: Sequence[int],
) -> List[str]:
    """
    Build the COMPLETE evaluation vocabulary from training data only.

    Deterministic ancestors are included even if a training record omitted one,
    although hierarchy closure is separately verified and should normally make
    this addition unnecessary.
    """

    labels: Set[str] = set()

    for sentence_index in training_sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            features = clean_features(record)
            labels.update(features)
            labels.update(expand_feature_hierarchy(features))

    if not labels:
        raise RuntimeError(
            "No semantic features were found in the training split."
        )

    return sorted(labels)


def build_prediction_label_vocabulary(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    training_sentence_indices: Sequence[int],
) -> List[str]:
    """Build the neural-output vocabulary from independent training targets."""

    labels: Set[str] = set()

    for sentence_index in training_sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            labels.update(
                get_independent_record_features(record)
            )

    if not labels:
        raise RuntimeError(
            "No independent semantic features were found in the training split."
        )

    return sorted(labels)


# =============================================================================
# Sentence selection and splitting
# =============================================================================

def get_eligible_sentence_indices(
    sentences: Sequence[Any],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> List[int]:
    """Eligible sentences contain at least one non-empty NOUN/PROPN target."""

    eligible: List[int] = []

    for sentence_index in range(len(sentences)):
        token_records = tagged_records.get(sentence_index, {})

        valid_records = [
            record
            for record in token_records.values()
            if clean_features(record)
        ]

        if valid_records:
            eligible.append(sentence_index)

    return eligible


def get_sentence_independent_feature_counts(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_index: int,
) -> Counter:
    """Count independent positive token targets in one source sentence."""

    counts: Counter = Counter()

    for record in tagged_records.get(sentence_index, {}).values():
        if not clean_features(record):
            continue

        for feature in get_independent_record_features(record):
            counts[feature] += 1

    return counts


def build_sentence_multilabel_matrix(
    eligible_indices: Sequence[int],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> Tuple[np.ndarray, List[str], Dict[int, Counter]]:
    """
    Build one multilabel stratification row per SENTENCE.

    y_sentence[sentence, feature] is 1 when at least one eligible NOUN/PROPN
    token in that sentence has the independent feature. Token counts are kept
    separately for the post-split training-coverage constraint.
    """

    sentence_feature_counts: Dict[int, Counter] = {
        sentence_index: get_sentence_independent_feature_counts(
            tagged_records,
            sentence_index,
        )
        for sentence_index in eligible_indices
    }

    stratification_labels = sorted(
        {
            feature
            for counts in sentence_feature_counts.values()
            for feature, count in counts.items()
            if count > 0
        }
    )

    if not stratification_labels:
        raise RuntimeError(
            "No independent semantic features were found in eligible "
            "training sentences."
        )

    label2column = {
        label: column
        for column, label in enumerate(stratification_labels)
    }

    matrix = np.zeros(
        (len(eligible_indices), len(stratification_labels)),
        dtype=np.int8,
    )

    for row, sentence_index in enumerate(eligible_indices):
        for feature in sentence_feature_counts[sentence_index]:
            matrix[row, label2column[feature]] = 1

    return matrix, stratification_labels, sentence_feature_counts


def _sentence_prevalence_error(
    y_all: np.ndarray,
    validation_rows: np.ndarray,
    validation_fraction: float,
) -> float:
    """
    Score how closely validation sentence-level label prevalences match target.

    Lower is better. The score is used only to choose among deterministic
    iterative-stratification attempts that satisfy the training-coverage rule.
    """

    total_support = y_all.sum(axis=0).astype(np.float64)
    validation_support = y_all[validation_rows].sum(axis=0).astype(np.float64)

    target_support = total_support * float(validation_fraction)

    # Scale by at least one expected example so rare and common labels both
    # contribute without allowing extremely common labels to dominate.
    scale = np.maximum(target_support, 1.0)

    return float(
        np.mean(
            np.abs(validation_support - target_support) / scale
        )
    )


def make_train_validation_split(
    eligible_indices: Sequence[int],
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
) -> Dict[str, Any]:
    """
    Create a SENTENCE-LEVEL multilabel-stratified split from TRAINING only.

    Each sentence receives a binary vector indicating which independent feature
    labels occur anywhere among its eligible NOUN/PROPN tokens. Iterative
    multilabel stratification then assigns whole sentences to train/validation.

    The split is retried deterministically when necessary so that every
    independent feature retains at least MIN_TRAIN_POSITIVES_PER_FEATURE
    positive TOKEN targets in the training subset.

    The official test corpus is deliberately not accepted by this function, so
    test labels cannot influence the split.
    """

    if not 0.0 < VALIDATION_FRACTION < 1.0:
        raise ValueError(
            "VALIDATION_FRACTION must be strictly between 0 and 1."
        )

    if MIN_TRAIN_POSITIVES_PER_FEATURE < 1:
        raise ValueError(
            "MIN_TRAIN_POSITIVES_PER_FEATURE must be at least 1."
        )

    if MAX_STRATIFICATION_ATTEMPTS < 1:
        raise ValueError(
            "MAX_STRATIFICATION_ATTEMPTS must be at least 1."
        )

    eligible_indices = list(eligible_indices)

    if len(eligible_indices) < 2:
        raise RuntimeError(
            "Need at least two eligible training sentences to create a "
            "train/validation split."
        )

    (
        sentence_matrix,
        stratification_labels,
        sentence_feature_counts,
    ) = build_sentence_multilabel_matrix(
        eligible_indices,
        tagged_records,
    )

    all_token_counts: Counter = Counter()

    for counts in sentence_feature_counts.values():
        all_token_counts.update(counts)

    x_dummy = np.zeros((len(eligible_indices), 1), dtype=np.int8)

    best_candidate: Optional[Dict[str, Any]] = None
    rejected_for_coverage = 0

    for attempt in range(MAX_STRATIFICATION_ATTEMPTS):
        attempt_seed = SPLIT_SEED + attempt

        splitter = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=VALIDATION_FRACTION,
            random_state=attempt_seed,
        )

        train_rows, validation_rows = next(
            splitter.split(x_dummy, sentence_matrix)
        )

        train_indices = [
            eligible_indices[int(row)]
            for row in train_rows
        ]
        validation_indices = [
            eligible_indices[int(row)]
            for row in validation_rows
        ]

        train_token_counts: Counter = Counter()

        for sentence_index in train_indices:
            train_token_counts.update(
                sentence_feature_counts[sentence_index]
            )

        missing_or_undercovered = sorted(
            feature
            for feature, total_count in all_token_counts.items()
            if (
                total_count > 0
                and train_token_counts[feature]
                < MIN_TRAIN_POSITIVES_PER_FEATURE
            )
        )

        if missing_or_undercovered:
            rejected_for_coverage += 1
            continue

        prevalence_error = _sentence_prevalence_error(
            sentence_matrix,
            validation_rows,
            VALIDATION_FRACTION,
        )

        candidate = {
            "train": train_indices,
            "validation": validation_indices,
            "split_seed_used": attempt_seed,
            "attempt": attempt + 1,
            "prevalence_error": prevalence_error,
            "stratification_labels": stratification_labels,
        }

        if (
            best_candidate is None
            or prevalence_error < best_candidate["prevalence_error"]
        ):
            best_candidate = candidate

        # Iterative stratification is already designed to balance multilabel
        # prevalence. Once a valid candidate is found, further attempts are not
        # needed unless coverage forced earlier candidates to be rejected.
        if rejected_for_coverage == 0:
            break

    if best_candidate is None:
        raise RuntimeError(
            "Could not create a sentence-level multilabel-stratified "
            "train/validation split that preserved the required training "
            f"support after {MAX_STRATIFICATION_ATTEMPTS} attempts."
        )

    train_indices = best_candidate["train"]
    validation_indices = best_candidate["validation"]

    train_set = set(train_indices)
    validation_set = set(validation_indices)

    if train_set & validation_set:
        raise RuntimeError(
            "Internal split error: at least one sentence occurs in both "
            "training and validation."
        )

    if train_set | validation_set != set(eligible_indices):
        raise RuntimeError(
            "Internal split error: sentence-level split does not exactly cover "
            "the eligible training sentences."
        )

    print()
    print("Sentence-level multilabel-stratified train/validation split:")
    print(
        f"  Eligible training sentences : {len(eligible_indices):,}"
    )
    print(
        f"  Target validation sentences : "
        f"{round(len(eligible_indices) * VALIDATION_FRACTION):,}"
    )
    print(
        f"  Actual validation sentences : {len(validation_indices):,}"
    )
    print(
        f"  Training sentences retained : {len(train_indices):,}"
    )
    print(
        f"  Split seed used             : "
        f"{best_candidate['split_seed_used']}"
    )
    print(
        f"  Stratification labels       : "
        f"{len(stratification_labels):,}"
    )
    print(
        f"  Coverage-rejected attempts  : {rejected_for_coverage:,}"
    )
    print(
        f"  Mean relative prevalence error: "
        f"{best_candidate['prevalence_error']:.6f}"
    )

    # Show sentence-level prevalence agreement, which is the quantity actually
    # controlled by the stratifier.
    validation_fraction_actual = (
        len(validation_indices) / len(eligible_indices)
    )

    label2column = {
        label: column
        for column, label in enumerate(stratification_labels)
    }

    row_by_sentence = {
        sentence_index: row
        for row, sentence_index in enumerate(eligible_indices)
    }

    train_rows_lookup = [
        row_by_sentence[index]
        for index in train_indices
    ]
    validation_rows_lookup = [
        row_by_sentence[index]
        for index in validation_indices
    ]

    print()
    print(
        f"{'Feature':35s} "
        f"{'All sent.':>10s} "
        f"{'Train':>10s} "
        f"{'Valid':>10s} "
        f"{'Valid %':>10s}"
    )
    print("-" * 80)

    for feature in stratification_labels:
        column = label2column[feature]
        all_support = int(sentence_matrix[:, column].sum())
        train_support = int(
            sentence_matrix[train_rows_lookup, column].sum()
        )
        validation_support = int(
            sentence_matrix[validation_rows_lookup, column].sum()
        )

        print(
            f"{feature:35s} "
            f"{all_support:10d} "
            f"{train_support:10d} "
            f"{validation_support:10d} "
            f"{validation_support / max(all_support, 1):10.3f}"
        )

    print(
        f"\nActual validation sentence fraction: "
        f"{validation_fraction_actual:.4f}"
    )
    print()

    return best_candidate


# =============================================================================
# Dataset
# =============================================================================

class SemanticFeatureDataset(Dataset):
    """
    RoBERTa multi-label token-classification dataset.

    IMPORTANT:
        labels contain only INDEPENDENT neural targets.

    Labels have shape:
        [sequence_length, number_of_prediction_features]

    For supervised tokens:
        0 = independent feature absent
        1 = independent feature present

    For ignored tokens:
        -100 across all labels

    Only the first RoBERTa subword corresponding to each source token receives
    supervision.
    """

    def __init__(
        self,
        sentences: Sequence[Any],
        sentence_indices: Sequence[int],
        tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
        tokenizer: Any,
        label2id: Dict[str, int],
        max_length: int,
    ) -> None:
        self.sentences = sentences
        self.sentence_indices = list(sentence_indices)
        self.tagged_records = tagged_records
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.num_labels = len(label2id)
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.sentence_indices)

    def __getitem__(
        self,
        dataset_index: int,
    ) -> Dict[str, Any]:
        sentence_index = self.sentence_indices[dataset_index]
        sentence = self.sentences[sentence_index]

        words, token_ids = get_sentence_words(sentence)

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=True,
        )

        word_ids = encoding.word_ids()
        labels: List[List[float]] = []
        previous_word_id: Optional[int] = None
        sentence_records = self.tagged_records.get(sentence_index, {})

        for word_id in word_ids:
            if word_id is None:
                labels.append([-100.0] * self.num_labels)
                previous_word_id = word_id
                continue

            # Ignore all but the first subword.
            if word_id == previous_word_id:
                labels.append([-100.0] * self.num_labels)
                continue

            previous_word_id = word_id

            if not 0 <= word_id < len(token_ids):
                labels.append([-100.0] * self.num_labels)
                continue

            conllu_token_id = token_ids[word_id]
            record = sentence_records.get(conllu_token_id)

            if record is None:
                labels.append([-100.0] * self.num_labels)
                continue

            full_features = clean_features(record)

            if not full_features and EXCLUDE_EMPTY_FEATURE_RECORDS:
                labels.append([-100.0] * self.num_labels)
                continue

            independent_features = reduce_to_independent_features(
                full_features
            )

            target = [0.0] * self.num_labels

            for feature in independent_features:
                label_id = self.label2id.get(feature)

                if label_id is not None:
                    target[label_id] = 1.0

            labels.append(target)

        encoding["labels"] = labels
        encoding["sentence_index"] = sentence_index

        return encoding


# =============================================================================
# Data collator
# =============================================================================

class MultiLabelTokenCollator:
    """Dynamically pad inputs and 3-D multi-label token targets."""

    def __init__(
        self,
        tokenizer: Any,
        num_labels: int,
    ) -> None:
        self.tokenizer = tokenizer
        self.num_labels = num_labels

    def __call__(
        self,
        features: List[Dict[str, Any]],
    ) -> Dict[str, torch.Tensor]:
        clean_features_list: List[Dict[str, Any]] = []
        labels_list: List[List[List[float]]] = []

        for item in features:
            item = dict(item)
            item.pop("sentence_index", None)

            labels = item.pop("labels")
            labels_list.append(labels)
            clean_features_list.append(item)

        batch = self.tokenizer.pad(
            clean_features_list,
            padding=True,
            return_tensors="pt",
        )

        batch_size = len(labels_list)
        max_seq_length = batch["input_ids"].shape[1]

        padded_labels = torch.full(
            (
                batch_size,
                max_seq_length,
                self.num_labels,
            ),
            -100.0,
            dtype=torch.float32,
        )

        for batch_index, labels in enumerate(labels_list):
            label_tensor = torch.tensor(
                labels,
                dtype=torch.float32,
            )

            sequence_length = label_tensor.shape[0]

            padded_labels[
                batch_index,
                :sequence_length,
                :,
            ] = label_tensor

        batch["labels"] = padded_labels

        return batch


# =============================================================================
# Model
# =============================================================================

class RobertaMultiLabelModel(nn.Module):
    """RoBERTa token classifier with BCE loss for independent feature labels."""

    def __init__(
        self,
        model_name: str,
        num_labels: int,
        id2label: Dict[int, str],
        label2id: Dict[str, int],
    ) -> None:
        super().__init__()

        self.model = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            id2label=id2label,
            label2id=label2id,
        )

        self.config = self.model.config

        self.loss_function = nn.BCEWithLogitsLoss(
            reduction="mean"
        )

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        **kwargs,
    ):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs,
        )

        logits = outputs.logits
        loss = None

        if labels is not None:
            valid_mask = labels[:, :, 0] != -100.0

            valid_logits = logits[valid_mask]
            valid_labels = labels[valid_mask]

            if valid_logits.numel() == 0:
                loss = logits.sum() * 0.0
            else:
                loss = self.loss_function(
                    valid_logits,
                    valid_labels,
                )

        return {
            "loss": loss,
            "logits": logits,
        }

    def save_pretrained(
        self,
        save_directory,
        **kwargs,
    ):
        return self.model.save_pretrained(
            save_directory,
            **kwargs,
        )


# =============================================================================
# Hierarchy matrices and array helpers
# =============================================================================

def build_hierarchy_expansion_matrix(
    prediction_labels: Sequence[str],
    full_labels: Sequence[str],
) -> np.ndarray:
    """
    Build a boolean matrix mapping neural outputs to complete feature sets.

    Shape:
        [number_of_prediction_features, number_of_full_features]
    """

    full_label2id = {
        label: index
        for index, label in enumerate(full_labels)
    }

    matrix = np.zeros(
        (len(prediction_labels), len(full_labels)),
        dtype=np.int8,
    )

    for prediction_id, feature in enumerate(prediction_labels):
        expanded = expand_feature_hierarchy({feature})

        for expanded_feature in expanded:
            full_id = full_label2id.get(expanded_feature)

            if full_id is None:
                raise RuntimeError(
                    "Hierarchy expansion produced a feature absent from the "
                    f"full vocabulary: {expanded_feature!r}."
                )

            matrix[prediction_id, full_id] = 1

    return matrix


def expand_binary_feature_matrix(
    independent_matrix: np.ndarray,
    hierarchy_matrix: np.ndarray,
) -> np.ndarray:
    """Expand [tokens, independent] binary values to [tokens, complete]."""

    expanded = independent_matrix.astype(np.int32) @ hierarchy_matrix.astype(
        np.int32
    )

    return (expanded > 0).astype(np.int32)


def extract_valid_token_arrays(
    logits: np.ndarray,
    labels: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return probabilities and true independent labels for supervised tokens."""

    valid_token_mask = labels[:, :, 0] != -100.0

    probabilities = sigmoid_numpy(logits)[
        valid_token_mask
    ]

    true_labels = labels[
        valid_token_mask
    ].astype(np.int32)

    return probabilities, true_labels


# =============================================================================
# Metrics and thresholds
# =============================================================================

def sigmoid_numpy(x: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid."""

    x = np.clip(x, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-x))


def make_threshold_vector(
    prediction_labels: Sequence[str],
    thresholds: Dict[str, float],
) -> np.ndarray:
    """Return thresholds in neural-output column order."""

    return np.asarray(
        [
            float(
                thresholds.get(
                    label,
                    DEFAULT_PREDICTION_THRESHOLD,
                )
            )
            for label in prediction_labels
        ],
        dtype=np.float32,
    )


def threshold_probabilities(
    probabilities: np.ndarray,
    threshold_vector: np.ndarray,
) -> np.ndarray:
    """Apply one threshold per independent feature."""

    return (
        probabilities >= threshold_vector.reshape(1, -1)
    ).astype(np.int32)


def calculate_average_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    prefix: str = "",
) -> Dict[str, float]:
    """Calculate micro and macro precision/recall/F1."""

    if y_true.shape[0] == 0:
        return {
            f"{prefix}precision_micro": 0.0,
            f"{prefix}recall_micro": 0.0,
            f"{prefix}f1_micro": 0.0,
            f"{prefix}precision_macro": 0.0,
            f"{prefix}recall_macro": 0.0,
            f"{prefix}f1_macro": 0.0,
        }

    (
        precision_micro,
        recall_micro,
        f1_micro,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="micro",
        zero_division=0,
    )

    (
        precision_macro,
        recall_macro,
        f1_macro,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    return {
        f"{prefix}precision_micro": float(precision_micro),
        f"{prefix}recall_micro": float(recall_micro),
        f"{prefix}f1_micro": float(f1_micro),
        f"{prefix}precision_macro": float(precision_macro),
        f"{prefix}recall_macro": float(recall_macro),
        f"{prefix}f1_macro": float(f1_macro),
    }


def calculate_hierarchical_metrics_from_probabilities(
    probabilities: np.ndarray,
    y_true_independent: np.ndarray,
    threshold_vector: np.ndarray,
    hierarchy_matrix: np.ndarray,
    y_true_full: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    """
    Report both direct neural-output metrics and complete hierarchy metrics.

    Unprefixed metrics are the COMPLETE hierarchy metrics.  During ordinary
    training/validation, complete truth can be reconstructed from the
    independent targets.  For a genuinely held-out test corpus, ``y_true_full``
    may instead be supplied directly from the source annotations so an unseen
    direct independent label is not silently erased from the gold truth.
    """

    y_pred_independent = threshold_probabilities(
        probabilities,
        threshold_vector,
    )

    if y_true_full is None:
        y_true_full = expand_binary_feature_matrix(
            y_true_independent,
            hierarchy_matrix,
        )

    if y_true_full.shape[0] != y_true_independent.shape[0]:
        raise RuntimeError(
            "Complete and independent truth matrices contain different "
            f"numbers of supervised tokens: {y_true_full.shape[0]} vs "
            f"{y_true_independent.shape[0]}."
        )

    y_pred_full = expand_binary_feature_matrix(
        y_pred_independent,
        hierarchy_matrix,
    )

    metrics = calculate_average_metrics(
        y_true_full,
        y_pred_full,
        prefix="",
    )

    metrics.update(
        calculate_average_metrics(
            y_true_independent,
            y_pred_independent,
            prefix="direct_",
        )
    )

    return metrics


def make_compute_metrics(
    prediction_labels: Sequence[str],
    hierarchy_matrix: np.ndarray,
):
    """Create Trainer metrics using the fixed default 0.5 thresholds."""

    default_thresholds = np.full(
        len(prediction_labels),
        DEFAULT_PREDICTION_THRESHOLD,
        dtype=np.float32,
    )

    def compute_metrics(eval_prediction) -> Dict[str, float]:
        logits, labels = eval_prediction

        probabilities, y_true_independent = extract_valid_token_arrays(
            logits,
            labels,
        )

        return calculate_hierarchical_metrics_from_probabilities(
            probabilities=probabilities,
            y_true_independent=y_true_independent,
            threshold_vector=default_thresholds,
            hierarchy_matrix=hierarchy_matrix,
        )

    return compute_metrics


def build_threshold_grid() -> np.ndarray:
    """Build an inclusive, stable validation threshold search grid."""

    if THRESHOLD_STEP <= 0:
        raise ValueError("THRESHOLD_STEP must be > 0.")

    if not 0.0 < THRESHOLD_MIN < 1.0:
        raise ValueError("THRESHOLD_MIN must be between 0 and 1.")

    if not 0.0 < THRESHOLD_MAX < 1.0:
        raise ValueError("THRESHOLD_MAX must be between 0 and 1.")

    if THRESHOLD_MIN > THRESHOLD_MAX:
        raise ValueError("THRESHOLD_MIN cannot exceed THRESHOLD_MAX.")

    count = int(
        round(
            (THRESHOLD_MAX - THRESHOLD_MIN)
            / THRESHOLD_STEP
        )
    ) + 1

    return np.linspace(
        THRESHOLD_MIN,
        THRESHOLD_MAX,
        count,
        dtype=np.float64,
    )


def binary_metrics_for_column(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> Tuple[float, float, float, int]:
    """Return binary precision, recall, F1 and positive support."""

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    support = int(y_true.sum())

    return (
        float(precision),
        float(recall),
        float(f1),
        support,
    )


def optimise_feature_thresholds(
    probabilities: np.ndarray,
    y_true_independent: np.ndarray,
    prediction_labels: Sequence[str],
) -> Tuple[Dict[str, float], Dict[str, Dict[str, float]]]:
    """
    Optimise one sigmoid threshold per independent feature on VALIDATION ONLY.

    Objective:
        maximise that feature's binary F1.

    Stability rules:
    - If validation has no positives or no negatives for a feature, keep 0.5.
    - If several thresholds tie for best F1, choose the one closest to 0.5.
      This avoids unnecessarily extreme thresholds for flat validation curves.
    """

    grid = build_threshold_grid()

    thresholds: Dict[str, float] = {}
    diagnostics: Dict[str, Dict[str, float]] = {}

    for label_id, feature in enumerate(prediction_labels):
        y_true = y_true_independent[:, label_id]
        probs = probabilities[:, label_id]

        positive_support = int(y_true.sum())
        negative_support = int((1 - y_true).sum())

        default_pred = (
            probs >= DEFAULT_PREDICTION_THRESHOLD
        ).astype(np.int32)

        (
            default_precision,
            default_recall,
            default_f1,
            _,
        ) = binary_metrics_for_column(
            y_true,
            default_pred,
        )

        if positive_support == 0 or negative_support == 0:
            best_threshold = DEFAULT_PREDICTION_THRESHOLD
        else:
            best_f1 = -1.0
            best_candidates: List[float] = []

            for threshold in grid:
                y_pred = (probs >= threshold).astype(np.int32)

                _, _, f1, _ = binary_metrics_for_column(
                    y_true,
                    y_pred,
                )

                if f1 > best_f1 + 1e-12:
                    best_f1 = f1
                    best_candidates = [float(threshold)]
                elif abs(f1 - best_f1) <= 1e-12:
                    best_candidates.append(float(threshold))

            best_threshold = min(
                best_candidates,
                key=lambda value: (
                    abs(value - DEFAULT_PREDICTION_THRESHOLD),
                    value,
                ),
            )

        best_pred = (
            probs >= best_threshold
        ).astype(np.int32)

        (
            best_precision,
            best_recall,
            best_f1,
            _,
        ) = binary_metrics_for_column(
            y_true,
            best_pred,
        )

        thresholds[feature] = float(best_threshold)

        diagnostics[feature] = {
            "threshold": float(best_threshold),
            "support": positive_support,
            "negative_support": negative_support,
            "precision_default": default_precision,
            "recall_default": default_recall,
            "f1_default": default_f1,
            "precision_optimised": best_precision,
            "recall_optimised": best_recall,
            "f1_optimised": best_f1,
            "f1_change": best_f1 - default_f1,
        }

    return thresholds, diagnostics


# =============================================================================
# Dataset diagnostics and validation
# =============================================================================

def count_feature_occurrences(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_indices: Sequence[int],
    independent_only: bool = False,
) -> Counter:
    """Count complete or independent positive labels in a split."""

    counts: Counter = Counter()

    for sentence_index in sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            if independent_only:
                features = get_independent_record_features(record)
            else:
                features = clean_features(record)

            for feature in features:
                counts[feature] += 1

    return counts


def count_empty_records(
    tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    sentence_indices: Sequence[int],
) -> int:
    """Count NOUN/PROPN records having no complete feature tags."""

    count = 0

    for sentence_index in sentence_indices:
        for record in tagged_records.get(sentence_index, {}).values():
            if not clean_features(record):
                count += 1

    return count


def verify_full_label_coverage(
    full_labels: Sequence[str],
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    """Ensure validation contains no complete label unseen in training."""

    training_labels = set(full_labels)

    for split_name in ("validation",):
        tagged_records, sentence_indices = split_sources[split_name]
        observed: Set[str] = set()

        for sentence_index in sentence_indices:
            for record in tagged_records.get(sentence_index, {}).values():
                observed.update(clean_features(record))

        unknown = observed - training_labels

        if unknown:
            raise RuntimeError(
                f"{split_name} contains full labels unseen in training: "
                f"{sorted(unknown)}"
            )


def verify_prediction_label_coverage(
    prediction_labels: Sequence[str],
    split_sources: Dict[
        str, Tuple[Dict[int, Dict[int, Dict[str, Any]]], Sequence[int]]
    ],
) -> None:
    """Require every validation independent target to occur in training."""

    training_labels = set(prediction_labels)
    records, indices = split_sources["validation"]
    observed = {
        feature
        for sentence_index in indices
        for record in records.get(sentence_index, {}).values()
        for feature in get_independent_record_features(record)
    }
    unknown = observed - training_labels
    if unknown:
        raise RuntimeError(
            "Validation contains independent targets without a training "
            f"output head: {sorted(unknown)}"
        )


def verify_hierarchy_closure(
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    """
    Ensure source feature sets already contain all ancestors required by the
    hierarchy. This prevents evaluation against inconsistent source targets.
    """

    problems: List[str] = []
    max_examples = 20

    for split_name in ("train", "validation"):
        tagged_records, sentence_indices = split_sources[split_name]
        problem_count = 0

        for sentence_index in sentence_indices:
            for token_id, record in tagged_records.get(
                sentence_index,
                {},
            ).items():
                features = clean_features(record)

                if not features:
                    continue

                missing = missing_hierarchy_features(features)

                if not missing:
                    continue

                problem_count += 1

                if len(problems) < max_examples:
                    problems.append(
                        f"{split_name}: sentence={sentence_index}, "
                        f"token={token_id}, "
                        f"form={record.get('form')!r}, "
                        f"missing={sorted(missing)}"
                    )

        if problem_count:
            print(
                f"Hierarchy closure problems in {split_name}: "
                f"{problem_count:,} token(s)"
            )

    if problems:
        example_text = "\n".join(
            f"  - {problem}"
            for problem in problems
        )

        raise RuntimeError(
            "The input dataset is not hierarchy-closed. Fix the source tags "
            "before training so deterministic reduction/expansion is exact.\n"
            f"Examples:\n{example_text}"
        )


def print_split_statistics(
    split_sources: Dict[
        str,
        Tuple[
            Dict[int, Dict[int, Dict[str, Any]]],
            Sequence[int],
        ],
    ],
) -> None:
    print("=" * 80)
    print("DATASET SPLIT")
    print("=" * 80)

    for split_name in ("train", "validation"):
        tagged_records, indices = split_sources[split_name]

        full_counts = count_feature_occurrences(
            tagged_records,
            indices,
            independent_only=False,
        )

        independent_counts = count_feature_occurrences(
            tagged_records,
            indices,
            independent_only=True,
        )

        empty_count = count_empty_records(
            tagged_records,
            indices,
        )

        noun_count = sum(
            len(tagged_records.get(index, {}))
            for index in indices
        )

        source_name = (
            "training file"
            if split_name in {"train", "validation"}
            else "test file"
        )

        print(
            f"{split_name:12s}: "
            f"{len(indices):,} sentences | "
            f"{noun_count:,} NOUN/PROPN tokens | "
            f"{sum(full_counts.values()):,} complete positives -> "
            f"{sum(independent_counts.values()):,} independent positives | "
            f"{empty_count:,} empty-feature tokens excluded | "
            f"source={source_name}"
        )

    print()


def print_prediction_label_statistics(
    prediction_labels: Sequence[str],
    full_labels: Sequence[str],
    train_tagged_records: Dict[int, Dict[int, Dict[str, Any]]],
    train_sentence_indices: Sequence[int],
) -> None:
    """Show which labels are neural outputs versus deterministic-only parents."""

    train_independent_counts = count_feature_occurrences(
        train_tagged_records,
        train_sentence_indices,
        independent_only=True,
    )

    prediction_set = set(prediction_labels)
    deterministic_only = [
        feature
        for feature in full_labels
        if feature not in prediction_set
    ]

    print("=" * 80)
    print("NEURAL OUTPUT FEATURES")
    print("=" * 80)
    print(
        f"Complete evaluation features: {len(full_labels)}"
    )
    print(
        f"Independent neural outputs: {len(prediction_labels)}"
    )
    print(
        f"Deterministic-only features in this training split: "
        f"{len(deterministic_only)}"
    )
    print()

    print(
        f"{'Feature':35s} "
        f"{'Independent train support':>26s}"
    )
    print("-" * 64)

    for feature in prediction_labels:
        print(
            f"{feature:35s} "
            f"{train_independent_counts[feature]:26d}"
        )

    if deterministic_only:
        print()
        print("Deterministic-only in this split:")

        for feature in deterministic_only:
            print(f"  - {feature}")

    print()


def make_json_serialisable(value: Any) -> Any:
    """Recursively convert NumPy scalars/arrays to ordinary JSON values."""

    if isinstance(value, dict):
        return {
            str(key): make_json_serialisable(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            make_json_serialisable(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, (np.floating, np.integer)):
        return value.item()

    return value


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    validate_feature_hierarchy()
    check_cuda()

    if not LEARNING_RATES or not PER_DEVICE_TRAIN_BATCH_SIZES or not EFFECTIVE_BATCH_SIZES:
        raise ValueError("All hyperparameter grid lists must be non-empty.")
    if any(lr <= 0 for lr in LEARNING_RATES):
        raise ValueError("Learning rates must be positive.")

    n_devices = torch.cuda.device_count()
    if n_devices < 1:
        raise RuntimeError("At least one CUDA device is required.")

    grid = []
    for lr, device_batch, effective_batch in product(
        LEARNING_RATES, PER_DEVICE_TRAIN_BATCH_SIZES, EFFECTIVE_BATCH_SIZES
    ):
        denominator = device_batch * n_devices
        if device_batch <= 0 or effective_batch <= 0 or effective_batch % denominator:
            raise ValueError(
                "Each effective batch size must be a positive multiple of "
                "per-device batch size times the number of CUDA devices: "
                f"device_batch={device_batch}, effective_batch={effective_batch}, "
                f"CUDA devices={n_devices}."
            )
        grid.append((lr, device_batch, effective_batch,
                     effective_batch // denominator))
    if len(grid) != len(set(grid)):
        raise ValueError("The hyperparameter grid contains duplicate configurations.")

    print("=" * 80)
    print("VALIDATION-ONLY HYPERPARAMETER GRID SEARCH")
    print("=" * 80)
    print(f"Configurations: {len(grid)} | fine-tuning seed: {TUNING_SEED}")
    print(f"CUDA devices: {n_devices}")
    print("Selection metric: hierarchy-expanded validation macro F1 at threshold 0.5")
    print("The official test corpus is not read during this search.\n")

    train_sentences = load_conllu(TRAIN_CONLLU_PATH)
    train_tagged_records = load_tagged_records(TRAIN_JSONL_PATH)
    validate_feature_switches(train_tagged_records)

    eligible_indices = get_eligible_sentence_indices(
        train_sentences, train_tagged_records
    )
    split = make_train_validation_split(eligible_indices, train_tagged_records)
    train_indices = split["train"]
    validation_indices = split["validation"]
    sources = {
        "train": (train_tagged_records, train_indices),
        "validation": (train_tagged_records, validation_indices),
    }
    split_document = {
        "split_unit": "sentence",
        "stratification": "iterative_multilabel_sentence_presence",
        "split_seed_requested": SPLIT_SEED,
        "split_seed_used": split["split_seed_used"],
        "split_attempt": split["attempt"],
        "validation_fraction": VALIDATION_FRACTION,
        "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
        "prevalence_error": split["prevalence_error"],
        "stratification_labels": split["stratification_labels"],
        "train_jsonl": str(TRAIN_JSONL_PATH),
        "train_conllu": str(TRAIN_CONLLU_PATH),
        "index_spaces": {"train": "training corpus", "validation": "training corpus"},
        "train": train_indices,
        "validation": validation_indices,
    }
    with SPLIT_PATH.open("w", encoding="utf-8") as handle:
        json.dump(make_json_serialisable(split_document), handle,
                  ensure_ascii=False, indent=2)

    verify_hierarchy_closure(sources)
    full_labels = build_full_label_vocabulary(train_tagged_records, train_indices)
    prediction_labels = build_prediction_label_vocabulary(
        train_tagged_records, train_indices
    )
    verify_full_label_coverage(full_labels, sources)
    verify_prediction_label_coverage(prediction_labels, sources)

    label2id = {label: index for index, label in enumerate(prediction_labels)}
    id2label = {index: label for label, index in label2id.items()}
    hierarchy_matrix = build_hierarchy_expansion_matrix(
        prediction_labels, full_labels
    )
    label_document = {
        "labels": prediction_labels,
        "label2id": label2id,
        "id2label": {str(key): value for key, value in id2label.items()},
        "prediction_labels": prediction_labels,
        "prediction_label2id": label2id,
        "full_labels": full_labels,
        "full_label2id": {label: index for index, label in enumerate(full_labels)},
        "feature_parents": {
            child: sorted(parents) for child, parents in sorted(FEATURE_PARENTS.items())
        },
        "target_representation": "per_token_independent_features",
        "split_unit": "sentence",
        "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
        "disabled_features": sorted(
            feature for feature, enabled in FEATURE_SWITCHES.items() if not enabled
        ),
    }
    with LABELS_PATH.open("w", encoding="utf-8") as handle:
        json.dump(label_document, handle, ensure_ascii=False, indent=2)

    print_split_statistics(sources)
    print_prediction_label_statistics(
        prediction_labels, full_labels, train_tagged_records, train_indices
    )

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, use_fast=True, add_prefix_space=True
    )
    train_dataset = SemanticFeatureDataset(
        sentences=train_sentences,
        sentence_indices=train_indices,
        tagged_records=train_tagged_records,
        tokenizer=tokenizer,
        label2id=label2id,
        max_length=MAX_LENGTH,
    )
    validation_dataset = SemanticFeatureDataset(
        sentences=train_sentences,
        sentence_indices=validation_indices,
        tagged_records=train_tagged_records,
        tokenizer=tokenizer,
        label2id=label2id,
        max_length=MAX_LENGTH,
    )
    collator = MultiLabelTokenCollator(
        tokenizer=tokenizer, num_labels=len(prediction_labels)
    )
    use_bf16 = torch.cuda.is_bf16_supported()
    results = []
    work_root = OUTPUT_DIR / "_grid_trainer_work"
    work_root.mkdir(parents=True, exist_ok=True)

    def save_results() -> None:
        # A partial result survives a later interrupted or out-of-memory trial.
        ranked = sorted(
            results,
            key=lambda row: (-row["validation_f1_macro_at_0_5"],
                             -row["validation_f1_micro_at_0_5"], row["trial"]),
        )
        document = {
            "model": MODEL_NAME,
            "selection_data": "validation_only",
            "official_test_evaluated": False,
            "selection_metric": "hierarchy_expanded_validation_f1_macro_at_0_5",
            "tie_breaker": "validation_f1_micro_at_0_5_then_grid_order",
            "training_seed": TUNING_SEED,
            "split_seed": split["split_seed_used"],
            "epochs": NUM_EPOCHS,
            "cuda_devices": n_devices,
            "feature_switches": dict(sorted(FEATURE_SWITCHES.items())),
            "grid_size": len(grid),
            "completed_trials": len(results),
            "best": ranked[0] if ranked else None,
            "results_ranked": ranked,
        }
        with GRID_RESULTS_PATH.open("w", encoding="utf-8") as handle:
            json.dump(make_json_serialisable(document), handle,
                      ensure_ascii=False, indent=2)
        columns = [
            "rank", "trial", "learning_rate", "per_device_train_batch_size",
            "gradient_accumulation_steps", "effective_batch_size",
            "training_seed", "best_epoch",
            "best_epoch_validation_f1_macro_at_0_5",
            "validation_f1_macro_at_0_5", "validation_f1_micro_at_0_5",
            "validation_f1_macro_calibrated_diagnostic",
            "validation_f1_micro_calibrated_diagnostic",
        ]
        with GRID_CSV_PATH.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=columns)
            writer.writeheader()
            for rank, row in enumerate(ranked, start=1):
                writer.writerow({name: rank if name == "rank" else row.get(name)
                                 for name in columns})

    for trial, (lr, device_batch, effective_batch, accumulation) in enumerate(
        grid, start=1
    ):
        print("\n" + "=" * 80)
        print(f"TRIAL {trial}/{len(grid)}: LR={lr:g}, device batch={device_batch}, "
              f"accumulation={accumulation}, effective batch={effective_batch}")
        print("=" * 80)
        set_seed(TUNING_SEED)
        work_dir = work_root / f"trial_{trial:02d}"
        if work_dir.exists():
            shutil.rmtree(work_dir)

        model = RobertaMultiLabelModel(
            model_name=MODEL_NAME,
            num_labels=len(prediction_labels),
            id2label=id2label,
            label2id=label2id,
        )
        args = TrainingArguments(
            output_dir=str(work_dir),
            learning_rate=lr,
            lr_scheduler_type="linear",
            weight_decay=WEIGHT_DECAY,
            per_device_train_batch_size=device_batch,
            per_device_eval_batch_size=EVAL_BATCH_SIZE,
            gradient_accumulation_steps=accumulation,
            num_train_epochs=NUM_EPOCHS,
            eval_strategy=EVAL_STRATEGY,
            save_strategy="epoch",
            logging_strategy="steps",
            logging_steps=50,
            load_best_model_at_end=True,
            metric_for_best_model="f1_macro",
            greater_is_better=True,
            save_total_limit=2,
            bf16=use_bf16,
            fp16=not use_bf16,
            dataloader_num_workers=4,
            dataloader_pin_memory=True,
            optim="adamw_torch",
            report_to="none",
            seed=TUNING_SEED,
            data_seed=TUNING_SEED,
            remove_unused_columns=False,
            eval_accumulation_steps=16,
        )
        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=validation_dataset,
            processing_class=tokenizer,
            data_collator=collator,
            compute_metrics=make_compute_metrics(
                prediction_labels, hierarchy_matrix
            ),
        )
        trainer.train()
        validation_output = trainer.predict(
            validation_dataset, metric_key_prefix="validation"
        )
        probabilities, true_independent = extract_valid_token_arrays(
            validation_output.predictions, validation_output.label_ids
        )
        default_thresholds = np.full(
            len(prediction_labels), DEFAULT_PREDICTION_THRESHOLD,
            dtype=np.float32,
        )
        default_metrics = calculate_hierarchical_metrics_from_probabilities(
            probabilities=probabilities,
            y_true_independent=true_independent,
            threshold_vector=default_thresholds,
            hierarchy_matrix=hierarchy_matrix,
        )
        calibrated_thresholds, _ = optimise_feature_thresholds(
            probabilities=probabilities,
            y_true_independent=true_independent,
            prediction_labels=prediction_labels,
        )
        calibrated_metrics = calculate_hierarchical_metrics_from_probabilities(
            probabilities=probabilities,
            y_true_independent=true_independent,
            threshold_vector=make_threshold_vector(
                prediction_labels, calibrated_thresholds
            ),
            hierarchy_matrix=hierarchy_matrix,
        )
        best_metric = trainer.state.best_metric
        best_checkpoint = trainer.state.best_model_checkpoint
        best_epoch = None
        if best_checkpoint:
            for entry in trainer.state.log_history:
                if "eval_f1_macro" in entry and "step" in entry:
                    # The best checkpoint's step is encoded in its directory.
                    if best_checkpoint.endswith(f"checkpoint-{entry['step']}"):
                        best_epoch = entry.get("epoch")
                        break

        row = {
            "trial": trial,
            "learning_rate": lr,
            "per_device_train_batch_size": device_batch,
            "gradient_accumulation_steps": accumulation,
            "effective_batch_size": effective_batch,
            "training_seed": TUNING_SEED,
            "best_epoch": best_epoch,
            "best_epoch_validation_f1_macro_at_0_5": best_metric,
            "validation_f1_macro_at_0_5": default_metrics["f1_macro"],
            "validation_f1_micro_at_0_5": default_metrics["f1_micro"],
            "validation_f1_macro_calibrated_diagnostic": calibrated_metrics["f1_macro"],
            "validation_f1_micro_calibrated_diagnostic": calibrated_metrics["f1_micro"],
            "validation_metrics_at_0_5": default_metrics,
            "validation_metrics_calibrated_diagnostic": calibrated_metrics,
            "calibrated_thresholds_diagnostic": calibrated_thresholds,
        }
        results.append(row)
        save_results()
        print(f"Validation macro F1 @ 0.5: {row['validation_f1_macro_at_0_5']:.4f}")
        print(f"Validation micro F1 @ 0.5: {row['validation_f1_micro_at_0_5']:.4f}")
        print(f"Results saved: {GRID_RESULTS_PATH.resolve()}")

        del validation_output, trainer, model
        gc.collect()
        torch.cuda.empty_cache()
        if work_dir.exists():
            shutil.rmtree(work_dir)

    if work_root.exists() and not any(work_root.iterdir()):
        work_root.rmdir()

    print("\n" + "=" * 80)
    print("GRID SEARCH RANKING (VALIDATION ONLY)")
    print("=" * 80)
    ranked = sorted(
        results,
        key=lambda row: (-row["validation_f1_macro_at_0_5"],
                         -row["validation_f1_micro_at_0_5"], row["trial"]),
    )
    print(f"{'Rank':>4} {'LR':>10} {'Device':>7} {'Accum':>7} "
          f"{'Effective':>9} {'Macro F1':>10} {'Micro F1':>10}")
    for rank, row in enumerate(ranked, start=1):
        print(f"{rank:4d} {row['learning_rate']:10.1e} "
              f"{row['per_device_train_batch_size']:7d} "
              f"{row['gradient_accumulation_steps']:7d} "
              f"{row['effective_batch_size']:9d} "
              f"{row['validation_f1_macro_at_0_5']:10.4f} "
              f"{row['validation_f1_micro_at_0_5']:10.4f}")
    print(f"\nBest candidate: LR={ranked[0]['learning_rate']:g}, "
          f"device batch={ranked[0]['per_device_train_batch_size']}, "
          f"accumulation={ranked[0]['gradient_accumulation_steps']}, "
          f"effective batch={ranked[0]['effective_batch_size']}")
    print("Use these settings for the separate 30-seed experiment; "
          "this single-seed ranking is provisional.")
    print(f"JSON: {GRID_RESULTS_PATH.resolve()}")
    print(f"CSV:  {GRID_CSV_PATH.resolve()}")


if __name__ == "__main__":
    main()


CUDA
PyTorch version: 2.13.0+cu130
CUDA available: True
CUDA runtime: 13.0
GPU: NVIDIA GeForce RTX 5070 Ti
VRAM: 15.92 GiB

VALIDATION-ONLY HYPERPARAMETER GRID SEARCH
Configurations: 16 | fine-tuning seed: 11
CUDA devices: 1
Selection metric: hierarchy-expanded validation macro F1 at threshold 0.5
The official test corpus is not read during this search.

FEATURE SWITCHES
Disabled features: Rigidity, Shape, State, State_of_Matter
All unlisted features are enabled.


Sentence-level multilabel-stratified train/validation split:
  Eligible training sentences : 11,198
  Target validation sentences : 2,240
  Actual validation sentences : 2,313
  Training sentences retained : 8,885
  Split seed used             : 42
  Stratification labels       : 37
  Coverage-rejected attempts  : 0
  Mean relative prevalence error: 0.008441

Feature                              All sent.      Train      Valid    Valid %
--------------------------------------------------------------------------------
Abstrac

Loading weights: 100%|███████████████████████████████████████████████████████████████| 389/389 [00:00<00:00, 953.48it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision Micro,Recall Micro,F1 Micro,Precision Macro,Recall Macro,F1 Macro,Direct Precision Micro,Direct Recall Micro,Direct F1 Micro,Direct Precision Macro,Direct Recall Macro,Direct F1 Macro
1,0.204412,0.041069,0.931988,0.877214,0.903772,0.812644,0.716606,0.750057,0.876768,0.808425,0.841211,0.715300,0.607322,0.643526
2,0.135776,0.034404,0.937254,0.905577,0.921143,0.903504,0.822989,0.850645,0.876684,0.858315,0.867402,0.861567,0.751166,0.783718
3,0.096449,0.031419,0.944545,0.917695,0.930927,0.904144,0.865921,0.882004,0.897372,0.867985,0.882434,0.862035,0.794345,0.818466
4,0.064173,0.031492,0.938410,0.929078,0.933721,0.902086,0.870703,0.882149,0.887296,0.879560,0.883411,0.871654,0.803651,0.823643
5,0.053944,0.031515,0.941940,0.928567,0.935205,0.907585,0.875303,0.887690,0.893072,0.882930,0.887972,0.889740,0.813697,0.835629


Validation macro F1 @ 0.5: 0.8877
Validation micro F1 @ 0.5: 0.9352
Results saved: /home/mark/verbnet-roberta/models/roberta_semantic_features_grid_search/hyperparameter_grid_results.json

TRIAL 2/16: LR=2.5e-05, device batch=4, accumulation=8, effective batch=32


Loading weights: 100%|███████████████████████████████████████████████████████████████| 389/389 [00:00<00:00, 958.34it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Epoch,Training Loss,Validation Loss,Precision Micro,Recall Micro,F1 Micro,Precision Macro,Recall Macro,F1 Macro,Direct Precision Micro,Direct Recall Micro,Direct F1 Micro,Direct Precision Macro,Direct Recall Macro,Direct F1 Macro
1,0.296833,0.149114,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.296033,0.148609,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.299997,0.148451,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.295788,0.148184,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0.292440,0.148513,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


Validation macro F1 @ 0.5: 0.0000
Validation micro F1 @ 0.5: 0.0000
Results saved: /home/mark/verbnet-roberta/models/roberta_semantic_features_grid_search/hyperparameter_grid_results.json

TRIAL 16/16: LR=4e-05, device batch=8, accumulation=4, effective batch=32


Loading weights: 100%|███████████████████████████████████████████████████████████████| 389/389 [00:00<00:00, 911.92it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision Micro,Recall Micro,F1 Micro,Precision Macro,Recall Macro,F1 Macro,Direct Precision Micro,Direct Recall Micro,Direct F1 Micro,Direct Precision Macro,Direct Recall Macro,Direct F1 Macro
1,0.208357,0.041950,0.935976,0.851698,0.891850,0.835800,0.713549,0.757252,0.879393,0.793773,0.834392,0.735356,0.611892,0.654371
2,0.136902,0.034497,0.936754,0.906472,0.921364,0.894577,0.841246,0.861633,0.885223,0.853187,0.868910,0.855509,0.763543,0.791395
3,0.098460,0.031526,0.942216,0.915553,0.928693,0.896394,0.865869,0.877507,0.894765,0.866447,0.880378,0.853061,0.792591,0.811210
4,0.069202,0.031015,0.940156,0.923291,0.931647,0.902112,0.880936,0.888677,0.890402,0.876703,0.883499,0.873014,0.812606,0.828778
5,0.055650,0.031033,0.942115,0.930517,0.936280,0.904049,0.884999,0.893126,0.892981,0.886374,0.889665,0.883910,0.823689,0.840243


Validation macro F1 @ 0.5: 0.8931
Validation micro F1 @ 0.5: 0.9363
Results saved: /home/mark/verbnet-roberta/models/roberta_semantic_features_grid_search/hyperparameter_grid_results.json

GRID SEARCH RANKING (VALIDATION ONLY)
Rank         LR  Device   Accum Effective   Macro F1   Micro F1
   1    3.5e-05       8       2        16     0.9061     0.9394
   2    4.0e-05       4       4        16     0.9039     0.9376
   3    3.0e-05       8       2        16     0.9001     0.9365
   4    3.5e-05       4       4        16     0.8951     0.9341
   5    4.0e-05       8       4        32     0.8931     0.9363
   6    4.0e-05       4       8        32     0.8909     0.9353
   7    3.0e-05       4       4        16     0.8897     0.9339
   8    2.5e-05       8       2        16     0.8880     0.9352
   9    2.5e-05       4       4        16     0.8877     0.9352
  10    3.5e-05       4       8        32     0.8842     0.9330
  11    3.5e-05       8       4        32     0.8832     0.9341
  12 